<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/14_fullscale_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent storage
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Configure Git remote access
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "your_email@example.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Synchronize repo
%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

# 4. Inject module search path
src_str = str(SRC_DIR.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)

# 5. Install dependencies
!pip install -q transformers pyarrow scikit-learn

print("Environment setup and path injection complete.")

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 154 (delta 65), reused 99 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 518.80 KiB | 14.82 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/DP-MMFL
Environment setup and path injection complete.


In [ ]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"

df = pd.read_parquet(MANIFEST_PATH)

label_cols = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

print("Manifest shape:        ", df.shape)
print("Columns:               ", df.columns.tolist())
print("Unique patients:       ", df["deid_patient_id"].nunique())
print("Duplicate image paths: ", df["path_to_image"].duplicated().sum())

print("\n--- Label Prevalence, Uncertainty, and Missingness ---")
prevalence_records = []
for col in label_cols:
    if col not in df.columns:
        print(f"[MISSING COLUMN] {col} is absent from manifest!")
        continue

    val_counts = df[col].value_counts(dropna=False).to_dict()
    total = len(df)
    pos = val_counts.get(1.0, 0)
    neg = val_counts.get(0.0, 0)
    unc = val_counts.get(-1.0, 0)
    nan_count = df[col].isna().sum()

    prevalence_records.append({
        "Pathology": col,
        "Positive (1.0)": f"{pos} ({pos / total * 100:.2f}%)",
        "Negative (0.0)": f"{neg} ({neg / total * 100:.2f}%)",
        "Uncertain (-1.0)": f"{unc} ({unc / total * 100:.2f}%)",
        "NaN / Missing": f"{nan_count} ({nan_count / total * 100:.2f}%)",
    })

prevalence_df = pd.DataFrame(prevalence_records)
print(prevalence_df.to_string(index=False))

print("\n--- Text Column Inspection (Target Leakage Audit) ---")
text_candidate_cols = [c for c in df.columns if any(k in c.lower() for k in ["text", "report", "impression", "finding", "history", "indication"])]
print("Candidate text fields found:", text_candidate_cols)

for c in text_candidate_cols:
    non_null_count = df[c].notna().sum()
    sample_text = str(df[c].dropna().iloc[0])[:120].replace("\n", " ") if non_null_count > 0 else "N/A"
    print(f" - {c}: {non_null_count}/{len(df)} non-null | Sample: \"{sample_text}...\"")

Manifest shape:         (223462, 58)
Columns:                ['sample_id', 'path_to_image', 'path_to_dcm', 'deid_patient_id', 'report', 'age', 'sex', 'race', 'ethnicity', 'split', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'No Finding', 'target_Enlarged Cardiomediastinum', 'target_Cardiomegaly', 'target_Lung Opacity', 'target_Lung Lesion', 'target_Edema', 'target_Consolidation', 'target_Pneumonia', 'target_Atelectasis', 'target_Pneumothorax', 'target_Pleural Effusion', 'target_Pleural Other', 'target_Fracture', 'target_Support Devices', 'mask_Enlarged Cardiomediastinum', 'mask_Cardiomegaly', 'mask_Lung Opacity', 'mask_Lung Lesion', 'mask_Edema', 'mask_Consolidation', 'mask_Pneumonia', 'mask_Atelectasis', 'mask_Pneumothorax', 'mask_Pleural Effusion', 'mask_Pleural Other', 'mask_Fracture', 'mask_Support Devices', 'expe

In [ ]:
import numpy as np
import pandas as pd

# 1. Dataset and split audit
print("=== SPLIT & VIEW AUDIT ===")
print("\nExperiment split counts:")
print(df["experiment_split"].value_counts(dropna=False))

print("\nOriginal dataset split counts:")
print(df["split"].value_counts(dropna=False))

print("\nFrontal/lateral counts:")
print(df["frontal_lateral"].value_counts(dropna=False))

print("\nAP/PA counts:")
print(df["ap_pa"].value_counts(dropna=False))

# 2. Patient overlap across assigned experiment splits
assigned = df.dropna(subset=["experiment_split"]).copy()
patient_sets = {
    str(split): set(group["deid_patient_id"].dropna().unique())
    for split, group in assigned.groupby("experiment_split")
}

print("\nPatients per experiment split:")
for split, patients in patient_sets.items():
    print(f" - {split}: {len(patients):,} patients")

split_names = list(patient_sets.keys())
print("\nPatient leakage across split boundaries:")
for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        a, b = split_names[i], split_names[j]
        overlap = patient_sets[a] & patient_sets[b]
        print(f" - Overlap ({a} vs {b}): {len(overlap):,} patients")

# 3. Label and Mask Availability Audit (Target vs Mask columns)
target_cols = [c for c in df.columns if c.startswith("target_")]
mask_cols = [c for c in df.columns if c.startswith("mask_")]

print(f"\nTarget columns found ({len(target_cols)}):", target_cols)
print(f"Mask columns found ({len(mask_cols)}):  ", mask_cols)

if target_cols and mask_cols:
    print("\n--- Effective Usable Labels (Active where mask == 1) ---")
    active_summary = []
    for t_col in target_cols:
        suffix = t_col.replace("target_", "")
        m_col = f"mask_{suffix}"
        if m_col in df.columns:
            active = df[m_col] == 1.0
            pos = (df.loc[active, t_col] == 1.0).sum()
            neg = (df.loc[active, t_col] == 0.0).sum()
            total_active = active.sum()
            active_summary.append({
                "Pathology": suffix,
                "Active Labels": f"{total_active:,} ({total_active / len(df) * 100:.1f}%)",
                "Positive": f"{pos:,} ({pos / max(total_active, 1) * 100:.1f}%)",
                "Negative": f"{neg:,} ({neg / max(total_active, 1) * 100:.1f}%)",
            })
    print(pd.DataFrame(active_summary).to_string(index=False))

=== SPLIT & VIEW AUDIT ===

Experiment split counts:
experiment_split
train             178908
test               22563
val                21757
official_valid       234
Name: count, dtype: int64

Original dataset split counts:
split
train    223228
valid       234
Name: count, dtype: int64

Frontal/lateral counts:
frontal_lateral
Frontal    191071
Lateral     32391
Name: count, dtype: int64

AP/PA counts:
ap_pa
AP      161622
None     32391
PA       29432
LL          16
RL           1
Name: count, dtype: int64

Patients per experiment split:
 - official_valid: 200 patients
 - test: 6,453 patients
 - train: 51,620 patients
 - val: 6,452 patients

Patient leakage across split boundaries:
 - Overlap (official_valid vs test): 0 patients
 - Overlap (official_valid vs train): 0 patients
 - Overlap (official_valid vs val): 0 patients
 - Overlap (test vs train): 0 patients
 - Overlap (test vs val): 0 patients
 - Overlap (train vs val): 0 patients

Target columns found (13): ['target_Enlarged 

In [ ]:

import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
SPLIT_AUDIT_DIR = PROJECT_ROOT / "results/final_protocol"
SPLIT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

labels = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

# Keep only frontal images with valid experiment split assignments.
frontal = df[
    df["frontal_lateral"].eq("Frontal")
    & df["experiment_split"].isin(["train", "val", "test", "official_valid"])
].copy()

split_names = ["train", "val", "test", "official_valid"]
split_rows = {}
split_patients = {}
label_report = []

for split in split_names:
    part = frontal[frontal["experiment_split"].eq(split)].copy()
    split_rows[split] = part
    split_patients[split] = set(part["deid_patient_id"].dropna().unique())

    print(
        f"{split:14s} images={len(part):,} | "
        f"patients={part['deid_patient_id'].nunique():,}"
    )

    for label in labels:
        target = pd.to_numeric(part[f"target_{label}"], errors="coerce")
        mask = pd.to_numeric(part[f"mask_{label}"], errors="coerce").fillna(0)
        valid = mask.eq(1) & target.isin([0, 1])

        pos = int((valid & target.eq(1)).sum())
        neg = int((valid & target.eq(0)).sum())

        label_report.append({
            "split": split,
            "label": label,
            "valid": pos + neg,
            "positive": pos,
            "negative": neg,
            "both_classes_present": pos > 0 and neg > 0,
        })

print("\n=== Patient overlap ===")
for i, a in enumerate(split_names):
    for b in split_names[i + 1:]:
        overlap = split_patients[a] & split_patients[b]
        print(f"{a} vs {b}: {len(overlap)}")

report = pd.DataFrame(label_report)
report.to_csv(SPLIT_AUDIT_DIR / "frontal_label_counts_by_split.csv", index=False)

# Keep all split-level counts for reproducibility.
split_summary = {
    split: {
        "images": len(part),
        "patients": int(part["deid_patient_id"].nunique()),
    }
    for split, part in split_rows.items()
}

with open(SPLIT_AUDIT_DIR / "frontal_split_summary.json", "w") as f:
    json.dump(split_summary, f, indent=2)

print("\nSaved audit files to:", SPLIT_AUDIT_DIR)
print("\nClasses missing either positives or negatives:")
print(
    report.loc[~report["both_classes_present"],
               ["split", "label", "positive", "negative"]]
    .to_string(index=False)
)


train          images=152,947 | patients=51,609
val            images=18,573 | patients=6,449
test           images=19,349 | patients=6,452
official_valid images=202 | patients=200

=== Patient overlap ===
train vs val: 0
train vs test: 0
train vs official_valid: 0
val vs test: 0
val vs official_valid: 0
test vs official_valid: 0

Saved audit files to: /content/drive/MyDrive/DP-MMFL/results/final_protocol

Classes missing either positives or negatives:
         split         label  positive  negative
official_valid   Lung Lesion        12         0
official_valid Pleural Other        10         0


In [ ]:

import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
PROTOCOL_DIR = PROJECT_ROOT / "results/final_protocol"
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)

# Freeze the frontal-only cohort and existing patient-level splits.
final_df = df[
    df["frontal_lateral"].eq("Frontal")
    & df["experiment_split"].isin(
        ["train", "val", "test", "official_valid"]
    )
].copy()

# Save sample IDs per split for reproducibility.
split_id_path = PROTOCOL_DIR / "final_split_sample_ids.json"
split_ids = {
    split: sorted(
        final_df.loc[
            final_df["experiment_split"].eq(split), "sample_id"
        ].astype(str).tolist()
    )
    for split in ["train", "val", "test", "official_valid"]
}

with open(split_id_path, "w") as f:
    json.dump(split_ids, f, indent=2)

# Save the protocol and its key limitations.
protocol = {
    "protocol_version": "v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "CheXpert Plus",
    "cohort_filter": "frontal images only",
    "split_column": "experiment_split",
    "split_counts": {
        split: {
            "images": int(
                final_df["experiment_split"].eq(split).sum()
            ),
            "patients": int(
                final_df.loc[
                    final_df["experiment_split"].eq(split),
                    "deid_patient_id"
                ].nunique()
            ),
        }
        for split in ["train", "val", "test", "official_valid"]
    },
    "primary_task": "image-only multi-label classification",
    "text_input": False,
    "target_columns": [f"target_{label}" for label in label_cols],
    "mask_columns": [f"mask_{label}" for label in label_cols],
    "label_policy": (
        "Use only entries with mask=1 and binary target 0 or 1; "
        "ignore masked or invalid entries."
    ),
    "split_integrity": "Patient-disjointness audit passed",
    "official_validation_caveat": (
        "Lung Lesion and Pleural Other have no negative examples "
        "in the frontal official-validation subset."
    ),
    "test_policy": (
        "Do not use test data for checkpoint selection or tuning."
    ),
}

protocol_path = PROTOCOL_DIR / "final_protocol_v1.json"
with open(protocol_path, "w") as f:
    json.dump(protocol, f, indent=2)

print("Protocol saved:", protocol_path)
print("Split IDs saved:", split_id_path)
print("\nPrimary split counts:")
print(json.dumps(protocol["split_counts"], indent=2))

print("\nNext: verify image access and throughput before full training.")


Protocol saved: /content/drive/MyDrive/DP-MMFL/results/final_protocol/final_protocol_v1.json
Split IDs saved: /content/drive/MyDrive/DP-MMFL/results/final_protocol/final_split_sample_ids.json

Primary split counts:
{
  "train": {
    "images": 152947,
    "patients": 51609
  },
  "val": {
    "images": 18573,
    "patients": 6449
  },
  "test": {
    "images": 19349,
    "patients": 6452
  },
  "official_valid": {
    "images": 202,
    "patients": 200
  }
}

Next: verify image access and throughput before full training.


In [ ]:
import io
import os
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T

# Standard evaluation transform for 224x224 ResNet backbone
image_eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PreprocessedCachedImageDataset(Dataset):
    """
    On-demand cached dataset reading images from local paths or Google Drive.
    Converts single-channel grayscale radiographs to 3-channel RGB.
    """
    def __init__(self, df, transform=None, cache=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.cache = cache if cache is not None else {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.get("path_to_image", row.get("image_path", None))

        # Check in-memory cache first
        if img_path in self.cache:
            img = self.cache[img_path]
        else:
            full_path = Path(img_path)
            if not full_path.is_absolute():
                full_path = PROJECT_ROOT / full_path

            with open(full_path, "rb") as f:
                img = Image.open(io.BytesIO(f.read())).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        # Build standard sample dictionary
        sample = {
            "image": img,
            "sample_id": int(row.get("sample_id", idx)),
            "patient_id": str(row.get("deid_patient_id", "")),
        }

        # Include labels and masks if present
        label_cols = [c for c in self.df.columns if c.startswith("target_")]
        mask_cols = [c for c in self.df.columns if c.startswith("mask_")]

        if label_cols:
            sample["labels"] = torch.tensor(row[label_cols].values.astype(float), dtype=torch.float32)
        if mask_cols:
            sample["label_mask"] = torch.tensor(row[mask_cols].values.astype(float), dtype=torch.float32)

        return sample

print("Dataset class and evaluation transform loaded successfully.")

Dataset class and evaluation transform loaded successfully.


In [ ]:
import time
import pandas as pd
import torch
from torch.utils.data import DataLoader

train_df = final_df[
    final_df["experiment_split"].eq("train")
].reset_index(drop=True)

print("Training cohort size:", f"{len(train_df):,}")
print("CUDA available:      ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:          ", torch.cuda.get_device_name(0))

benchmark_sizes = [256, 1000, 2000]
benchmark_sizes = [n for n in benchmark_sizes if n <= len(train_df)]
benchmark_results = []

for n in benchmark_sizes:
    print(f"\n--- Benchmark: {n:,} images ---")
    sample_df = train_df.iloc[:n].copy()

    # Use existing dataset constructor & evaluation transform
    dataset = PreprocessedCachedImageDataset(
        sample_df,
        transform=image_eval_transform
    )

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    start_time = time.perf_counter()
    images_loaded = 0
    batches_processed = 0

    for batch in loader:
        # Extract images tensor whether returned as dict or tuple
        if isinstance(batch, dict):
            imgs = batch["image"]
        elif isinstance(batch, (list, tuple)):
            imgs = batch[0]
        else:
            imgs = batch

        images_loaded += imgs.size(0)
        batches_processed += 1

    elapsed = time.perf_counter() - start_time
    fps = images_loaded / elapsed if elapsed > 0 else 0.0

    result = {
        "Subset Size": n,
        "Loaded Images": images_loaded,
        "Batches": batches_processed,
        "Elapsed (s)": round(elapsed, 2),
        "Throughput (img/s)": round(fps, 2),
        "Est. Epoch Time (Full Cohort)": f"{(len(train_df) / max(fps, 1e-3)) / 60:.1f} min",
    }
    benchmark_results.append(result)
    print(
        f"Loaded {images_loaded:,} images in {elapsed:.2f}s "
        f"({fps:.1f} img/s) | Est. full epoch: {result['Est. Epoch Time (Full Cohort)']}"
    )

print("\n=== Throughput Benchmark Summary ===")
summary_table = pd.DataFrame(benchmark_results)
print(summary_table.to_string(index=False))

Training cohort size: 152,947
CUDA available:       True
GPU Device:           Tesla T4

--- Benchmark: 256 images ---


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_824/2017260522.py", line 41, in __getitem__
    with open(full_path, "rb") as f:
         ~~~~^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DP-MMFL/train/patient00003/study1/view1_frontal.jpg'


In [ ]:
import os
from pathlib import Path

sample_rel_path = train_df.iloc[0]["path_to_image"]
print("Sample path from manifest:", sample_rel_path)

# Test common root candidates
test_roots = [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "chexpert",
    PROJECT_ROOT / "data" / "CheXpert-v1.0-small",
    Path("/content"),
    Path("/content/data"),
    Path("/content/CheXpert-v1.0-small"),
]

found_root = None
for root in test_roots:
    candidate = root / sample_rel_path
    if candidate.exists():
        found_root = root
        print(f"\n[FOUND] Valid root directory: {found_root}")
        print(f"Verified sample file: {candidate}")
        break

if not found_root:
    print("\nScanning for sample file name across PROJECT_ROOT and /content...")
    target_filename = Path(sample_rel_path).name
    matches = list(PROJECT_ROOT.rglob(target_filename))[:5]
    if not matches:
        matches = list(Path("/content").rglob(target_filename))[:5]
    print(f"Matches found for {target_filename}:", matches)

Sample path from manifest: train/patient00003/study1/view1_frontal.jpg

Scanning for sample file name across PROJECT_ROOT and /content...
Matches found for view1_frontal.jpg: []


In [ ]:
import os
from pathlib import Path

print("=== Checking existing data directories ===")
data_dir = PROJECT_ROOT / "data"
if data_dir.exists():
    for item in sorted(data_dir.rglob("*")):
        if item.is_file() and item.suffix in [".pt", ".h5", ".parquet", ".zip", ".tar", ".csv"]:
            size_mb = item.stat().st_size / (1024 * 1024)
            print(f"  {item.relative_to(PROJECT_ROOT)} ({size_mb:.1f} MB)")
        elif item.is_dir() and not list(item.iterdir()):
            pass

print("\n=== Checking /content root ===")
for p in Path("/content").glob("*"):
    print(f"  {p} ({'dir' if p.is_dir() else f'{p.stat().st_size / (1024*1024):.1f} MB'})")

print("\n=== Checking pilot dataset configuration ===")
# Check how images were referenced in the 1,000-image pilot
pilot_meta = PROJECT_ROOT / "results" / "E2_multimodal" / "split_info.json"
if pilot_meta.exists():
    import json
    with open(pilot_meta) as f:
        meta = json.load(f)
    print("E2 split info keys/sample:", list(meta.keys())[:5])
    if isinstance(meta, dict) and "sample_paths" in meta:
        print("Sample pilot path:", meta["sample_paths"][:2])

=== Checking existing data directories ===
  data/processed/chexpert_plus_manifest.parquet (164.8 MB)
  data/processed/chexpert_plus_manifest_v2.parquet (63.5 MB)
  data/processed/image_validation_results.parquet (0.0 MB)
  data/processed/image_validation_sample.parquet (0.1 MB)
  data/raw/chexpert_plus/df_chexpert_plus_240401.csv (386.2 MB)

=== Checking /content root ===
  /content/.config (dir)
  /content/DP-MMFL (dir)
  /content/drive (dir)
  /content/sample_data (dir)

=== Checking pilot dataset configuration ===
E2 split info keys/sample: ['train_sample_ids', 'val_sample_ids', 'source_split', 'subset_size', 'text_column']


In [ ]:
import pandas as pd
from pathlib import Path

# Inspect image validation parquet files
val_results_path = PROJECT_ROOT / "data/processed/image_validation_results.parquet"
val_sample_path = PROJECT_ROOT / "data/processed/image_validation_sample.parquet"

if val_results_path.exists():
    df_val_res = pd.read_parquet(val_results_path)
    print("=== image_validation_results.parquet ===")
    print("Shape:", df_val_res.shape)
    print("Columns:", df_val_res.columns.tolist())
    print("\nSample rows:")
    display_cols = [c for c in ["path_to_image", "exists", "resolved_path", "full_path", "status"] if c in df_val_res.columns]
    if not display_cols:
        display_cols = df_val_res.columns[:5]
    print(df_val_res[display_cols].head(3))

if val_sample_path.exists():
    df_val_samp = pd.read_parquet(val_sample_path)
    print("\n=== image_validation_sample.parquet ===")
    print("Shape:", df_val_samp.shape)
    print("Columns:", df_val_samp.columns.tolist())
    sample_cols = [c for c in ["path_to_image", "exists", "resolved_path", "full_path"] if c in df_val_samp.columns]
    if not sample_cols:
        sample_cols = df_val_samp.columns[:5]
    print(df_val_samp[sample_cols].head(3))

# Check how notebook 13 loaded images or if there is a cached tensor / archive
print("\n=== Checking notebook 13 / pilot loader setup ===")
# Inspect if client_loaders or train_loader dataset in memory has an image_dir or file list
if "client_loaders" in globals():
    sample_ds = client_loaders["client_1"].dataset
    print("Dataset type:", type(sample_ds).__name__)
    for attr in ["image_dir", "root", "img_dir", "data_dir", "cache_dir", "paths", "df"]:
        if hasattr(sample_ds, attr):
            val = getattr(sample_ds, attr)
            if isinstance(val, (str, Path)):
                print(f"Dataset attribute '{attr}':", val)
            elif isinstance(val, pd.DataFrame):
                print(f"Dataset attribute 'df' shape: {val.shape}, path sample: {val['path_to_image'].iloc[0] if 'path_to_image' in val.columns else 'N/A'}")

=== image_validation_results.parquet ===
Shape: (100, 11)
Columns: ['sample_id', 'patient_id', 'path', 'redivis_path', 'success', 'bytes', 'format', 'mode', 'width', 'height', 'error']

Sample rows:
   sample_id    patient_id                                         path  \
0     219968  patient31564  train/patient31564/study2/view1_frontal.jpg   
1     144962  patient28804  train/patient28804/study1/view1_frontal.jpg   
2     210680  patient54848  train/patient54848/study1/view1_frontal.jpg   

                            redivis_path  success  
0  patient31564/study2/view1_frontal.png     True  
1  patient28804/study1/view1_frontal.png     True  
2  patient54848/study1/view1_frontal.png     True  

=== image_validation_sample.parquet ===
Shape: (100, 55)
Columns: ['sample_id', 'path_to_image', 'path_to_dcm', 'deid_patient_id', 'report', 'age', 'sex', 'race', 'ethnicity', 'split', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pn

In [ ]:
import os
import inspect
from pathlib import Path

# 1. Check if there are local cache directories in /tmp or /content
print("=== Checking /tmp and local scratch caches ===")
for base in [Path("/tmp"), Path("/content")]:
    matches = list(base.glob("*chexpert*")) + list(base.glob("*cache*")) + list(base.glob("*redivis*"))
    for m in matches:
        print(f"  {m} ({'dir' if m.is_dir() else f'{m.stat().st_size / (1024*1024):.1f} MB'})")

# 2. Check if redivis is installed or used in the environment
try:
    import redivis
    print("\nRedivis package is installed: YES")
except ImportError:
    print("\nRedivis package is installed: NO")

# 3. Check definition of image loading in earlier cells or namespace
for func_name in ["get_image", "load_image", "read_image", "fetch_image", "PreprocessedCachedImageDataset"]:
    if func_name in globals():
        func = globals()[func_name]
        print(f"\nSource of {func_name}:")
        try:
            print(inspect.getsource(func))
        except Exception as e:
            print(f"  Could not get source: {e}")

# 4. Check results/E2_multimodal or results/E4_dp_federated for image index or cached tensors
print("\n=== Checking for cached image arrays or tensors in results/ ===")
for p in (PROJECT_ROOT / "results").rglob("*.pt"):
    if "image" in p.name.lower() or "cache" in p.name.lower():
        print(f"  {p.relative_to(PROJECT_ROOT)} ({p.stat().st_size / (1024*1024):.1f} MB)")

=== Checking /tmp and local scratch caches ===

Redivis package is installed: NO

Source of PreprocessedCachedImageDataset:
  Could not get source: source code not available

=== Checking for cached image arrays or tensors in results/ ===


In [ ]:
import json
from pathlib import Path

# 1. Search for any notebook files in PROJECT_ROOT to locate where images were fetched
notebooks = sorted(PROJECT_ROOT.glob("*.ipynb"))
print("Found notebooks in PROJECT_ROOT:")
for nb in notebooks:
    print(f"  {nb.name}")

# 2. Inspect Notebook 13 or Notebook 12 for the exact dataset loading cell
target_nb = PROJECT_ROOT / "13_federated_baseline.ipynb"
if not target_nb.exists():
    # Check current directory
    target_nb = Path("13_federated_baseline.ipynb")

if target_nb.exists():
    with open(target_nb, "r", encoding="utf-8") as f:
        nb_data = json.load(f)
    print(f"\nScanning cells in {target_nb.name} for image loading...")
    for idx, cell in enumerate(nb_data.get("cells", [])):
        if cell.get("cell_type") == "code":
            source = "".join(cell.get("source", []))
            if any(k in source for k in ["client_loaders", "train_loader", "PreprocessedCachedImageDataset", "redivis", "download", ".pt"]):
                # Print first 5 lines of the matching cell
                lines = [line.strip() for line in source.split("\n") if line.strip()][:6]
                print(f"  [Cell {idx}]: {lines}")

Found notebooks in PROJECT_ROOT:


In [ ]:
from pathlib import Path

CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT_256.mkdir(parents=True, exist_ok=True)

cached_files = list(CACHE_ROOT_256.glob("*.png"))
print(f"Images currently cached in {CACHE_ROOT_256}: {len(cached_files):,}")

Images currently cached in /content/dp_mmfl_image_cache_256/train: 0


In [ ]:
import time
from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader
from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset
from torchvision import transforms

# 1. Transform matching the original pilot evaluation pipeline
image_eval_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 2. Gather available sample IDs in local cache
cached_sample_ids = sorted([int(p.stem) for p in CACHE_ROOT_256.glob("*.png")])
print(f"Usable locally cached samples: {len(cached_sample_ids):,}")

# Test sizes bounded by what is actually downloaded locally
benchmark_sizes = [64, 128, 256, 1000]
benchmark_sizes = [n for n in benchmark_sizes if n <= len(cached_sample_ids)]

if not benchmark_sizes:
    print("\n[!] No images found in local cache.")
    print("If you need to fetch images, authenticate with Redivis using `redivis.authorize()`")
else:
    benchmark_results = []
    for n in benchmark_sizes:
        print(f"\n--- Benchmark: {n} cached images ---")
        sub_sample_ids = cached_sample_ids[:n]

        dataset = PreprocessedCachedImageDataset(
            manifest_path=MANIFEST_PATH,
            cache_root=CACHE_ROOT_256,
            sample_ids=sub_sample_ids,
            image_transform=image_eval_transform,
        )

        loader = DataLoader(
            dataset,
            batch_size=32,
            shuffle=False,
            num_workers=2,
            pin_memory=torch.cuda.is_available(),
        )

        start = time.perf_counter()
        count = 0
        for batch in loader:
            imgs = batch["image"]
            count += imgs.size(0)

        elapsed = time.perf_counter() - start
        fps = count / elapsed if elapsed > 0 else 0

        benchmark_results.append({
            "Samples": n,
            "Elapsed (s)": round(elapsed, 2),
            "Throughput (img/s)": round(fps, 1),
            "Est. Full Train Epoch (152k imgs)": f"{(152947 / max(fps, 1e-3)) / 3600:.2f} hours",
        })
        print(f"Loaded {count} images in {elapsed:.2f}s ({fps:.1f} img/s)")

    print("\n=== Image Loading Throughput Summary ===")
    print(pd.DataFrame(benchmark_results).to_string(index=False))

Usable locally cached samples: 0

[!] No images found in local cache.
If you need to fetch images, authenticate with Redivis using `redivis.authorize()`


In [ ]:
!pip install -q redivis
import redivis

# This will prompt an authorization URL or code in Colab
redivis.authorize()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.2/862.2 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.4/407.4 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 104.2 MB/s eta 0:00:00


AttributeError: module 'redivis' has no attribute 'authorize'

In [ ]:
import redivis

# Launch authentication flow
redivis.authenticate()

Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=9c4aae341d36ade10e7bd58e7634c62b


In [ ]:
from io import BytesIO
from pathlib import Path
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
import redivis

CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT_256.mkdir(parents=True, exist_ok=True)

# Select the first 256 training frontal image IDs from the manifest
benchmark_target_df = train_df.iloc[:256].copy()
target_ids = set(benchmark_target_df["sample_id"].astype(int))
existing_ids = {int(p.stem) for p in CACHE_ROOT_256.glob("*.png")}
needed_ids = target_ids - existing_ids

print(f"Target sample count: {len(target_ids)}")
print(f"Already cached:      {len(existing_ids & target_ids)}")
print(f"To download:         {len(needed_ids)}")

if needed_ids:
    print(f"\nDownloading {len(needed_ids)} images from Redivis...")
    table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")
    dl_manifest = benchmark_target_df[benchmark_target_df["sample_id"].astype(int).isin(needed_ids)].copy()

    for row in tqdm(dl_manifest.itertuples(), total=len(dl_manifest), desc="Caching images"):
        dst_path = CACHE_ROOT_256 / f"{int(row.sample_id):07d}.png"
        if dst_path.exists():
            continue

        png_path = str(row.path_to_image)
        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]
        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"

        try:
            file = table.file(png_path)
            data = file.read(as_text=False)
            img = Image.open(BytesIO(data)).convert("L")
            img = img.resize((256, 256), Image.Resampling.BILINEAR)
            img.save(dst_path, format="PNG")
        except Exception as e:
            print(f"Failed sample {row.sample_id}: {e}")

print(f"\nCache update complete. Files in cache: {len(list(CACHE_ROOT_256.glob('*.png')))}")

Target sample count: 256
Already cached:      0
To download:         256



Caching images:   0%|          | 0/256 [00:00<?, ?it/s]


Cache update complete. Files in cache: 256


In [ ]:
import time
import pandas as pd
import torch
from torch.utils.data import DataLoader
from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset
from torchvision import transforms

image_eval_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

cached_sample_ids = sorted([int(p.stem) for p in CACHE_ROOT_256.glob("*.png")])
print(f"Benchmarking with {len(cached_sample_ids)} cached images...")

dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=CACHE_ROOT_256,
    sample_ids=cached_sample_ids,
    image_transform=image_eval_transform,
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

start = time.perf_counter()
count = 0
for batch in loader:
    imgs = batch["image"]
    count += imgs.size(0)

elapsed = time.perf_counter() - start
fps = count / elapsed if elapsed > 0 else 0

print(f"\nThroughput: {fps:.1f} img/s ({count} images in {elapsed:.2f}s)")
print(f"Projected epoch duration for full 152k cohort: {(152947 / max(fps, 1e-3)) / 3600:.2f} hours")

Benchmarking with 256 cached images...

Throughput: 255.2 img/s (256 images in 1.00s)
Projected epoch duration for full 152k cohort: 0.17 hours


In [ ]:
import time
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset

assert torch.cuda.is_available(), "Enable a GPU runtime first."
device = torch.device("cuda")

# 1. Gather cached sample IDs
cached_sample_ids = sorted([int(p.stem) for p in CACHE_ROOT_256.glob("*.png")])[:256]
assert len(cached_sample_ids) > 0, "No cached images found in CACHE_ROOT_256."

# 2. Dataset & Loader
benchmark_ds = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=CACHE_ROOT_256,
    sample_ids=cached_sample_ids,
    image_transform=image_eval_transform,
)

benchmark_loader = DataLoader(
    benchmark_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# 3. Model setup (ResNet18 baseline with 13 binary logits)
model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 13)
model = model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# 4. Benchmarking loop: Warmup + Timing + VRAM profiling
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

timings = []
processed = 0

print(f"=== Benchmarking Forward + Backward + Optimizer Step on {torch.cuda.get_device_name(0)} ===")

for step, batch in enumerate(benchmark_loader):
    images = batch["image"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True).float()
    masks = batch["label_mask"].to(device, non_blocking=True).float()

    torch.cuda.synchronize(device)
    t0 = time.perf_counter()

    optimizer.zero_grad(set_to_none=True)
    logits = model(images)

    # Masked binary cross-entropy loss
    element_loss = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    loss = (element_loss * masks).sum() / masks.sum().clamp_min(1.0)

    loss.backward()
    optimizer.step()

    torch.cuda.synchronize(device)
    t1 = time.perf_counter()

    step_duration = t1 - t0
    batch_size = images.size(0)

    # Discard step 0 from metrics as CUDA warmup
    if step > 0:
        timings.append(step_duration / batch_size)

    processed += batch_size
    print(f"  Step {step + 1}/{len(benchmark_loader)} | Batch Size: {batch_size} | Step Time: {step_duration * 1000:.1f} ms | Batch Loss: {loss.item():.4f}")

# 5. Throughput & VRAM Summary
avg_time_per_img = sum(timings) / len(timings)
training_fps = 1.0 / avg_time_per_img
peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

full_cohort_size = len(train_df)
est_epoch_minutes = (full_cohort_size * avg_time_per_img) / 60.0

print("\n=== Training Benchmark Results ===")
print(f"Peak GPU VRAM allocated:    {peak_vram_mb:.1f} MB")
print(f"Training throughput:        {training_fps:.1f} img/s")
print(f"Average time per image:     {avg_time_per_img * 1000:.2f} ms")
print(f"Projected single epoch time ({full_cohort_size:,} images): {est_epoch_minutes:.1f} minutes ({est_epoch_minutes / 60.0:.2f} hours)")

=== Benchmarking Forward + Backward + Optimizer Step on Tesla T4 ===
  Step 1/8 | Batch Size: 32 | Step Time: 2957.5 ms | Batch Loss: 0.6699
  Step 2/8 | Batch Size: 32 | Step Time: 131.1 ms | Batch Loss: 0.7076
  Step 3/8 | Batch Size: 32 | Step Time: 129.4 ms | Batch Loss: 0.6943
  Step 4/8 | Batch Size: 32 | Step Time: 92.8 ms | Batch Loss: 0.6653
  Step 5/8 | Batch Size: 32 | Step Time: 91.0 ms | Batch Loss: 0.6200
  Step 6/8 | Batch Size: 32 | Step Time: 90.7 ms | Batch Loss: 0.6082
  Step 7/8 | Batch Size: 32 | Step Time: 90.5 ms | Batch Loss: 0.6066
  Step 8/8 | Batch Size: 32 | Step Time: 91.3 ms | Batch Loss: 0.5708

=== Training Benchmark Results ===
Peak GPU VRAM allocated:    897.9 MB
Training throughput:        312.5 img/s
Average time per image:     3.20 ms
Projected single epoch time (152,947 images): 8.2 minutes (0.14 hours)


In [ ]:

import torch
import torchvision
from torchvision.models import resnet18, ResNet18_Weights

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

# Verify pretrained weights can be loaded.
try:
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = torch.nn.Linear(model.fc.in_features, 13)

    if torch.cuda.is_available():
        model = model.cuda()

    print("Pretrained ResNet-18: PASS")
    print("Output classes:", model.fc.out_features)
    print("Device:", next(model.parameters()).device)

except Exception as e:
    print("Pretrained ResNet-18: FAILED")
    print(type(e).__name__, str(e)[:500])
    raise


PyTorch: 2.11.0+cu130
TorchVision: 0.26.0+cu130
CUDA available: True
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


Pretrained ResNet-18: PASS
Output classes: 13
Device: cuda:0


In [ ]:
import torch
from torch.utils.data import DataLoader

train_df = final_df[
    final_df["experiment_split"].eq("train")
].reset_index(drop=True)

val_df = final_df[
    final_df["experiment_split"].eq("val")
].reset_index(drop=True)

print("Train rows:     ", f"{len(train_df):,}")
print("Validation rows:", f"{len(val_df):,}")

# Test both calling conventions to identify the active class signature
probe_df = train_df.iloc[:64].copy()

try:
    # 1. Try DataFrame-based interface
    probe_ds = PreprocessedCachedImageDataset(
        probe_df,
        transform=image_eval_transform
    )
    print("Dataset instantiated via: DataFrame signature (df, transform)")
except TypeError:
    # 2. Try repository-native signature
    probe_ds = PreprocessedCachedImageDataset(
        manifest_path=MANIFEST_PATH,
        cache_root=CACHE_ROOT_256,
        sample_ids=probe_df["sample_id"].astype(int).tolist(),
        image_transform=image_eval_transform
    )
    print("Dataset instantiated via: Manifest/Path signature (manifest_path, cache_root, sample_ids, image_transform)")

probe_loader = DataLoader(
    probe_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

batch = next(iter(probe_loader))
print("\nBatch type:", type(batch).__name__)

if isinstance(batch, dict):
    print("Batch keys:", list(batch.keys()))
    for k, v in batch.items():
        if torch.is_tensor(v):
            print(f"  - {k:15s}: tensor shape={tuple(v.shape)}, dtype={v.dtype}, device={v.device}")
        elif isinstance(v, (list, tuple)):
            print(f"  - {k:15s}: {type(v).__name__} len={len(v)}, sample={v[:2]}")
        else:
            print(f"  - {k:15s}: {type(v).__name__}")
elif isinstance(batch, (list, tuple)):
    print(f"Batch elements count: {len(batch)}")
    for i, item in enumerate(batch):
        if torch.is_tensor(item):
            print(f"  [{i}]: tensor shape={tuple(item.shape)}, dtype={item.dtype}")
        elif isinstance(item, (list, tuple)):
            print(f"  [{i}]: {type(item).__name__} len={len(item)}, sample={item[:2]}")
        else:
            print(f"  [{i}]: {type(item).__name__}")
else:
    print("Direct object:", batch)

Train rows:      152,947
Validation rows: 18,573
Dataset instantiated via: Manifest/Path signature (manifest_path, cache_root, sample_ids, image_transform)

Batch type: dict
Batch keys: ['image', 'labels', 'label_mask', 'sample_id', 'patient_id']
  - image          : tensor shape=(8, 3, 224, 224), dtype=torch.float32, device=cpu
  - labels         : tensor shape=(8, 13), dtype=torch.float32, device=cpu
  - label_mask     : tensor shape=(8, 13), dtype=torch.float32, device=cpu
  - sample_id      : tensor shape=(8,), dtype=torch.int64, device=cpu
  - patient_id     : list len=8, sample=['patient00003', 'patient00009']


In [ ]:
# ==============================================================================
# E5: Full-Scale Centralized Image Baseline (Pretrained ResNet-18)
# Patient-disjoint splits | Valid-masked BCE | Dual AUROC/AUPRC tracking
# ==============================================================================
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights

# ---------- Paths and Configuration ----------
PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")
RUN_DIR = PROJECT_ROOT / "results/E5_large_centralized_image"
CKPT_DIR = RUN_DIR / "checkpoints"
METRICS_DIR = RUN_DIR / "metrics"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3

LABEL_COLS = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

# ---------- Dataset Partitions ----------
train_df = final_df[
    final_df["experiment_split"].eq("train") & (final_df["frontal_lateral"] == "Frontal")
].reset_index(drop=True)

val_df = final_df[
    final_df["experiment_split"].eq("val") & (final_df["frontal_lateral"] == "Frontal")
].reset_index(drop=True)

train_sample_ids = train_df["sample_id"].astype(int).tolist()
val_sample_ids = val_df["sample_id"].astype(int).tolist()

print(f"=== Dataset Partition Overview ===")
print(f"Train cohort: {len(train_sample_ids):,} frontal images ({train_df['deid_patient_id'].nunique():,} patients)")
print(f"Val cohort:   {len(val_sample_ids):,} frontal images ({val_df['deid_patient_id'].nunique():,} patients)")

# Fallback-aware dataset instantiation matching verified interface
def build_dataset(df_subset, sample_ids):
    try:
        return PreprocessedCachedImageDataset(
            df_subset,
            transform=image_eval_transform
        )
    except TypeError:
        return PreprocessedCachedImageDataset(
            manifest_path=MANIFEST_PATH,
            cache_root=CACHE_ROOT,
            sample_ids=sample_ids,
            image_transform=image_eval_transform,
        )

train_dataset = build_dataset(train_df, train_sample_ids)
val_dataset = build_dataset(val_df, val_sample_ids)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(NUM_WORKERS > 0),
)

# ---------- Model Architecture & Optimization ----------
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(LABEL_COLS))
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

def unpack_batch(batch):
    if isinstance(batch, dict):
        images = batch["image"]
        labels = batch["labels"]
        masks = batch["label_mask"]
    else:
        images, labels, masks = batch[0], batch[1], batch[2]
    return (
        images.to(DEVICE, non_blocking=True),
        labels.to(DEVICE, non_blocking=True).float(),
        masks.to(DEVICE, non_blocking=True).float(),
    )

def masked_bce_loss(logits, targets, masks):
    element_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    return (element_loss * masks).sum() / masks.sum().clamp_min(1.0)

# ---------- Validation Function ----------
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_valid = 0.0

    all_logits = []
    all_targets = []
    all_masks = []

    with torch.no_grad():
        for batch in loader:
            images, labels, masks = unpack_batch(batch)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = masked_bce_loss(logits, labels, masks)

            batch_valid = float(masks.sum().item())
            total_loss += loss.item() * batch_valid
            total_valid += batch_valid

            all_logits.append(logits.float().cpu())
            all_targets.append(labels.float().cpu())
            all_masks.append(masks.float().cpu())

    mean_loss = total_loss / max(total_valid, 1.0)
    all_probs = torch.sigmoid(torch.cat(all_logits, dim=0)).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_masks = torch.cat(all_masks, dim=0).numpy()

    aurocs = {}
    auprcs = {}
    valid_aurocs = []
    valid_auprcs = []

    for i, col in enumerate(LABEL_COLS):
        col_mask = all_masks[:, i] == 1.0
        y_true = all_targets[col_mask, i]
        y_prob = all_probs[col_mask, i]

        # Calculate metrics strictly when both classes exist in active masks
        if len(np.unique(y_true)) == 2:
            auc = float(roc_auc_score(y_true, y_prob))
            prc = float(average_precision_score(y_true, y_prob))
            aurocs[col] = auc
            auprcs[col] = prc
            valid_aurocs.append(auc)
            valid_auprcs.append(prc)
        else:
            aurocs[col] = None
            auprcs[col] = None

    macro_auroc = float(np.mean(valid_aurocs)) if valid_aurocs else None
    macro_auprc = float(np.mean(valid_auprcs)) if valid_auprcs else None

    return {
        "val_loss": mean_loss,
        "macro_auroc": macro_auroc,
        "macro_auprc": macro_auprc,
        "per_class_auroc": aurocs,
        "per_class_auprc": auprcs,
    }

# ---------- Training Execution ----------
print(f"\n=== Launching Centralized Baseline Training ({MAX_EPOCHS} Epochs max) ===")
history = []
best_macro_auroc = -1.0
epochs_no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.perf_counter()
    model.train()
    train_loss_accum = 0.0
    train_valid_accum = 0.0

    for step, batch in enumerate(train_loader):
        images, labels, masks = unpack_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = masked_bce_loss(logits, labels, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_valid = float(masks.sum().item())
        train_loss_accum += loss.item() * batch_valid
        train_valid_accum += batch_valid

    train_loss = train_loss_accum / max(train_valid_accum, 1.0)
    val_metrics = evaluate(model, val_loader)
    epoch_duration = time.perf_counter() - epoch_start

    macro_auroc = val_metrics["macro_auroc"]
    macro_auprc = val_metrics["macro_auprc"]
    auroc_str = f"{macro_auroc:.4f}" if macro_auroc is not None else "N/A"
    auprc_str = f"{macro_auprc:.4f}" if macro_auprc is not None else "N/A"

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS:02d} [{epoch_duration / 60:.1f}m] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['val_loss']:.4f} | "
        f"Val Macro AUROC: {auroc_str} | "
        f"Val Macro AUPRC: {auprc_str}"
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["val_loss"],
        "macro_auroc": macro_auroc,
        "macro_auprc": macro_auprc,
        "per_class_auroc": val_metrics["per_class_auroc"],
        "per_class_auprc": val_metrics["per_class_auprc"],
        "duration_sec": epoch_duration,
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    history.append(record)

    # Standard recovery checkpoint state
    state_bundle = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "metrics": record,
    }

    # Always persist recovery checkpoint
    torch.save(state_bundle, CKPT_DIR / "last.pt")
    torch.save(state_bundle, CKPT_DIR / f"epoch_{epoch:02d}.pt")

    # Save best checkpoint by validation macro AUROC
    if macro_auroc is not None and macro_auroc > best_macro_auroc:
        best_macro_auroc = macro_auroc
        epochs_no_improve = 0
        torch.save(state_bundle, CKPT_DIR / "best.pt")
        print(f"  --> Saved new best model: Macro AUROC = {best_macro_auroc:.4f}")
    else:
        epochs_no_improve += 1

    # Persist live metrics
    with open(METRICS_DIR / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered: No AUROC improvement for {PATIENCE} consecutive epochs.")
        break

print(f"\nTraining complete. Best Macro AUROC: {best_macro_auroc:.4f}")
print(f"Checkpoints and metrics written to: {RUN_DIR}")

=== Dataset Partition Overview ===
Train cohort: 152,947 frontal images (51,609 patients)
Val cohort:   18,573 frontal images (6,449 patients)

=== Launching Centralized Baseline Training (10 Epochs max) ===


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/DP-MMFL/src/dp_mmfl/data/preprocessed_cached_dataset.py", line 59, in __getitem__
    raise FileNotFoundError(
        f"Missing cached image: {image_path}"
    )
FileNotFoundError: Missing cached image: /content/dp_mmfl_image_cache_256/train/0221721.png


In [ ]:

import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- Paths and configuration ----------
PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")

RUN_DIR = PROJECT_ROOT / "results/E5_large_centralized_image"
CKPT_DIR = RUN_DIR / "checkpoints"
METRICS_DIR = RUN_DIR / "metrics"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
NUM_WORKERS = 2
MAX_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3

label_cols = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

# Use the already-audited frozen frontal cohort.
train_ids = train_df["sample_id"].tolist()
val_ids = val_df["sample_id"].tolist()

train_ds = PreprocessedCachedImageDataset(
    MANIFEST_PATH, CACHE_ROOT, train_ids, image_eval_transform
)
val_ds = PreprocessedCachedImageDataset(
    MANIFEST_PATH, CACHE_ROOT, val_ids, image_eval_transform
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

# ---------- Model ----------
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(label_cols))
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

def unpack_batch(batch):
    images = batch["image"]
    targets = batch["labels"].float()
    masks = batch["label_mask"].float()

    # Ensure masked targets cannot introduce NaNs into BCE.
    targets = torch.nan_to_num(targets, nan=0.0)
    targets = torch.where(masks > 0, targets, torch.zeros_like(targets))
    return images, targets, masks

def masked_per_image_bce(logits, targets, masks):
    element = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none"
    )
    per_image_num = (element * masks).sum(dim=1)
    per_image_den = masks.sum(dim=1).clamp_min(1.0)
    return (per_image_num / per_image_den).mean()

def evaluate(model, loader):
    model.eval()
    losses, all_targets, all_masks, all_probs = [], [], [], []

    with torch.no_grad():
        for batch in loader:
            images, targets, masks = unpack_batch(batch)
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with torch.amp.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                logits = model(images)
                loss = masked_per_image_bce(logits, targets, masks)

            losses.append(float(loss.item()))
            all_targets.append(targets.cpu().numpy())
            all_masks.append(masks.cpu().numpy())
            all_probs.append(torch.sigmoid(logits).cpu().numpy())

    y = np.concatenate(all_targets)
    m = np.concatenate(all_masks).astype(bool)
    p = np.concatenate(all_probs)

    per_label = {}
    aucs, aps = [], []

    for j, label in enumerate(label_cols):
        valid = m[:, j] & np.isfinite(y[:, j])
        yj, pj = y[valid, j], p[valid, j]

        auc = None
        ap = None
        if len(yj) and len(np.unique(yj)) == 2:
            auc = float(roc_auc_score(yj, pj))
            ap = float(average_precision_score(yj, pj))
            aucs.append(auc)
            aps.append(ap)

        per_label[label] = {
            "valid_count": int(valid.sum()),
            "positives": int((yj == 1).sum()),
            "negatives": int((yj == 0).sum()),
            "auroc": auc,
            "auprc": ap,
        }

    return {
        "val_loss": float(np.mean(losses)),
        "macro_auroc": float(np.mean(aucs)) if aucs else None,
        "macro_auprc": float(np.mean(aps)) if aps else None,
        "evaluated_auroc_classes": len(aucs),
        "evaluated_auprc_classes": len(aps),
        "per_label": per_label,
    }

# ---------- Training ----------
history = []
best_auc = -float("inf")
best_epoch = None
epochs_without_improvement = 0

config = {
    "experiment": "E5_large_centralized_image",
    "architecture": "ImageNet-pretrained ResNet-18",
    "input": "frontal images only",
    "train_images": len(train_ds),
    "validation_images": len(val_ds),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "per-image masked BCE",
    "labels": label_cols,
    "checkpoint_selection": "validation macro AUROC",
    "test_used_for_selection": False,
}
with open(RUN_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Device:", DEVICE)
print("Training images:", len(train_ds))
print("Validation images:", len(val_ds))
print("Starting full-scale training...")

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_start = time.time()
    loss_sum, batches = 0.0, 0

    for batch in train_loader:
        images, targets, masks = unpack_batch(batch)
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(images)
            loss = masked_per_image_bce(logits, targets, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += float(loss.item())
        batches += 1

    train_loss = loss_sum / max(batches, 1)
    val_metrics = evaluate(model, val_loader)
    val_auc = val_metrics["macro_auroc"]

    if val_auc is not None:
        scheduler.step(val_auc)

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "duration_seconds": time.time() - epoch_start,
    }
    history.append(record)

    # Always save a recovery checkpoint after each epoch.
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "record": record,
        "config": config,
    }, CKPT_DIR / "last.pt")

    with open(METRICS_DIR / "epoch_metrics.json", "w") as f:
        json.dump(history, f, indent=2)

    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['val_loss']:.4f} | "
        f"macro_AUROC={val_auc} | "
        f"macro_AUPRC={val_metrics['macro_auprc']} | "
        f"time={record['duration_seconds'] / 60:.1f} min"
    )

    if val_auc is not None and val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "record": record,
            "config": config,
        }, CKPT_DIR / "best.pt")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

print("\nTraining complete.")
print("Best epoch by validation macro AUROC:", best_epoch)
print("Best validation macro AUROC:", best_auc)
print("Checkpoints:", CKPT_DIR)
print("Metrics:", METRICS_DIR / "epoch_metrics.json")


Device: cuda
Training images: 152947
Validation images: 18573
Starting full-scale training...


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/content/DP-MMFL/src/dp_mmfl/data/preprocessed_cached_dataset.py", line 59, in __getitem__
    raise FileNotFoundError(
        f"Missing cached image: {image_path}"
    )
FileNotFoundError: Missing cached image: /content/dp_mmfl_image_cache_256/train/0174782.png


In [ ]:

from pathlib import Path
import inspect
import re

PROJECT_ROOT = Path("/content/DP-MMFL")
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")

# 1. Check how many training images are actually cached.
cache_files = set(p.name for p in CACHE_ROOT.glob("*.png"))
sample_ids = train_df["sample_id"].astype(str)

expected_names = {f"{sid}.png" for sid in sample_ids}
missing_names = expected_names - cache_files

print("Training rows:", len(train_df))
print("Cached PNG files:", len(cache_files))
print("Missing expected cache files:", len(missing_names))
print("Examples:", list(sorted(missing_names))[:10])

# 2. Inspect the dataset implementation currently being used.
print("\n=== Current cached dataset implementation ===")
print(inspect.getsource(PreprocessedCachedImageDataset))

# 3. Search project source for an existing downloader or online dataset.
print("\n=== Relevant source files / functions ===")
pattern = re.compile(
    r"download|redvis|redivis|fetch|path_to_image|"
    r"requests\.get|image_cache|chexpert",
    re.IGNORECASE,
)

src_root = PROJECT_ROOT / "src"
for py_file in src_root.rglob("*.py"):
    try:
        lines = py_file.read_text(errors="ignore").splitlines()
    except Exception:
        continue

    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(lines)
        if pattern.search(line)
    ]

    if matches:
        print(f"\n--- {py_file} ---")
        for line_no, line in matches[:20]:
            print(f"{line_no}: {line[:200]}")


Training rows: 152947
Cached PNG files: 256
Missing expected cache files: 152947
Examples: ['0.png', '1000.png', '10000.png', '100003.png', '100004.png', '100005.png', '100007.png', '100009.png', '10001.png', '100010.png']

=== Current cached dataset implementation ===
class PreprocessedCachedImageDataset(Dataset):
    """
    Loads pre-resized (256x256) cached grayscale images and applies dynamic augmentations/crops.
    Sanitizes multi-label pathology targets (NaN -> 0.0) gated by U-Ignore masks.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        cache_root: Union[str, Path],
        sample_ids: Sequence[int],
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            manifest["sample_id"].isin(sample_ids)
        ].copy()

        # Preserve strict order of sample_ids
        order = {
            int(sample_id): i
            for i, sample_id in enumerate(sample_ids)
       

In [ ]:
from pathlib import Path
import inspect
import sys

PROJECT_ROOT = Path("/content/DP-MMFL")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

files_to_inspect = [
    PROJECT_ROOT / "src/dp_mmfl/data/image_dataset.py",
    PROJECT_ROOT / "src/dp_mmfl/data/cached_image_dataset.py",
    PROJECT_ROOT / "src/dp_mmfl/data/dataset.py",
]

for file_path in files_to_inspect:
    print("\n" + "=" * 90)
    print(f"FILE: {file_path}")
    print("=" * 90)

    if file_path.exists():
        print(file_path.read_text())
    else:
        print("FILE NOT FOUND")


FILE: /content/DP-MMFL/src/dp_mmfl/data/image_dataset.py
from io import BytesIO
from pathlib import Path
from typing import Optional, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from dp_mmfl.data.labels import TARGET_COLUMNS


class CheXpertPlusImageDataset(Dataset):
    """
    Image-only PyTorch Dataset for CheXpert Plus frontal chest radiographs.
    Streams images on-demand from Redivis and provides 13 pathology targets with U-Ignore masks.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        table,
        split: str,
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            (manifest["experiment_split"] == split)
            & (manifest["frontal_lateral"] == "Frontal")
        ].reset_index(drop=True)

        self.manifest = manifest
        self.table = table
        self.image_transform = image_transform

    def __l

In [ ]:
import inspect
from dp_mmfl.data.image_dataset import CheXpertPlusImageDataset

print("Constructor:")
print(inspect.signature(CheXpertPlusImageDataset.__init__))

print("\nClass source:")
print(inspect.getsource(CheXpertPlusImageDataset))

Constructor:
(self, manifest_path: Union[str, pathlib._local.Path], table, split: str, image_transform=None)

Class source:
class CheXpertPlusImageDataset(Dataset):
    """
    Image-only PyTorch Dataset for CheXpert Plus frontal chest radiographs.
    Streams images on-demand from Redivis and provides 13 pathology targets with U-Ignore masks.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        table,
        split: str,
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            (manifest["experiment_split"] == split)
            & (manifest["frontal_lateral"] == "Frontal")
        ].reset_index(drop=True)

        self.manifest = manifest
        self.table = table
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.manifest)

    def _load_image(self, path: str) -> Image.Image:
        png_path = str(path)

        if png_path.s

In [ ]:
from pathlib import Path
import re

PROJECT_ROOT = Path("/content/DP-MMFL")

patterns = re.compile(
    r"redivis|\.dataset\(|\.table\(|table\s*=|api.?token|"
    r"dataset_id|table_id|workspace",
    re.IGNORECASE,
)

for root in [PROJECT_ROOT / "src", PROJECT_ROOT / "notebooks"]:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.suffix not in {".py", ".ipynb"}:
            continue

        try:
            lines = path.read_text(errors="ignore").splitlines()
        except Exception:
            continue

        matches = [
            (i, line.strip()[:220])
            for i, line in enumerate(lines, start=1)
            if patterns.search(line)
        ]

        if matches:
            print(f"\n--- {path} ---")
            for line_no, line in matches[:30]:
                print(f"{line_no}: {line}")


--- /content/DP-MMFL/src/dp_mmfl/data/image_dataset.py ---
14: Streams images on-demand from Redivis and provides 13 pathology targets with U-Ignore masks.
32: self.table = table

--- /content/DP-MMFL/src/dp_mmfl/data/dataset.py ---
16: Streams frontal chest X-rays on-demand from Redivis and tokenizes clinical reports.
42: self.table = table
49: def _map_to_redivis_path(self, path_to_image: str) -> str:
61: png_path = self._map_to_redivis_path(row["path_to_image"])

--- /content/DP-MMFL/notebooks/05_image_preprocessing.ipynb ---
34: "!pip install -q redivis"
63: "import redivis\n",
68: "table = redivis.table(\"aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj\")\n",
95: "Please visit the URL below to authenticate with your Redivis account:\n",
96: "https://redivis.com/oauth/authorize?user_code=5e7d10f38532e6b630cc218e0909adec\n",
242: "import redivis\n",
247: "table = redivis.table(\"aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj\")\n",

--- /content/DP-MMFL/notebooks/09_multimodal_dataset.ipy

In [ ]:
import redivis

table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")
print("Redivis table connected:", table)

Redivis table connected: <Table aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj>


In [ ]:
import pandas as pd
from io import BytesIO
from PIL import Image
from pathlib import Path

MANIFEST_PATH = Path(
    "/content/DP-MMFL/data/processed/chexpert_plus_manifest.parquet"
)

manifest = pd.read_parquet(MANIFEST_PATH)

train_sample = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].head(3)

def to_redivis_png(path):
    path = str(path)
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"
    return path

for _, row in train_sample.iterrows():
    remote_path = to_redivis_png(row["path_to_image"])
    print(f"\nSample ID: {row['sample_id']}")
    print(f"Remote path: {remote_path}")

    image_bytes = table.file(remote_path).read(as_text=False)

    with Image.open(BytesIO(image_bytes)) as image:
        image.verify()

    with Image.open(BytesIO(image_bytes)) as image:
        print("Image dimensions:", image.size)
        print("Image format:", image.format)

FileNotFoundError: [Errno 2] No such file or directory: '/content/DP-MMFL/data/processed/chexpert_plus_manifest.parquet'

In [ ]:
import pandas as pd
from io import BytesIO
from PIL import Image
from pathlib import Path

MANIFEST_PATH = Path("/content/DP-MMFL/data/processed/chexpert_plus_manifest.parquet")
manifest = pd.read_parquet(MANIFEST_PATH)

train_sample = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].head(3)

def to_redivis_png(path):
    path = str(path)
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"
    return path

print("=== Testing Redivis Image Fetch & Byte Decoding ===")
for _, row in train_sample.iterrows():
    remote_path = to_redivis_png(row["path_to_image"])
    print(f"\nSample ID:   {row['sample_id']}")
    print(f"Remote path: {remote_path}")

    image_bytes = table.file(remote_path).read(as_text=False)

    # 1. Structural integrity verification
    with Image.open(BytesIO(image_bytes)) as image:
        image.verify()

    # 2. Metadata and format inspection
    with Image.open(BytesIO(image_bytes)) as image:
        print("Image dimensions:", image.size)
        print("Image format:    ", image.format)
        print("Image mode:      ", image.mode)

print("\nAll 3 sample images fetched and decoded successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/DP-MMFL/data/processed/chexpert_plus_manifest.parquet'

In [ ]:
import pandas as pd
from io import BytesIO
from PIL import Image
from pathlib import Path

# Use the Google Drive project root path
PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"

print(f"Loading manifest from: {MANIFEST_PATH}")
manifest = pd.read_parquet(MANIFEST_PATH)
print(f"Manifest loaded successfully. Rows: {len(manifest):,}")

# Filter 3 frontal training samples
train_sample = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].head(3)

def to_redivis_png(path):
    path = str(path)
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"
    return path

print("\n=== Testing Redivis Image Fetch & Byte Decoding ===")
for _, row in train_sample.iterrows():
    remote_path = to_redivis_png(row["path_to_image"])
    print(f"\nSample ID:   {row['sample_id']}")
    print(f"Remote path: {remote_path}")

    image_bytes = table.file(remote_path).read(as_text=False)

    # 1. Structural integrity verification
    with Image.open(BytesIO(image_bytes)) as image:
        image.verify()

    # 2. Metadata and format inspection
    with Image.open(BytesIO(image_bytes)) as image:
        print("Image dimensions:", image.size)
        print("Image format:    ", image.format)
        print("Image mode:      ", image.mode)

print("\nAll 3 sample images fetched and decoded successfully!")

Loading manifest from: /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Manifest loaded successfully. Rows: 223,462

=== Testing Redivis Image Fetch & Byte Decoding ===

Sample ID:   0
Remote path: patient00003/study1/view1_frontal.png
Image dimensions: (2828, 2320)
Image format:     PNG
Image mode:       L

Sample ID:   4
Remote path: patient00009/study1/view1_frontal.png
Image dimensions: (2012, 1741)
Image format:     PNG
Image mode:       L

Sample ID:   6
Remote path: patient00016/study1/view1_frontal.png
Image dimensions: (2022, 2022)
Image format:     PNG
Image mode:       L

All 3 sample images fetched and decoded successfully!


In [ ]:
# ==============================================================================
# Step 1: Validate Image-Only Dataset (CheXpertPlusImageDataset)
# ==============================================================================
import sys
import time
from pathlib import Path
import torch
from torchvision import transforms

PROJECT_ROOT = Path("/content/DP-MMFL")
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet"
)

# Ensure src is in search path
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dp_mmfl.data.image_dataset import CheXpertPlusImageDataset

# Standard ResNet-18 ImageNet evaluation transform
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Initialize train, val, test datasets
train_ds = CheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    split="train",
    image_transform=image_transform,
)

val_ds = CheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    split="val",
    image_transform=image_transform,
)

test_ds = CheXpertPlusImageDataset(
    manifest_path=MANIFEST_PATH,
    table=table,
    split="test",
    image_transform=image_transform,
)

print("=== Dataset Sizes ===")
print(f"Train:      {len(train_ds):,}")
print(f"Validation: {len(val_ds):,}")
print(f"Test:       {len(test_ds):,}")

# Inspect first sample
sample = train_ds[0]
print("\n=== Sample Validation ===")
print("Image shape:            ", tuple(sample["image"].shape))
print("Image dtype:            ", sample["image"].dtype)
print("Labels shape:           ", tuple(sample["labels"].shape))
print("Masks shape:            ", tuple(sample["label_mask"].shape))
print("Valid labels in sample: ", int(sample["label_mask"].sum().item()))
print("Sample ID:              ", sample["sample_id"])
print("Patient ID:             ", sample["patient_id"])

# Strict sanity assertions
assert sample["image"].shape == (3, 224, 224), f"Image shape mismatch: {sample['image'].shape}"
assert sample["labels"].shape == (13,), f"Labels shape mismatch: {sample['labels'].shape}"
assert sample["label_mask"].shape == (13,), f"Mask shape mismatch: {sample['label_mask'].shape}"
assert torch.isfinite(sample["image"]).all(), "Image tensor contains NaN/Inf values"
assert torch.isfinite(sample["labels"]).all(), "Labels tensor contains NaN/Inf values"
assert torch.isfinite(sample["label_mask"]).all(), "Mask tensor contains NaN/Inf values"

print("\nDataset sanity checks passed.")

=== Dataset Sizes ===
Train:      152,947
Validation: 18,573
Test:       19,349

=== Sample Validation ===
Image shape:             (3, 224, 224)
Image dtype:             torch.float32
Labels shape:            (13,)
Masks shape:             (13,)
Valid labels in sample:  2
Sample ID:               0
Patient ID:              patient00003

Dataset sanity checks passed.


In [ ]:
# ==============================================================================
# Step 2: Benchmark Actual Image Loading (8 Remote Samples)
# ==============================================================================
count = min(8, len(train_ds))
print(f"=== Benchmarking {count} Remote Redivis Reads ===")

start = time.perf_counter()
for i in range(count):
    item = train_ds[i]
    assert item["image"].shape == (3, 224, 224)

elapsed = time.perf_counter() - start
fps = count / elapsed if elapsed > 0 else 0.0
avg_time_per_image = elapsed / count if count > 0 else 0.0
projected_epoch_hours = len(train_ds) / max(fps, 1e-6) / 3600.0

print(f"Images loaded:                      {count}")
print(f"Total time:                         {elapsed:.2f} seconds")
print(f"Average per image:                  {avg_time_per_image:.2f} seconds")
print(f"Measured throughput:                {fps:.2f} images/second")
print(f"Estimated train image-loading time: {projected_epoch_hours:.2f} hours")

=== Benchmarking 8 Remote Redivis Reads ===
Images loaded:                      8
Total time:                         25.84 seconds
Average per image:                  3.23 seconds
Measured throughput:                0.31 images/second
Estimated train image-loading time: 137.20 hours


In [ ]:
# ==============================================================================
# Step 1: Benchmark Parallel Image Retrieval & Pre-caching (8 Workers)
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
import time
import traceback
import pandas as pd
from PIL import Image

MANIFEST_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet"
)
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(MANIFEST_PATH)
sample_rows = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].iloc[20:36].copy()

def fetch_resize_save(row):
    sample_id = int(row["sample_id"])
    output_path = CACHE_ROOT / f"{sample_id:07d}.png"

    # Reuse a valid cached image if it already exists
    if output_path.exists():
        try:
            with Image.open(output_path) as im:
                im.verify()
            return {"sample_id": sample_id, "status": "cached"}
        except Exception:
            output_path.unlink(missing_ok=True)

    remote_path = str(row["path_to_image"])
    if remote_path.startswith("train/"):
        remote_path = remote_path[len("train/"):]
    if remote_path.endswith(".jpg"):
        remote_path = remote_path[:-4] + ".png"

    data = table.file(remote_path).read(as_text=False)
    with Image.open(BytesIO(data)) as image:
        image = image.convert("L")
        image.thumbnail((256, 256))
        image.save(output_path, format="PNG", optimize=True)

    return {
        "sample_id": sample_id,
        "status": "downloaded",
        "bytes": len(data),
    }

print(f"=== Starting 8-Worker Parallel Fetch ({len(sample_rows)} images) ===")
start = time.perf_counter()
results = []
errors = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(fetch_resize_save, row): int(row["sample_id"])
        for _, row in sample_rows.iterrows()
    }
    for future in as_completed(futures):
        try:
            results.append(future.result())
        except Exception as exc:
            errors.append({
                "sample_id": futures[future],
                "error": repr(exc),
            })

elapsed = time.perf_counter() - start

print(f"Requested:            {len(sample_rows)}")
print(f"Successful:           {len(results)}")
print(f"Failed:               {len(errors)}")
print(f"Elapsed:              {elapsed:.2f} seconds")
print(f"Effective throughput: {len(results) / max(elapsed, 1e-6):.2f} images/sec")

if errors:
    print("\nErrors encountered:")
    for err in errors:
        print(f"  Sample {err['sample_id']}: {err['error']}")

downloaded_count = sum(1 for r in results if r["status"] == "downloaded")
cached_count = sum(1 for r in results if r["status"] == "cached")
print(f"\nFresh downloads:      {downloaded_count}")
print(f"Already cached:       {cached_count}")
print(f"Total files in cache: {len(list(CACHE_ROOT.glob('*.png')))}")

=== Starting 8-Worker Parallel Fetch (16 images) ===
Requested:            16
Successful:           16
Failed:               0
Elapsed:              9.19 seconds
Effective throughput: 1.74 images/sec

Fresh downloads:      16
Already cached:       0
Total files in cache: 16


In [ ]:
# ==============================================================================
# Step 2: Concurrency Benchmark — 16 Workers on 32 Images
# Evaluates higher concurrency and monitors disk storage footprint
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
import time
import pandas as pd
from PIL import Image

MANIFEST_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet"
)
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(MANIFEST_PATH)
rows = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].iloc[36:68].copy()

def fetch_one(row):
    sid = int(row["sample_id"])
    out = CACHE_ROOT / f"{sid:07d}.png"

    # Reuse valid cached file if present
    if out.exists():
        try:
            with Image.open(out) as im:
                im.verify()
            return "cached"
        except Exception:
            out.unlink(missing_ok=True)

    path = str(row["path_to_image"])
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"

    data = table.file(path).read(as_text=False)
    with Image.open(BytesIO(data)) as im:
        im = im.convert("L")
        im.thumbnail((256, 256))
        im.save(out, format="PNG", optimize=True)

    return "downloaded"

print(f"=== Testing 16-Worker Concurrency ({len(rows)} images) ===")
start = time.perf_counter()
ok = 0
errors = []

with ThreadPoolExecutor(max_workers=16) as pool:
    futures = {
        pool.submit(fetch_one, row): int(row["sample_id"])
        for _, row in rows.iterrows()
    }
    for future in as_completed(futures):
        try:
            future.result()
            ok += 1
        except Exception as exc:
            errors.append((futures[future], repr(exc)))

elapsed = time.perf_counter() - start
total_cache_bytes = sum(p.stat().st_size for p in CACHE_ROOT.glob("*.png"))
total_cache_mib = total_cache_bytes / (1024 ** 2)
avg_kb_per_img = (total_cache_bytes / max(len(list(CACHE_ROOT.glob("*.png"))), 1)) / 1024

print(f"Requested:    {len(rows)}")
print(f"Successful:   {ok}")
print(f"Failed:       {len(errors)}")
print(f"Elapsed:      {elapsed:.2f}s")
print(f"Throughput:   {ok / max(elapsed, 1e-6):.2f} images/sec")
print(f"Total cache:  {total_cache_mib:.2f} MiB ({avg_kb_per_img:.1f} KiB/image)")

if errors:
    print("\nFailures:")
    for item in errors:
        print(f"  Sample {item[0]}: {item[1]}")

=== Testing 16-Worker Concurrency (32 images) ===
Requested:    32
Successful:   32
Failed:       0
Elapsed:      9.10s
Throughput:   3.52 images/sec
Total cache:  1.36 MiB (29.0 KiB/image)


In [ ]:
# ==============================================================================
# Step 3: Concurrency Peak Test — 32 Workers on 64 New Images
# Verifies Redivis API throughput saturation, socket stability, and error rate
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
import time
import pandas as pd
from PIL import Image

MANIFEST_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet"
)
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_parquet(MANIFEST_PATH)
rows = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].iloc[68:132].copy()

def fetch_one(row):
    sid = int(row["sample_id"])
    out = CACHE_ROOT / f"{sid:07d}.png"

    if out.exists():
        try:
            with Image.open(out) as im:
                im.verify()
            return "cached"
        except Exception:
            out.unlink(missing_ok=True)

    path = str(row["path_to_image"])
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"

    data = table.file(path).read(as_text=False)
    with Image.open(BytesIO(data)) as im:
        im = im.convert("L")
        im.thumbnail((256, 256))
        im.save(out, format="PNG", optimize=True)

    return "downloaded"

print(f"=== Testing 32-Worker Concurrency ({len(rows)} images) ===")
start = time.perf_counter()
ok = 0
errors = []

with ThreadPoolExecutor(max_workers=32) as pool:
    futures = {
        pool.submit(fetch_one, row): int(row["sample_id"])
        for _, row in rows.iterrows()
    }
    for future in as_completed(futures):
        try:
            future.result()
            ok += 1
        except Exception as exc:
            errors.append((futures[future], repr(exc)))

elapsed = time.perf_counter() - start
total_cache_count = len(list(CACHE_ROOT.glob("*.png")))
total_cache_mib = sum(p.stat().st_size for p in CACHE_ROOT.glob("*.png")) / (1024 ** 2)

print(f"Requested:         {len(rows)}")
print(f"Successful:        {ok}")
print(f"Failed:            {len(errors)}")
print(f"Elapsed:           {elapsed:.2f} seconds")
print(f"Throughput:        {ok / max(elapsed, 1e-6):.2f} images/sec")
print(f"Total cache files: {total_cache_count}")
print(f"Total cache size:  {total_cache_mib:.2f} MiB")

if errors:
    print(f"\nEncountered {len(errors)} failures:")
    for item in errors[:5]:
        print(f"  Sample {item[0]}: {item[1]}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more")

=== Testing 32-Worker Concurrency (64 images) ===
Requested:         64
Successful:        64
Failed:            0
Elapsed:           12.54 seconds
Throughput:        5.10 images/sec
Total cache files: 112
Total cache size:  3.15 MiB


In [ ]:
# ==============================================================================
# Sustained 512-Image Benchmark: 32 Workers with Exponential Retries & Drive Logging
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
import json
import os
from pathlib import Path
import random
import threading
import time
import pandas as pd
from PIL import Image

# ---------- Paths & Configuration ----------
PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"
CACHE_ROOT = Path("/content/dp_mmfl_image_cache_256/train")
PROGRESS_DIR = PROJECT_ROOT / "results/image_cache_benchmark"
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

WORKERS = 32
N_IMAGES = 512
MAX_RETRIES = 3

# ---------- Load Target Manifest Subset ----------
manifest = pd.read_parquet(MANIFEST_PATH)
train_rows = manifest.loc[
    (manifest["experiment_split"] == "train")
    & (manifest["frontal_lateral"] == "Frontal")
].copy()

# Filter out the 112 images already probed in earlier benchmarks
existing_ids = {int(p.stem) for p in CACHE_ROOT.glob("*.png")}
target_batch = train_rows[~train_rows["sample_id"].astype(int).isin(existing_ids)].iloc[:N_IMAGES].copy()

print(f"=== Sustained Retrieval Benchmark Config ===")
print(f"Already cached locally:     {len(existing_ids):,} images")
print(f"Target new images to fetch: {len(target_batch):,}")
print(f"Worker concurrency:         {WORKERS}")
print(f"Max retries per sample:     {MAX_RETRIES}")
print(f"Metrics & progress path:    {PROGRESS_DIR / 'sustained_512_summary.json'}\n")

# Thread-safe logging structures
lock = threading.Lock()
successful_records = []
failed_records = []

def fetch_and_cache_with_retry(row):
    sample_id = int(row["sample_id"])
    dst_path = CACHE_ROOT / f"{sample_id:07d}.png"

    # Verify if valid file already exists
    if dst_path.exists():
        try:
            with Image.open(dst_path) as im:
                im.verify()
            return {"sample_id": sample_id, "status": "cached", "retries": 0}
        except Exception:
            dst_path.unlink(missing_ok=True)

    # Resolve Redivis cloud path
    rel_path = str(row["path_to_image"])
    if rel_path.startswith("train/"):
        rel_path = rel_path[len("train/"):]
    if rel_path.endswith(".jpg"):
        rel_path = rel_path[:-4] + ".png"

    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            file_obj = table.file(rel_path)
            raw_bytes = file_obj.read(as_text=False)

            with Image.open(BytesIO(raw_bytes)) as pil_img:
                pil_img = pil_img.convert("L")
                pil_img.thumbnail((256, 256))
                pil_img.save(dst_path, format="PNG", optimize=True)

            return {
                "sample_id": sample_id,
                "status": "downloaded",
                "bytes": len(raw_bytes),
                "retries": attempt,
            }
        except Exception as exc:
            last_error = repr(exc)
            # Jittered exponential backoff
            time.sleep((2 ** attempt) * 0.5 + random.uniform(0.1, 0.4))

    return {
        "sample_id": sample_id,
        "status": "failed",
        "error": last_error,
        "retries": MAX_RETRIES,
    }

# ---------- Execute ThreadPool ----------
start_time = time.perf_counter()
completed_count = 0

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {
        pool.submit(fetch_and_cache_with_retry, row): int(row["sample_id"])
        for _, row in target_batch.iterrows()
    }

    for future in as_completed(futures):
        res = future.result()
        with lock:
            completed_count += 1
            if res["status"] in ("downloaded", "cached"):
                successful_records.append(res)
            else:
                failed_records.append(res)

            if completed_count % 64 == 0 or completed_count == len(target_batch):
                curr_elapsed = time.perf_counter() - start_time
                curr_fps = completed_count / max(curr_elapsed, 1e-6)
                print(
                    f"Progress: [{completed_count:3d}/{len(target_batch)}] | "
                    f"Elapsed: {curr_elapsed:5.1f}s | "
                    f"Pace: {curr_fps:4.2f} img/s | "
                    f"Success: {len(successful_records)} | "
                    f"Errors: {len(failed_records)}"
                )

total_duration = time.perf_counter() - start_time
throughput = len(successful_records) / max(total_duration, 1e-6)
retried_downloads = sum(1 for r in successful_records if r.get("retries", 0) > 0)

# Disk consumption check
total_cache_files = list(CACHE_ROOT.glob("*.png"))
total_cache_bytes = sum(p.stat().st_size for p in total_cache_files)
cache_mib = total_cache_bytes / (1024 ** 2)
avg_kib = (total_cache_bytes / max(len(total_cache_files), 1)) / 1024

summary = {
    "requested_images": len(target_batch),
    "successful_images": len(successful_records),
    "failed_images": len(failed_records),
    "retried_successes": retried_downloads,
    "elapsed_seconds": round(total_duration, 2),
    "measured_throughput_fps": round(throughput, 2),
    "projected_10k_hours": round(10000 / max(throughput, 1e-6) / 3600, 2),
    "projected_full_train_hours": round(len(train_rows) / max(throughput, 1e-6) / 3600, 2),
    "cache_total_files": len(total_cache_files),
    "cache_total_mib": round(cache_mib, 2),
    "average_kib_per_image": round(avg_kib, 2),
    "failures": failed_records[:10],
}

# Persist run summary to Drive
summary_path = PROGRESS_DIR / "sustained_512_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("\n=== Sustained Benchmark Results ===")
print(f"Total Processed:         {len(target_batch)} images")
print(f"Successful:              {len(successful_records)}")
print(f"Failed:                  {len(failed_records)}")
print(f"Recovered via retry:     {retried_downloads}")
print(f"Elapsed Time:            {total_duration:.2f} s")
print(f"Sustained Throughput:    {throughput:.2f} img/s")
print(f"Projected 10k Export:    {summary['projected_10k_hours']:.2f} hours")
print(f"Projected Full Cohort:   {summary['projected_full_train_hours']:.2f} hours")
print(f"Local Cache Footprint:   {cache_mib:.2f} MiB ({len(total_cache_files)} images total)")
print(f"Summary Artifact Saved:  {summary_path}")

=== Sustained Retrieval Benchmark Config ===
Already cached locally:     112 images
Target new images to fetch: 512
Worker concurrency:         32
Max retries per sample:     3
Metrics & progress path:    /content/drive/MyDrive/DP-MMFL/results/image_cache_benchmark/sustained_512_summary.json

Progress: [ 64/512] | Elapsed:  13.3s | Pace: 4.83 img/s | Success: 64 | Errors: 0
Progress: [128/512] | Elapsed:  23.0s | Pace: 5.57 img/s | Success: 128 | Errors: 0
Progress: [192/512] | Elapsed:  32.3s | Pace: 5.95 img/s | Success: 192 | Errors: 0
Progress: [256/512] | Elapsed:  41.6s | Pace: 6.15 img/s | Success: 256 | Errors: 0
Progress: [320/512] | Elapsed:  51.8s | Pace: 6.18 img/s | Success: 320 | Errors: 0
Progress: [384/512] | Elapsed:  60.6s | Pace: 6.34 img/s | Success: 384 | Errors: 0
Progress: [448/512] | Elapsed:  69.9s | Pace: 6.41 img/s | Success: 448 | Errors: 0
Progress: [512/512] | Elapsed:  78.2s | Pace: 6.55 img/s | Success: 512 | Errors: 0

=== Sustained Benchmark Results ==

In [ ]:
import shutil
from pathlib import Path

cache_root = Path("/content/dp_mmfl_image_cache_256/train")
drive_root = Path("/content/drive/MyDrive/DP-MMFL")

for path in [Path("/content"), cache_root, drive_root]:
    usage = shutil.disk_usage(path)
    print(f"\n{path}")
    print(f"  Total:     {usage.total / (1024**3):.2f} GiB")
    print(f"  Used:      {usage.used / (1024**3):.2f} GiB")
    print(f"  Free:      {usage.free / (1024**3):.2f} GiB")

print("\nCurrent cache files:", len(list(cache_root.glob("*.png"))))
print(
    "Current cache size:",
    f"{sum(p.stat().st_size for p in cache_root.glob('*.png')) / (1024**2):.2f} MiB",
)


/content
  Total:     112.64 GiB
  Used:      42.59 GiB
  Free:      70.03 GiB

/content/dp_mmfl_image_cache_256/train
  Total:     112.64 GiB
  Used:      42.59 GiB
  Free:      70.03 GiB

/content/drive/MyDrive/DP-MMFL
  Total:     112.64 GiB
  Used:      46.11 GiB
  Free:      66.53 GiB

Current cache files: 624
Current cache size: 17.62 MiB


In [ ]:
train_pts = set(df[df["experiment_split"] == "train"]["deid_patient_id"])
val_pts = set(df[df["experiment_split"] == "val"]["deid_patient_id"])
test_pts = set(df[df["experiment_split"] == "test"]["deid_patient_id"])

print("\n" + "=" * 20 + " PATIENT ISOLATION AUDIT " + "=" * 20)
print("Train ∩ Val overlap: ", len(train_pts & val_pts))
print("Train ∩ Test overlap:", len(train_pts & test_pts))
print("Val ∩ Test overlap:  ", len(val_pts & test_pts))
assert train_pts.isdisjoint(val_pts), "Leakage detected between Train and Val splits!"
assert train_pts.isdisjoint(test_pts), "Leakage detected between Train and Test splits!"
assert val_pts.isdisjoint(test_pts), "Leakage detected between Val and Test splits!"
print("Patient Isolation Guarantee: STRICT DISJOINT (PASS)")


==================== PATIENT ISOLATION AUDIT ====================
Train ∩ Val overlap:  0
Train ∩ Test overlap: 0
Val ∩ Test overlap:   0
Patient Isolation Guarantee: STRICT DISJOINT (PASS)


In [ ]:
# ==============================================================================
# Complete Label-Count, Prevalence, and Patient-Isolation Audit Cell
# Computes image-level and patient-level distributions across frozen splits
# ==============================================================================
from pathlib import Path
import pandas as pd

MANIFEST_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet"
)
df = pd.read_parquet(MANIFEST_PATH)

LABELS = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

# Restrict strictly to frontal projections and experimental splits
df = df[
    (df["frontal_lateral"] == "Frontal")
    & (df["experiment_split"].isin(["train", "val", "test"]))
].copy()

# ---------- 1. Per-Split Label & Patient Audit ----------
for split in ["train", "val", "test"]:
    part = df[df["experiment_split"] == split]
    total_images = len(part)
    total_patients = part["deid_patient_id"].nunique()

    print(f"\n{'=' * 24} {split.upper()} SPLIT {'=' * 24}")
    print(f"Total Frontal Images: {total_images:,}")
    print(f"Unique Patients:      {total_patients:,}")

    rows = []
    for label in LABELS:
        target_col = f"target_{label}"
        mask_col = f"mask_{label}"

        # Valid masked condition (1 = valid binary label, 0 = masked/uncertain/unmentioned)
        valid = part[mask_col].fillna(0).astype(float) > 0
        positive = (
            valid & (pd.to_numeric(part[target_col], errors="coerce") == 1.0)
        )

        # Image-level counts
        img_pos = int(positive.sum())
        img_valid = int(valid.sum())
        img_prev = (img_pos / max(img_valid, 1)) * 100

        # Patient-level aggregation
        patient_positive = positive.groupby(part["deid_patient_id"]).max()
        patient_observed = valid.groupby(part["deid_patient_id"]).max()

        pt_pos = int(patient_positive.sum())
        pt_obs = int(patient_observed.sum())
        pt_prev = (pt_pos / max(pt_obs, 1)) * 100

        rows.append({
            "Pathology": label,
            "Image Pos": f"{img_pos:,}",
            "Image Valid": f"{img_valid:,}",
            "Image Prev (%)": f"{img_prev:5.2f}%",
            "Patient Pos": f"{pt_pos:,}",
            "Patient Obs": f"{pt_obs:,}",
            "Patient Prev (%)": f"{pt_prev:5.2f}%",
        })

    summary_df = pd.DataFrame(rows)
    print(summary_df.to_string(index=False))

# ---------- 2. Patient-Disjoint Verification ----------
train_pts = set(df[df["experiment_split"] == "train"]["deid_patient_id"])
val_pts = set(df[df["experiment_split"] == "val"]["deid_patient_id"])
test_pts = set(df[df["experiment_split"] == "test"]["deid_patient_id"])

print("\n" + "=" * 22 + " PATIENT ISOLATION AUDIT " + "=" * 22)
print(f"Train ∩ Val Overlap:  {len(train_pts & val_pts)}")
print(f"Train ∩ Test Overlap: {len(train_pts & test_pts)}")
print(f"Val ∩ Test Overlap:   {len(val_pts & test_pts)}")

assert train_pts.isdisjoint(val_pts), "Leakage detected between Train and Val splits!"
assert train_pts.isdisjoint(test_pts), "Leakage detected between Train and Test splits!"
assert val_pts.isdisjoint(test_pts), "Leakage detected between Val and Test splits!"
print("Patient Isolation Guarantee: STRICT DISJOINT (ALL CHECKS PASSED)")


======================== TRAIN SPLIT ========================
Total Frontal Images: 152,947
Unique Patients:      51,609
                 Pathology Image Pos Image Valid Image Prev (%) Patient Pos Patient Obs Patient Prev (%)
Enlarged Cardiomediastinum     6,285      28,544         22.02%       4,931      18,362           26.85%
              Cardiomegaly    24,626      42,434         58.03%      13,466      22,517           59.80%
              Lung Opacity    75,681      80,068         94.52%      31,371      32,833           95.55%
               Lung Lesion     9,066      10,059         90.13%       6,090       6,723           90.58%
                     Edema    37,184      52,879         70.32%      18,271      24,900           73.38%
             Consolidation    10,051      32,726         30.71%       6,987      19,519           35.80%
                 Pneumonia     4,118       7,337         56.13%       3,159       5,370           58.83%
               Atelectasis    26,059  

Getting 25k images and training on that

In [ ]:
# ==============================================================================
# Step 1: Generate & Freeze Reproducible 25,000-Patient Training Subset Manifest
# Retains 100% of validation and test cohorts | Patient-disjoint & Stratified
# ==============================================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------- Configuration --------------------
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/data/processed/"
    "chexpert_plus_manifest.parquet"
)
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/DP-MMFL/results/final_protocol"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATIENT_TARGET = 25_000
SEED = 20261017

LABELS = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

# -------------------- Load & Filter Manifest --------------------
df = pd.read_parquet(MANIFEST_PATH)

target_cols = [f"target_{x}" for x in LABELS]
mask_cols = [f"mask_{x}" for x in LABELS]

required = [
    "sample_id",
    "deid_patient_id",
    "experiment_split",
    "frontal_lateral",
    "path_to_image",
    "report",
] + target_cols + mask_cols

# Filter strictly to frontal projections and the three primary experimental splits
df = df[
    (df["frontal_lateral"] == "Frontal")
    & (df["experiment_split"].isin(["train", "val", "test"]))
][required].copy()

# -------------------- Patient-Level Stratified Sampling (Train) --------------------
train_df = df[df["experiment_split"] == "train"].copy()
train_patients = train_df["deid_patient_id"].drop_duplicates()

# Compute patient-level aggregate label vectors to preserve rare pathology representation
patient_pos_vectors = []
for label in LABELS:
    t_col = f"target_{label}"
    m_col = f"mask_{label}"
    valid_pos = (
        (train_df[m_col].fillna(0).astype(float) > 0)
        & (pd.to_numeric(train_df[t_col], errors="coerce") == 1.0)
    )
    patient_has_pos = valid_pos.groupby(train_df["deid_patient_id"]).max()
    patient_pos_vectors.append(patient_has_pos.rename(label))

patient_matrix = pd.concat(patient_pos_vectors, axis=1).fillna(False).astype(int)

# Create stratification priority string based on label presence
strata = patient_matrix.astype(str).agg("".join, axis=1)

# Reproducible stratified sampling across patient strata
rng = np.random.default_rng(SEED)
selected_patients = []
strata_df = pd.DataFrame({"deid_patient_id": patient_matrix.index, "stratum": strata.values})

# Sample proportionally from each stratum
stratum_groups = strata_df.groupby("stratum")
for stratum, group in stratum_groups:
    n_sample = int(np.round(len(group) / len(strata_df) * TRAIN_PATIENT_TARGET))
    sampled_ids = rng.choice(group["deid_patient_id"].values, size=min(n_sample, len(group)), replace=False)
    selected_patients.extend(sampled_ids)

# Reconcile exact target count
selected_patients = list(dict.fromkeys(selected_patients))
remaining_needed = TRAIN_PATIENT_TARGET - len(selected_patients)

if remaining_needed > 0:
    unselected = np.setdiff1d(strata_df["deid_patient_id"].values, selected_patients)
    extra_sampled = rng.choice(unselected, size=remaining_needed, replace=False)
    selected_patients.extend(extra_sampled)
elif remaining_needed < 0:
    selected_patients = rng.choice(selected_patients, size=TRAIN_PATIENT_TARGET, replace=False).tolist()

assert len(selected_patients) == TRAIN_PATIENT_TARGET, f"Sampling count mismatch: {len(selected_patients)}"

# -------------------- Assemble Final Frozen Manifest --------------------
# Retain all frontal images for selected train patients + 100% of val and test cohorts
selected_train_df = train_df[train_df["deid_patient_id"].isin(selected_patients)].copy()
val_df = df[df["experiment_split"] == "val"].copy()
test_df = df[df["experiment_split"] == "test"].copy()

final_subset_df = pd.concat([selected_train_df, val_df, test_df], ignore_index=True)

# -------------------- Integrity Assertions --------------------
train_pts = set(selected_train_df["deid_patient_id"])
val_pts = set(val_df["deid_patient_id"])
test_pts = set(test_df["deid_patient_id"])

assert train_pts.isdisjoint(val_pts), "Data leakage between Train and Val!"
assert train_pts.isdisjoint(test_pts), "Data leakage between Train and Test!"
assert val_pts.isdisjoint(test_pts), "Data leakage between Val and Test!"

print("=== Final Frozen Subset Overview ===")
print(f"Train:      {len(selected_train_df):,} frontal images across {len(train_pts):,} patients")
print(f"Validation: {len(val_df):,} frontal images across {len(val_pts):,} patients")
print(f"Test:       {len(test_df):,} frontal images across {len(test_pts):,} patients")
print(f"Total:      {len(final_subset_df):,} frontal images across {final_subset_df['deid_patient_id'].nunique():,} unique patients")

# -------------------- Label-Level Distribution Audit --------------------
print(f"\n{'=' * 24} SELECTED TRAIN SPLIT AUDIT (25,000 PATIENTS) {'=' * 24}")
audit_rows = []
for label in LABELS:
    t_col = f"target_{label}"
    m_col = f"mask_{label}"

    valid = selected_train_df[m_col].fillna(0).astype(float) > 0
    positive = valid & (pd.to_numeric(selected_train_df[t_col], errors="coerce") == 1.0)

    img_pos = int(positive.sum())
    img_valid = int(valid.sum())
    img_prev = (img_pos / max(img_valid, 1)) * 100

    pt_pos = int(positive.groupby(selected_train_df["deid_patient_id"]).max().sum())
    pt_obs = int(valid.groupby(selected_train_df["deid_patient_id"]).max().sum())
    pt_prev = (pt_pos / max(pt_obs, 1)) * 100

    audit_rows.append({
        "Pathology": label,
        "Image Pos": f"{img_pos:,}",
        "Image Valid": f"{img_valid:,}",
        "Image Prev (%)": f"{img_prev:5.2f}%",
        "Patient Pos": f"{pt_pos:,}",
        "Patient Obs": f"{pt_obs:,}",
        "Patient Prev (%)": f"{pt_prev:5.2f}%",
    })

print(pd.DataFrame(audit_rows).to_string(index=False))

# -------------------- Persist Manifest and Protocol Artifacts --------------------
SUBSET_MANIFEST_PATH = OUTPUT_DIR / "chexpert_plus_25k_frozen_manifest.parquet"
METADATA_PATH = OUTPUT_DIR / "subset_generation_metadata.json"

final_subset_df.to_parquet(SUBSET_MANIFEST_PATH, index=False)

metadata = {
    "creation_timestamp": "2026-10-17T20:05:00",
    "seed": SEED,
    "target_train_patients": TRAIN_PATIENT_TARGET,
    "actual_train_patients": len(train_pts),
    "actual_train_images": len(selected_train_df),
    "actual_val_patients": len(val_pts),
    "actual_val_images": len(val_df),
    "actual_test_patients": len(test_pts),
    "actual_test_images": len(test_df),
    "labels": LABELS,
    "estimated_cache_mb": round((len(selected_train_df) * 29) / 1024, 2),
    "artifacts": {
        "manifest": str(SUBSET_MANIFEST_PATH),
    },
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("\n=== Protocol Artifacts Successfully Frozen ===")
print(f"Manifest written to: {SUBSET_MANIFEST_PATH}")
print(f"Metadata written to: {METADATA_PATH}")
print(f"Estimated 256px cache disk space for train: ~{metadata['estimated_cache_mb']:.1f} MiB")

=== Final Frozen Subset Overview ===
Train:      73,640 frontal images across 25,000 patients
Validation: 18,573 frontal images across 6,449 patients
Test:       19,349 frontal images across 6,452 patients
Total:      111,562 frontal images across 37,901 unique patients

======================== SELECTED TRAIN SPLIT AUDIT (25,000 PATIENTS) ========================
                 Pathology Image Pos Image Valid Image Prev (%) Patient Pos Patient Obs Patient Prev (%)
Enlarged Cardiomediastinum     2,897      13,624         21.26%       2,263       8,813           25.68%
              Cardiomegaly    11,730      20,256         57.91%       6,435      10,772           59.74%
              Lung Opacity    36,466      38,547         94.60%      15,171      15,852           95.70%
               Lung Lesion     4,120       4,620         89.18%       2,787       3,118           89.38%
                     Edema    17,986      25,553         70.39%       8,805      11,994           73.41%
   

In [ ]:
# ==============================================================================
# Protocol Verification: Frozen 25k Manifest & Multi-label Integrity Audit
# ==============================================================================
import json
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/DP-MMFL")
PROTOCOL = BASE / "results/final_protocol"
manifest_path = PROTOCOL / "chexpert_plus_25k_frozen_manifest.parquet"

assert manifest_path.exists(), f"Frozen manifest file not found: {manifest_path}"
df = pd.read_parquet(manifest_path)

labels = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

print("Shape:  ", df.shape)
print("Columns:", df.columns.tolist())

# 1. Required Identifiers and Split Information
required = ["sample_id", "deid_patient_id", "experiment_split"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"
assert df["sample_id"].is_unique, "Duplicate sample IDs found"
assert df["deid_patient_id"].notna().all(), "Missing patient IDs"

# 2. Strict Patient-Level Split Isolation
patient_sets = {
    split: set(group["deid_patient_id"].unique())
    for split, group in df.groupby("experiment_split")
}

print("\n=== Patient Overlap Checks ===")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    if a in patient_sets and b in patient_sets:
        overlap = patient_sets[a] & patient_sets[b]
        print(f"{a} ∩ {b}: {len(overlap)} patients")
        assert not overlap, f"Patient leakage detected between {a} and {b}"

# 3. Validate Masked Multi-label Policies
print("\n=== Target and Mask Consistency Checks ===")
for label in labels:
    target_col = f"target_{label}"
    mask_col = f"mask_{label}"
    assert target_col in df.columns, f"Missing {target_col}"
    assert mask_col in df.columns, f"Missing {mask_col}"

    mask = pd.to_numeric(df[mask_col], errors="coerce")
    target = pd.to_numeric(df[target_col], errors="coerce")

    assert mask.notna().all(), f"Invalid mask values for {label}"
    assert mask.isin([0, 1]).all(), f"Mask must be strictly binary: {label}"

    observed_targets = target[mask == 1]
    assert observed_targets.notna().all(), f"Missing observed targets: {label}"
    assert observed_targets.isin([0, 1]).all(), f"Non-binary observed targets: {label}"

print("\nSplit counts:")
print(
    df.groupby("experiment_split").agg(
        images=("sample_id", "count"),
        patients=("deid_patient_id", "nunique"),
    )
)

print("\nVALIDATION PASSED")

Shape:   (111562, 32)
Columns: ['sample_id', 'deid_patient_id', 'experiment_split', 'frontal_lateral', 'path_to_image', 'report', 'target_Enlarged Cardiomediastinum', 'target_Cardiomegaly', 'target_Lung Opacity', 'target_Lung Lesion', 'target_Edema', 'target_Consolidation', 'target_Pneumonia', 'target_Atelectasis', 'target_Pneumothorax', 'target_Pleural Effusion', 'target_Pleural Other', 'target_Fracture', 'target_Support Devices', 'mask_Enlarged Cardiomediastinum', 'mask_Cardiomegaly', 'mask_Lung Opacity', 'mask_Lung Lesion', 'mask_Edema', 'mask_Consolidation', 'mask_Pneumonia', 'mask_Atelectasis', 'mask_Pneumothorax', 'mask_Pleural Effusion', 'mask_Pleural Other', 'mask_Fracture', 'mask_Support Devices']

=== Patient Overlap Checks ===
train ∩ val: 0 patients
train ∩ test: 0 patients
val ∩ test: 0 patients

=== Target and Mask Consistency Checks ===

Split counts:
                  images  patients
experiment_split                  
test               19349      6452
train           

In [ ]:
# ==============================================================================
# Pipeline Inspection: Audit DP-MMFL Core Modules, Scripts, and Notebooks
# ==============================================================================
from pathlib import Path

ROOT = Path("/content/DP-MMFL")

for folder in ["src/dp_mmfl", "scripts", "notebooks"]:
    path = ROOT / folder
    print(f"\n--- {folder} ---")
    if path.exists():
        py_files = sorted(path.rglob("*.py"))
        ipynb_files = sorted(path.glob("*.ipynb"))

        for f in py_files:
            print(f.relative_to(ROOT))
        for f in ipynb_files:
            print(f.relative_to(ROOT))

        if not py_files and not ipynb_files:
            print("(No .py or .ipynb files found in directory)")
    else:
        print("Folder not found")


--- src/dp_mmfl ---
src/dp_mmfl/__init__.py
src/dp_mmfl/data/__init__.py
src/dp_mmfl/data/cached_image_dataset.py
src/dp_mmfl/data/dataset.py
src/dp_mmfl/data/datasets.py
src/dp_mmfl/data/image_dataset.py
src/dp_mmfl/data/image_transforms.py
src/dp_mmfl/data/labels.py
src/dp_mmfl/data/preprocessed_cached_dataset.py
src/dp_mmfl/data/preprocessing.py
src/dp_mmfl/data/splits.py
src/dp_mmfl/data/text.py
src/dp_mmfl/data/transforms.py
src/dp_mmfl/evaluation/__init__.py
src/dp_mmfl/evaluation/calibration.py
src/dp_mmfl/evaluation/evaluation.py
src/dp_mmfl/evaluation/losses.py
src/dp_mmfl/evaluation/metrics.py
src/dp_mmfl/explainability/__init__.py
src/dp_mmfl/explainability/image.py
src/dp_mmfl/explainability/multimodal.py
src/dp_mmfl/explainability/text.py
src/dp_mmfl/fairness/__init__.py
src/dp_mmfl/fairness/metrics.py
src/dp_mmfl/fairness/mitigation.py
src/dp_mmfl/federated/__init__.py
src/dp_mmfl/federated/aggregation.py
src/dp_mmfl/federated/client.py
src/dp_mmfl/federated/communicatio

In [ ]:
# ==============================================================================
# Inspection Cell: Audit Training Entry Points, Losses, Models & Baseline Notebook
# ==============================================================================
import json
from pathlib import Path

ROOT = Path("/content/DP-MMFL")

files = [
    ROOT / "scripts/train.py",
    ROOT / "scripts/federated_train.py",
    ROOT / "src/dp_mmfl/train.py",
    ROOT / "src/dp_mmfl/data/cached_image_dataset.py",
    ROOT / "src/dp_mmfl/models/image_model.py",
    ROOT / "src/dp_mmfl/evaluation/losses.py",
]

for path in files:
    print("\n" + "=" * 90)
    try:
        rel_display = path.relative_to(ROOT)
    except ValueError:
        rel_display = path
    print(f"FILE: {rel_display}")
    print("=" * 90)
    if path.exists():
        content = path.read_text(encoding="utf-8", errors="replace")
        print(content[:14000])
    else:
        print("FILE NOT FOUND")

# Inspect the centralized-baseline notebook's code cells
nb_path = ROOT / "notebooks/03_centralized_baseline.ipynb"
print("\n" + "=" * 90)
print("NOTEBOOK: notebooks/03_centralized_baseline.ipynb")
print("=" * 90)

if nb_path.exists():
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8", errors="replace"))
        for i, cell in enumerate(nb.get("cells", [])):
            if cell.get("cell_type") == "code":
                source = "".join(cell.get("source", []))
                if source.strip():
                    print(f"\n--- Code cell {i} ---")
                    print(source[:5000])
    except Exception as exc:
        print(f"Failed to parse notebook JSON: {exc}")
else:
    print("FILE NOT FOUND")


FILE: scripts/train.py


FILE: scripts/federated_train.py


FILE: src/dp_mmfl/train.py


FILE: src/dp_mmfl/data/cached_image_dataset.py
from pathlib import Path
from typing import Sequence, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from dp_mmfl.data.labels import TARGET_COLUMNS


class CachedCheXpertPlusImageDataset(Dataset):
    """
    Cached image-only PyTorch Dataset for CheXpert Plus frontal radiographs.
    Loads locally stored PNG files indexed by 7-digit zero-padded sample_id.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        cache_root: Union[str, Path],
        sample_ids: Sequence[int],
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            manifest["sample_id"].isin(sample_ids)
        ].copy()

        # Preserve the exact requested sample order
        order = {
            int(sample_id): i
            

In [ ]:
# ==============================================================================
# Diagnostic Cell: Environment, File Sizes, Manifest Integrity & Cache Status
# Note: Falls back to Google Drive path if manifest is not under /content/DP-MMFL
# ==============================================================================
import json
from pathlib import Path
import pandas as pd
import torch

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

# Resolve manifest location across possible root paths
PROTOCOL_LOCAL = ROOT / "results/final_protocol"
PROTOCOL_DRIVE = DRIVE_ROOT / "results/final_protocol"

if (PROTOCOL_LOCAL / "chexpert_plus_25k_frozen_manifest.parquet").exists():
    MANIFEST = PROTOCOL_LOCAL / "chexpert_plus_25k_frozen_manifest.parquet"
elif (PROTOCOL_DRIVE / "chexpert_plus_25k_frozen_manifest.parquet").exists():
    MANIFEST = PROTOCOL_DRIVE / "chexpert_plus_25k_frozen_manifest.parquet"
else:
    MANIFEST = PROTOCOL_LOCAL / "chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")

print("=== 1. Training files ===")
for rel in [
    "scripts/train.py",
    "scripts/federated_train.py",
    "src/dp_mmfl/train.py",
    "notebooks/03_centralized_baseline.ipynb",
]:
    p = ROOT / rel
    size = f"{p.stat().st_size:,} bytes" if p.exists() else "N/A"
    print(f"{rel:40s} | exists={str(p.exists()):5s} | size={size}")

print("\n=== 2. Notebook validity ===")
nb_path = ROOT / "notebooks/03_centralized_baseline.ipynb"
if nb_path.exists():
    try:
        nb = json.loads(nb_path.read_text(encoding="utf-8", errors="replace"))
        cells = nb.get("cells", [])
        code_cells = [c for c in cells if c.get("cell_type") == "code"]
        print(f"Valid JSON: Total cells={len(cells)}, Code cells={len(code_cells)}")
    except Exception as e:
        print("Notebook parse error:", repr(e))
else:
    print(f"Notebook not found at: {nb_path}")

print("\n=== 3. Frozen manifest ===")
if MANIFEST.exists():
    df = pd.read_parquet(MANIFEST)
    train = df[df["experiment_split"] == "train"]
    val = df[df["experiment_split"] == "val"]
    test = df[df["experiment_split"] == "test"]
    print(f"Manifest Path:     {MANIFEST}")
    print(f"Training images:   {len(train):,} (Patients: {train['deid_patient_id'].nunique():,})")
    print(f"Validation images: {len(val):,} (Patients: {val['deid_patient_id'].nunique():,})")
    print(f"Test images:       {len(test):,} (Patients: {test['deid_patient_id'].nunique():,})")
else:
    print(f"Manifest not found at {MANIFEST}")
    df = None

print("\n=== 4. Cached image coverage ===")
if CACHE.exists() and df is not None:
    cached = {p.stem for p in CACHE.glob("*.png")}
    expected = {f"{int(x):07d}" for x in train["sample_id"]}
    found = len(cached & expected)
    pct = (found / len(expected)) * 100 if expected else 0.0
    print(f"Cache directory:        {CACHE}")
    print(f"Training images cached: {found:,} / {len(expected):,} ({pct:.2f}%)")
    print(f"Total PNG files in dir: {len(cached):,}")
else:
    print(f"Cache check skipped or directory missing: {CACHE}")

print("\n=== 5. Runtime & GPU Setup ===")
print("PyTorch:        ", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:            ", torch.cuda.get_device_name(0))
    print("VRAM (GiB):     ", round(props.total_memory / (1024**3), 2))
    print("Compute Cap:    ", f"{props.major}.{props.minor}")

=== 1. Training files ===
scripts/train.py                         | exists=True  | size=0 bytes
scripts/federated_train.py               | exists=True  | size=0 bytes
src/dp_mmfl/train.py                     | exists=True  | size=0 bytes
notebooks/03_centralized_baseline.ipynb  | exists=True  | size=0 bytes

=== 2. Notebook validity ===
Notebook parse error: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')

=== 3. Frozen manifest ===
Manifest Path:     /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
Training images:   73,640 (Patients: 25,000)
Validation images: 18,573 (Patients: 6,449)
Test images:       19,349 (Patients: 6,452)

=== 4. Cached image coverage ===
Cache directory:        /content/dp_mmfl_image_cache_256/train
Training images cached: 271 / 73,640 (0.37%)
Total PNG files in dir: 624

=== 5. Runtime & GPU Setup ===
PyTorch:         2.11.0+cu130
CUDA available:  True
GPU:             Tesla T4
VRAM (GiB):      14.5

In [ ]:
# ==============================================================================
# GPU Smoke Test: Verified Cached Samples, ResNet-18, DP-Compliant Masked BCE
# ==============================================================================
import time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

# 1. Configuration & Paths
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MANIFEST_PATH = Path("/content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet")
CACHE_DIR = Path("/content/dp_mmfl_image_cache_256/train")

LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

target_cols = [f"target_{x}" for x in LABELS]
mask_cols = [f"mask_{x}" for x in LABELS]

# 2. Filter manifest strictly to existing cached images
df = pd.read_parquet(MANIFEST_PATH)
train_df = df[df["experiment_split"] == "train"].copy()

cached_ids = {int(p.stem) for p in CACHE_DIR.glob("*.png")}
smoke_df = train_df[train_df["sample_id"].astype(int).isin(cached_ids)].reset_index(drop=True)

print(f"=== GPU Smoke Test Setup ===")
print(f"Device:               {DEVICE} ({torch.cuda.get_device_name(0)})")
print(f"Available in cache:   {len(smoke_df)} samples")
assert len(smoke_df) >= 16, f"Need at least 16 cached samples for smoke test, found {len(smoke_df)}"

# 3. Lightweight Dataset targeting local PNG files
class LocalCachedCheXpertDataset(Dataset):
    def __init__(self, subset_df, cache_dir, transform=None):
        self.df = subset_df.reset_index(drop=True)
        self.cache_dir = cache_dir
        self.transform = transform
        self.targets = self.df[target_cols].values.astype(np.float32)
        self.masks = self.df[mask_cols].values.astype(np.float32)
        self.sample_ids = self.df["sample_id"].values.astype(int)
        self.patient_ids = self.df["deid_patient_id"].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        img_path = self.cache_dir / f"{sid:07d}.png"

        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform:
                img_t = self.transform(img)
            else:
                img_t = transforms.ToTensor()(img)

        # Replace NaN with 0.0 in targets
        t = torch.nan_to_num(torch.from_numpy(self.targets[idx]), nan=0.0)
        m = torch.from_numpy(self.masks[idx])

        return {
            "image": img_t,
            "labels": t,
            "label_mask": m,
            "sample_id": sid,
            "patient_id": self.patient_ids[idx],
        }

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

smoke_dataset = LocalCachedCheXpertDataset(smoke_df, CACHE_DIR, transform=transform)
smoke_loader = DataLoader(smoke_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

# 4. DP-Compliant Per-Image Masked BCE Loss
def per_image_masked_bce(logits, targets, mask):
    """
    Computes masked binary cross entropy normalized per image.
    Essential for DP-SGD gradient clipping consistency.
    """
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    numerator = (element_loss * mask).sum(dim=1)
    denominator = mask.sum(dim=1).clamp_min(1.0)
    return numerator / denominator

# 5. Model, Optimizer & Single-Batch Step
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(LABELS))
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print("\nRunning single-batch forward, loss, backward, and optimizer step...")
model.train()
torch.cuda.reset_peak_memory_stats()
start_t = time.perf_counter()

batch = next(iter(smoke_loader))
images = batch["image"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)
masks = batch["label_mask"].to(DEVICE, non_blocking=True)

optimizer.zero_grad(set_to_none=True)
logits = model(images)

per_sample_loss = per_image_masked_bce(logits, labels, masks)
loss = per_sample_loss.mean()

loss.backward()
optimizer.step()
torch.cuda.synchronize()

elapsed = time.perf_counter() - start_t
peak_vram_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

# Assertions
assert logits.shape == (16, 13), f"Logits shape mismatch: {logits.shape}"
assert per_sample_loss.shape == (16,), f"Per-sample loss shape mismatch: {per_sample_loss.shape}"
assert torch.isfinite(loss).item(), "Loss is non-finite (NaN/Inf)"

print("\n=== SMOKE TEST PASSED ===")
print(f"Batch shape:           {tuple(images.shape)}")
print(f"Logits shape:          {tuple(logits.shape)}")
print(f"Mean loss value:       {loss.item():.4f}")
print(f"Step duration:         {elapsed * 1000:.1f} ms")
print(f"Peak VRAM used:        {peak_vram_mib:.1f} MiB / 14,910 MiB")

=== GPU Smoke Test Setup ===
Device:               cuda:0 (Tesla T4)
Available in cache:   271 samples
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 99.9MB/s]



Running single-batch forward, loss, backward, and optimizer step...

=== SMOKE TEST PASSED ===
Batch shape:           (16, 3, 224, 224)
Logits shape:          (16, 13)
Mean loss value:       0.6629
Step duration:         4133.5 ms
Peak VRAM used:        473.2 MiB / 14,910 MiB


In [ ]:
# ==============================================================================
# DP-MMFL: GPU Smoke Test Using Existing Cached Images & Native Repo Classes
# ==============================================================================
import sys
import time
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

# Ensure src/ is on the Python module search path
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dp_mmfl.data.cached_image_dataset import CachedCheXpertPlusImageDataset
from dp_mmfl.models.image_model import ResNet18Classifier

# Resolve frozen manifest location
MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")

assert torch.cuda.is_available(), "CUDA GPU not available"
assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"
assert CACHE.exists(), f"Cache not found: {CACHE}"

DEVICE = torch.device("cuda:0")

# 1. Inspect cached sample IDs to create a zero-download probe dataframe
df = pd.read_parquet(MANIFEST)
train_df = df[df["experiment_split"] == "train"].copy()

cached_ids = {int(p.stem) for p in CACHE.glob("*.png")}
probe_df = train_df[train_df["sample_id"].astype(int).isin(cached_ids)].reset_index(drop=True)

print(f"=== GPU Smoke Test Setup ===")
print(f"Device:             {DEVICE} ({torch.cuda.get_device_name(0)})")
print(f"Manifest path:      {MANIFEST}")
print(f"Cached train items: {len(probe_df):,} available")
assert len(probe_df) >= 8, f"Insufficient cached items for smoke test: {len(probe_df)}"

# 2. Evaluation image transform
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 3. Instantiate native repository CachedCheXpertPlusImageDataset
try:
    smoke_ds = CachedCheXpertPlusImageDataset(
        df=probe_df,
        cache_dir=CACHE,
        image_transform=image_transform,
    )
except TypeError:
    try:
        smoke_ds = CachedCheXpertPlusImageDataset(
            manifest_path=MANIFEST,
            cache_dir=CACHE,
            split="train",
            sample_ids=probe_df["sample_id"].astype(int).tolist(),
            image_transform=image_transform,
        )
    except TypeError:
        smoke_ds = CachedCheXpertPlusImageDataset(
            manifest=probe_df,
            cache_root=CACHE,
            transform=image_transform,
        )

smoke_loader = DataLoader(
    smoke_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# 4. DP-Compliant Per-Image Masked BCE Loss
def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    numerator = (element_loss * mask).sum(dim=1)
    denominator = mask.sum(dim=1).clamp_min(1.0)
    return numerator / denominator

# 5. Initialize native ResNet18Classifier (13 target labels)
model = ResNet18Classifier(num_classes=13, pretrained=True)
model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# 6. Step execution
model.train()
torch.cuda.reset_peak_memory_stats()
start_time = time.perf_counter()

batch = next(iter(smoke_loader))

if isinstance(batch, dict):
    images = batch["image"].to(DEVICE, non_blocking=True)
    labels = batch["labels"].to(DEVICE, non_blocking=True)
    masks = batch["label_mask"].to(DEVICE, non_blocking=True)
else:
    images = batch[0].to(DEVICE, non_blocking=True)
    labels = batch[1].to(DEVICE, non_blocking=True)
    masks = batch[2].to(DEVICE, non_blocking=True)

optimizer.zero_grad(set_to_none=True)
logits = model(images)

per_sample_loss = per_image_masked_bce(logits, labels, masks)
loss = per_sample_loss.mean()

loss.backward()
optimizer.step()
torch.cuda.synchronize()

duration_ms = (time.perf_counter() - start_time) * 1000
peak_vram_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

# Assertions
assert logits.shape == (8, 13), f"Logits shape mismatch: {logits.shape}"
assert per_sample_loss.shape == (8,), f"Per-sample loss shape mismatch: {per_sample_loss.shape}"
assert torch.isfinite(loss).item(), "Non-finite loss value detected"

print("\n=== SMOKE TEST PASSED ===")
print(f"Batch images tensor:  {tuple(images.shape)} ({images.dtype})")
print(f"Logits output tensor: {tuple(logits.shape)}")
print(f"Batch mean loss:      {loss.item():.4f}")
print(f"Step duration:        {duration_ms:.1f} ms")
print(f"Peak VRAM used:       {peak_vram_mib:.1f} MiB / 14,910 MiB")

=== GPU Smoke Test Setup ===
Device:             cuda:0 (Tesla T4)
Manifest path:      /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
Cached train items: 271 available


TypeError: CachedCheXpertPlusImageDataset.__init__() got an unexpected keyword argument 'manifest'

In [ ]:
import inspect
from pathlib import Path
import sys

ROOT = Path("/content/DP-MMFL")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from dp_mmfl.data.cached_image_dataset import CachedCheXpertPlusImageDataset

print("=== CachedCheXpertPlusImageDataset Signature ===")
print(inspect.signature(CachedCheXpertPlusImageDataset.__init__))

print("\n=== Implementation Source ===")
with open(ROOT / "src/dp_mmfl/data/cached_image_dataset.py", "r") as f:
    print(f.read())

=== CachedCheXpertPlusImageDataset Signature ===
(self, manifest_path: Union[str, pathlib._local.Path], cache_root: Union[str, pathlib._local.Path], sample_ids: Sequence[int], image_transform=None)

=== Implementation Source ===
from pathlib import Path
from typing import Sequence, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from dp_mmfl.data.labels import TARGET_COLUMNS


class CachedCheXpertPlusImageDataset(Dataset):
    """
    Cached image-only PyTorch Dataset for CheXpert Plus frontal radiographs.
    Loads locally stored PNG files indexed by 7-digit zero-padded sample_id.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        cache_root: Union[str, Path],
        sample_ids: Sequence[int],
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            manifest["sample_id"].isin(sample_ids)
        ].copy()

        # Preserve t

In [ ]:
# ==============================================================================
# GPU Smoke Test: Initializing CachedCheXpertPlusImageDataset with Exact Signature
# ==============================================================================
import sys
import time
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dp_mmfl.data.cached_image_dataset import CachedCheXpertPlusImageDataset
from dp_mmfl.models.image_model import ResNet18Classifier

# 1. Paths & Environment
MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

assert torch.cuda.is_available(), "CUDA GPU not available"
assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"
assert CACHE.exists(), f"Cache not found: {CACHE}"

# 2. Get list of available sample_ids currently on disk
cached_files = list(CACHE.glob("*.png"))
cached_ids = [int(p.stem) for p in cached_files]

df = pd.read_parquet(MANIFEST)
train_df = df[df["experiment_split"] == "train"]
valid_cached_ids = [sid for sid in cached_ids if sid in set(train_df["sample_id"].astype(int))]

print(f"=== GPU Smoke Test Setup ===")
print(f"Device:                 {DEVICE} ({torch.cuda.get_device_name(0)})")
print(f"Valid cached train IDs: {len(valid_cached_ids)}")
assert len(valid_cached_ids) >= 16, f"Need at least 16 cached IDs, found {len(valid_cached_ids)}"

# 3. Transform converting single-channel L to 3 channels for ResNet
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 4. Instantiate CachedCheXpertPlusImageDataset using exact signature
smoke_ds = CachedCheXpertPlusImageDataset(
    manifest_path=MANIFEST,
    cache_root=CACHE,
    sample_ids=valid_cached_ids[:32],
    image_transform=image_transform,
)

smoke_loader = DataLoader(
    smoke_ds,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# 5. DP-Compliant Per-Image Masked BCE Loss
def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    numerator = (element_loss * mask).sum(dim=1)
    denominator = mask.sum(dim=1).clamp_min(1.0)
    return numerator / denominator

# 6. Instantiate ResNet18Classifier
try:
    model = ResNet18Classifier(num_classes=13, pretrained=True)
except TypeError:
    try:
        model = ResNet18Classifier(num_classes=13)
    except TypeError:
        model = ResNet18Classifier()

model = model.to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# 7. Step execution
model.train()
torch.cuda.reset_peak_memory_stats()
start_time = time.perf_counter()

batch = next(iter(smoke_loader))
images = batch["image"].to(DEVICE, non_blocking=True)
labels = batch["labels"].to(DEVICE, non_blocking=True)
masks = batch["label_mask"].to(DEVICE, non_blocking=True)

optimizer.zero_grad(set_to_none=True)
logits = model(images)

per_sample_loss = per_image_masked_bce(logits, labels, masks)
loss = per_sample_loss.mean()

loss.backward()
optimizer.step()
torch.cuda.synchronize()

duration_ms = (time.perf_counter() - start_time) * 1000
peak_vram_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

# Assertions
assert logits.shape == (16, 13), f"Logits shape mismatch: {logits.shape}"
assert per_sample_loss.shape == (16,), f"Per-sample loss shape mismatch: {per_sample_loss.shape}"
assert torch.isfinite(loss).item(), "Non-finite loss value detected"

print("\n=== SMOKE TEST PASSED ===")
print(f"Batch images tensor:  {tuple(images.shape)} ({images.dtype})")
print(f"Labels tensor:        {tuple(labels.shape)}")
print(f"Masks tensor:         {tuple(masks.shape)}")
print(f"Logits output tensor: {tuple(logits.shape)}")
print(f"Batch mean loss:      {loss.item():.4f}")
print(f"Step duration:        {duration_ms:.1f} ms")
print(f"Peak VRAM used:       {peak_vram_mib:.1f} MiB / 14,910 MiB")

=== GPU Smoke Test Setup ===
Device:                 cuda:0 (Tesla T4)
Valid cached train IDs: 271

=== SMOKE TEST PASSED ===
Batch images tensor:  (16, 3, 224, 224) (torch.float32)
Labels tensor:        (16, 13)
Masks tensor:         (16, 13)
Logits output tensor: (16, 13)
Batch mean loss:      0.6562
Step duration:        438.6 ms
Peak VRAM used:       535.0 MiB / 14,910 MiB


In [ ]:
# ==============================================================================
# Inspection Cell: Audit Data Loader Modules, Transforms, and Downloader Scripts
# ==============================================================================
from pathlib import Path

ROOT = Path("/content/DP-MMFL")

for folder in [
    ROOT / "src/dp_mmfl/data",
    ROOT / "scripts",
]:
    print(f"\n=== {folder.relative_to(ROOT)} ===")
    if folder.exists():
        for p in sorted(folder.glob("*.py")):
            print(f"{p.name:35s}: {p.stat().st_size:,} bytes")
    else:
        print("Folder not found")

for rel in [
    "src/dp_mmfl/data/image_dataset.py",
    "src/dp_mmfl/data/image_transforms.py",
    "src/dp_mmfl/data/preprocessed_cached_dataset.py",
]:
    p = ROOT / rel
    print(f"\n{'='*70}\n{rel}\n{'='*70}")
    if p.exists():
        content = p.read_text(encoding="utf-8", errors="replace")
        print(content[:9000] if content.strip() else "(File is empty - 0 bytes)")
    else:
        print("FILE NOT FOUND")


=== src/dp_mmfl/data ===
__init__.py                        : 0 bytes
cached_image_dataset.py            : 2,491 bytes
dataset.py                         : 3,672 bytes
datasets.py                        : 0 bytes
image_dataset.py                   : 2,326 bytes
image_transforms.py                : 1,080 bytes
labels.py                          : 2,007 bytes
preprocessed_cached_dataset.py     : 2,492 bytes
preprocessing.py                   : 0 bytes
splits.py                          : 0 bytes
text.py                            : 1,382 bytes
transforms.py                      : 0 bytes

=== scripts ===
evaluate.py                        : 0 bytes
federated_train.py                 : 0 bytes
preprocess.py                      : 0 bytes
train.py                           : 0 bytes

src/dp_mmfl/data/image_dataset.py
from io import BytesIO
from pathlib import Path
from typing import Optional, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
# ==============================================================================
# Step: Benchmark & Cache 64 Training Images from Frozen 25k Manifest
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
import time
import pandas as pd
from PIL import Image

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")
CACHE.mkdir(parents=True, exist_ok=True)

# Connect to Redivis table if not present in the runtime
if "table" not in globals():
    import redivis
    table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")

df = pd.read_parquet(MANIFEST)
train = df[df["experiment_split"] == "train"].head(64)

def cache_one(row):
    sample_id = int(row["sample_id"])
    destination = CACHE / f"{sample_id:07d}.png"
    if destination.exists():
        try:
            with Image.open(destination) as im:
                im.verify()
            return "cached"
        except Exception:
            destination.unlink(missing_ok=True)

    path = str(row["path_to_image"])
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"

    data = table.file(path).read(as_text=False)
    with Image.open(BytesIO(data)) as im:
        im = im.convert("L")
        im.thumbnail((256, 256))
        im.save(destination, format="PNG", optimize=True)
    return "downloaded"

start = time.perf_counter()
results = []
errors = []

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {
        pool.submit(cache_one, row): int(row["sample_id"])
        for _, row in train.iterrows()
    }
    for future in as_completed(futures):
        sid = futures[future]
        try:
            results.append(future.result())
        except Exception as exc:
            errors.append((sid, repr(exc)))

elapsed = time.perf_counter() - start

print("=== CACHE BENCHMARK ===")
print("Requested:        ", len(train))
print("Downloaded:       ", results.count("downloaded"))
print("Already cached:   ", results.count("cached"))
print("Errors:           ", len(errors))
print("Elapsed seconds:  ", round(elapsed, 2))
print("Images/second:    ", round(len(results) / elapsed, 2) if elapsed else 0)
print(
    "Cache size MiB:   ",
    round(sum(p.stat().st_size for p in CACHE.glob("*.png")) / (1024 ** 2), 2),
)

if errors:
    print("\nFirst errors:")
    for item in errors[:5]:
        print(item)

=== CACHE BENCHMARK ===
Requested:         64
Downloaded:        0
Already cached:    64
Errors:            0
Elapsed seconds:   0.05
Images/second:     1410.13
Cache size MiB:    17.62


In [ ]:
# ==============================================================================
# Timed Training Benchmark: 4 Batches on Local Cache (ResNet-18 + T4 GPU)
# Measures pure GPU compute throughput without network latency
# ==============================================================================
import sys
import time
from pathlib import Path
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

# Ensure src/ is accessible
src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset
from dp_mmfl.data.image_transforms import get_train_transforms
from dp_mmfl.models.image_model import ResNet18Classifier

# Resolve frozen manifest path
MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")

assert MANIFEST.exists(), f"Manifest missing: {MANIFEST}"
assert CACHE.exists(), f"Cache missing: {CACHE}"

df = pd.read_parquet(MANIFEST)
train = df[df["experiment_split"] == "train"]

cached_ids = {
    int(p.stem) for p in CACHE.glob("*.png") if p.stem.isdigit()
}
sample_ids = [
    int(x) for x in train["sample_id"] if int(x) in cached_ids
][:64]

assert len(sample_ids) >= 16, f"Only {len(sample_ids)} cached training images available"

dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST,
    cache_root=CACHE,
    sample_ids=sample_ids,
    image_transform=get_train_transforms(),
)
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18Classifier(num_classes=13, pretrained=False).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

def per_image_loss(logits, targets, masks):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    masks = masks.float()
    element = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    return ((element * masks).sum(1) / masks.sum(1).clamp_min(1)).mean()

model.train()
times, losses, images_seen = [], [], 0

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for step, batch in enumerate(loader):
    if step >= 4:
        break
    start = time.perf_counter()
    images = batch["image"].to(device, non_blocking=True)
    targets = batch["labels"].to(device, non_blocking=True)
    masks = batch["label_mask"].to(device, non_blocking=True)

    optimizer.zero_grad(set_to_none=True)
    logits = model(images)
    loss = per_image_loss(logits, targets, masks)
    loss.backward()
    optimizer.step()

    if device.type == "cuda":
        torch.cuda.synchronize()

    times.append(time.perf_counter() - start)
    losses.append(loss.item())
    images_seen += images.size(0)

print("=== TIMED TRAINING BENCHMARK RESULTS ===")
print("Device:                         ", device)
print("Cached sample pool size:        ", len(dataset))
print("Images processed:               ", images_seen)
print("Batch losses:                   ", [round(x, 4) for x in losses])
print("Mean step duration:             ", f"{sum(times) / len(times):.4f} s")
print("Training throughput:            ", f"{images_seen / sum(times):.2f} images/sec")
if device.type == "cuda":
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print("Peak GPU memory:                ", f"{peak_vram:.1f} MiB")

=== TIMED TRAINING BENCHMARK RESULTS ===
Device:                          cuda
Cached sample pool size:         64
Images processed:                64
Batch losses:                    [0.642, 0.5335, 0.54, 0.5206]
Mean step duration:              0.0693 s
Training throughput:             230.77 images/sec
Peak GPU memory:                 646.3 MiB


In [ ]:
# ==============================================================================
# Genuine Uncached Retrieval Test (8 Images via Redivis API)
# Measures end-to-end network transfer, decompression, 256px resize, and PNG write
# ==============================================================================
from io import BytesIO
from pathlib import Path
import time
import pandas as pd
from PIL import Image

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")
CACHE.mkdir(parents=True, exist_ok=True)

# Ensure active Redivis table handle
if "table" not in globals():
    import redivis
    table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")

df = pd.read_parquet(MANIFEST)
train = df[df["experiment_split"] == "train"]

# Select 8 images whose PNG files do not exist in the local cache directory
existing_cached_ids = {
    int(p.stem) for p in CACHE.glob("*.png") if p.stem.isdigit()
}
uncached = train[
    ~train["sample_id"].astype(int).isin(existing_cached_ids)
].head(8)

assert len(uncached) > 0, "No uncached training images found in the selected manifest."

elapsed = []
failures = []

print(f"=== Starting Uncached Fetch ({len(uncached)} images) ===")
for _, row in uncached.iterrows():
    sid = int(row["sample_id"])
    path = str(row["path_to_image"])

    # Path resolution for Redivis CheXpert Plus PNG bucket
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"

    start = time.perf_counter()
    try:
        data = table.file(path).read(as_text=False)
        with Image.open(BytesIO(data)) as img:
            img = img.convert("L")
            img.thumbnail((256, 256))
            destination = CACHE / f"{sid:07d}.png"
            img.save(destination, format="PNG", optimize=True)

        dur = time.perf_counter() - start
        elapsed.append(dur)
        print(
            f"OK | sample_id={sid:07d} | "
            f"raw_bytes={len(data)/1024:6.1f} KiB | "
            f"time={dur:4.2f}s"
        )
    except Exception as exc:
        failures.append((sid, repr(exc)))
        print(f"FAILED | sample_id={sid:07d} | error={exc}")

print("\n=== REAL RETRIEVAL RESULTS ===")
print("Requested:              ", len(uncached))
print("Succeeded:              ", len(elapsed))
print("Failed:                 ", len(failures))
if elapsed:
    total_time = sum(elapsed)
    print("Total retrieval seconds:", round(total_time, 2))
    print("Mean seconds/image:     ", round(total_time / len(elapsed), 2))
    print("Images/second:          ", round(len(elapsed) / total_time, 2))

if failures:
    print("\nFailure details:")
    for failure in failures:
        print(f"  Sample {failure[0]}: {failure[1]}")

=== Starting Uncached Fetch (8 images) ===
OK | sample_id=0000918 | raw_bytes=3503.0 KiB | time=3.72s
OK | sample_id=0000921 | raw_bytes=3079.4 KiB | time=3.35s
OK | sample_id=0000922 | raw_bytes=1833.8 KiB | time=3.13s
OK | sample_id=0000923 | raw_bytes=3084.4 KiB | time=3.72s
OK | sample_id=0000929 | raw_bytes=3247.5 KiB | time=3.41s
OK | sample_id=0000931 | raw_bytes=3283.1 KiB | time=3.31s
OK | sample_id=0000933 | raw_bytes=3135.8 KiB | time=3.32s
OK | sample_id=0000934 | raw_bytes=3884.3 KiB | time=2.43s

=== REAL RETRIEVAL RESULTS ===
Requested:               8
Succeeded:               8
Failed:                  0
Total retrieval seconds: 26.39
Mean seconds/image:      3.3
Images/second:           0.3


In [ ]:
# ==============================================================================
# Genuine Concurrent Retrieval Benchmark: 8 Workers on 32 Uncached Images
# Measures true multi-threaded network transfer, decode, resize, and disk I/O
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
from pathlib import Path
import time
import pandas as pd
from PIL import Image

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")
CACHE.mkdir(parents=True, exist_ok=True)

assert "table" in globals(), "Initialize your existing Redivis table connection first."

df = pd.read_parquet(MANIFEST)
train = df[df["experiment_split"] == "train"]

# Select 32 images that genuinely do not exist in the local cache
existing_ids = {
    int(p.stem) for p in CACHE.glob("*.png") if p.stem.isdigit()
}
uncached = train[
    ~train["sample_id"].astype(int).isin(existing_ids)
].head(32)

assert len(uncached) > 0, "No uncached training images found."

def fetch_and_cache(row):
    sid = int(row["sample_id"])
    path = str(row["path_to_image"])
    if path.startswith("train/"):
        path = path[len("train/"):]
    if path.endswith(".jpg"):
        path = path[:-4] + ".png"

    data = table.file(path).read(as_text=False)
    with Image.open(BytesIO(data)) as im:
        im = im.convert("L")
        im.thumbnail((256, 256))
        im.save(CACHE / f"{sid:07d}.png", format="PNG", optimize=True)
    return sid, len(data)

print(f"=== Starting 8-Worker Benchmark ({len(uncached)} uncached images) ===")
start = time.perf_counter()
successes, failures = [], []

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {
        pool.submit(fetch_and_cache, row): int(row["sample_id"])
        for _, row in uncached.iterrows()
    }
    for future in as_completed(futures):
        try:
            successes.append(future.result())
        except Exception as exc:
            failures.append((futures[future], repr(exc)))

elapsed = time.perf_counter() - start
throughput = len(successes) / max(elapsed, 1e-6)

print("\n=== CONCURRENT RETRIEVAL BENCHMARK ===")
print("Workers:                 8")
print(f"Requested:               {len(uncached)}")
print(f"Succeeded:               {len(successes)}")
print(f"Failed:                  {len(failures)}")
print(f"Elapsed seconds:         {elapsed:.2f}")
print(f"Images/second:           {throughput:.3f}")
print(f"Raw MiB fetched:         {sum(n for _, n in successes) / (1024**2):.2f}")
print(f"Cache PNGs total:        {len(list(CACHE.glob('*.png')))}")

if failures:
    print(f"\nEncountered {len(failures)} failures:")
    for item in failures[:5]:
        print(f"  Sample {item[0]}: {item[1]}")

=== Starting 8-Worker Benchmark (32 uncached images) ===

=== CONCURRENT RETRIEVAL BENCHMARK ===
Workers:                 8
Requested:               32
Succeeded:               32
Failed:                  0
Elapsed seconds:         16.24
Images/second:           1.970
Raw MiB fetched:         94.04
Cache PNGs total:        664


In [ ]:
# ==============================================================================
# DP-MMFL: Resumable Background Caching Job (8 Workers -> Local SSD -> Drive Sync)
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
import json
from pathlib import Path
import shutil
from threading import Lock, Thread
import time
import pandas as pd
from PIL import Image

# 1. Paths & Verification
ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
DRIVE_CACHE = DRIVE_ROOT / "data/image_cache_256/train"
PROGRESS_FILE = DRIVE_ROOT / "data/image_cache_256/caching_progress.json"

LOCAL_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)

assert "table" in globals(), "Redivis table connection not initialized."
assert MANIFEST.exists(), f"Manifest missing: {MANIFEST}"

# 2. Identify Pending Images
df = pd.read_parquet(MANIFEST)
train = df[df["experiment_split"] == "train"].copy()
train["sample_id"] = train["sample_id"].astype(int)

# Check both local SSD and Google Drive
existing_local = {int(p.stem) for p in LOCAL_CACHE.glob("*.png") if p.stem.isdigit()}
existing_drive = {int(p.stem) for p in DRIVE_CACHE.glob("*.png") if p.stem.isdigit()}
all_cached_ids = existing_local | existing_drive

missing_df = train[~train["sample_id"].isin(all_cached_ids)].copy()

print("=== Cache Download Status ===")
print(f"Total frozen training set: {len(train):,} images")
print(f"Already cached in Drive:   {len(existing_drive):,} images")
print(f"Already cached on SSD:     {len(existing_local):,} images")
print(f"Total unique cached:       {len(all_cached_ids):,} images ({len(all_cached_ids)/len(train):.2%})")
print(f"Remaining to download:     {len(missing_df):,} images")

# 3. Thread-safe Downloader & Sync Logic
lock = Lock()
sync_batch_size = 128
pending_sync_files = []

def download_single_image(row):
    sid = int(row["sample_id"])
    local_path = LOCAL_CACHE / f"{sid:07d}.png"
    drive_path = DRIVE_CACHE / f"{sid:07d}.png"

    # If already on Drive but missing locally, copy local for fast training read
    if drive_path.exists() and not local_path.exists():
        shutil.copy2(drive_path, local_path)
        return sid, "restored"

    if local_path.exists() and drive_path.exists():
        return sid, "skipped"

    rel_path = str(row["path_to_image"])
    if rel_path.startswith("train/"):
        rel_path = rel_path[len("train/"):]
    if rel_path.endswith(".jpg"):
        rel_path = rel_path[:-4] + ".png"

    raw_bytes = table.file(rel_path).read(as_text=False)
    with Image.open(BytesIO(raw_bytes)) as im:
        im = im.convert("L")
        im.thumbnail((256, 256))
        im.save(local_path, format="PNG", optimize=True)

    with lock:
        pending_sync_files.append((local_path, drive_path))
        if len(pending_sync_files) >= sync_batch_size:
            for src, dst in pending_sync_files:
                if not dst.exists():
                    shutil.copy2(src, dst)
            pending_sync_files.clear()

    return sid, "downloaded"

def run_cache_worker():
    start_time = time.perf_counter()
    downloaded_count = 0
    errors = []

    print("\n[Cache Worker] Background thread started.")
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {
            pool.submit(download_single_image, row): int(row["sample_id"])
            for _, row in missing_df.iterrows()
        }

        for future in as_completed(futures):
            sid = futures[future]
            try:
                _, status = future.result()
                if status == "downloaded":
                    with lock:
                        downloaded_count += 1
                        if downloaded_count % 64 == 0:
                            elapsed = time.perf_counter() - start_time
                            fps = downloaded_count / max(elapsed, 1e-6)
                            print(
                                f"[Cache Worker] Processed: {downloaded_count:,}/{len(missing_df):,} | "
                                f"Rate: {fps:.2f} img/s | "
                                f"Remaining: {(len(missing_df) - downloaded_count) / max(fps, 0.01) / 3600:.1f} hrs"
                            )
            except Exception as exc:
                with lock:
                    errors.append((sid, repr(exc)))

    # Flush remaining batch to Drive
    with lock:
        for src, dst in pending_sync_files:
            if not dst.exists():
                shutil.copy2(src, dst)
        pending_sync_files.clear()

    total_time = time.perf_counter() - start_time
    summary = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "downloaded_this_session": downloaded_count,
        "errors_count": len(errors),
        "total_seconds": round(total_time, 2),
        "average_fps": round(downloaded_count / max(total_time, 1e-6), 2),
    }

    with open(PROGRESS_FILE, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n[Cache Worker] Background thread finished.")
    print(f"Downloaded: {downloaded_count:,} images in {total_time/60:.1f} minutes.")
    if errors:
        print(f"Encountered {len(errors)} errors. First 3: {errors[:3]}")

# 4. Start Background Thread
cache_thread = Thread(target=run_cache_worker, daemon=True)
cache_thread.start()
print("Background cache thread is active.")

=== Cache Download Status ===
Total frozen training set: 73,640 images
Already cached in Drive:   0 images
Already cached on SSD:     664 images
Total unique cached:       664 images (0.90%)
Remaining to download:     73,329 images

[Cache Worker] Background thread started.
Background cache thread is active.


In [ ]:
import time

is_running = cache_thread.is_alive()
cached_local_count = len(list(LOCAL_CACHE.glob("*.png")))
cached_drive_count = len(list(DRIVE_CACHE.glob("*.png")))

print(f"Worker running:      {is_running}")
print(f"Local SSD cached:    {cached_local_count:,} images")
print(f"Google Drive synced: {cached_drive_count:,} images")

Worker running:      True
Local SSD cached:    1,712 images
Google Drive synced: 1,024 images


In [ ]:
import time
from pathlib import Path

LOCAL = Path("/content/dp_mmfl_image_cache_256/train")
DRIVE = Path("/content/drive/MyDrive/DP-MMFL/data/image_cache_256/train")

print("Worker active:", cache_thread.is_alive())
print("Local images:", len(list(LOCAL.glob("*.png"))))
print("Drive images:", len(list(DRIVE.glob("*.png"))))

time.sleep(60)

print("\nAfter 60 seconds:")
print("Worker active:", cache_thread.is_alive())
print("Local images:", len(list(LOCAL.glob("*.png"))))
print("Drive images:", len(list(DRIVE.glob("*.png"))))

Worker active: True
Local images: 1963
Drive images: 1280
[Cache Worker] Processed: 1,344/73,329 | Rate: 2.70 img/s | Remaining: 7.4 hrs
[Cache Worker] Processed: 1,408/73,329 | Rate: 2.71 img/s | Remaining: 7.4 hrs
[Cache Worker] Processed: 1,472/73,329 | Rate: 2.73 img/s | Remaining: 7.3 hrs

After 60 seconds:
Worker active: True
Local images: 2144
Drive images: 1408


In [ ]:
# ==============================================================================
# DP-MMFL: 2-Epoch Pilot Baseline on Cached Training Subset (ResNet-18)
# Patient-disjoint internal split, per-image masked BCE, validation AUROC, Drive ckpt
# ==============================================================================
from pathlib import Path
import time
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

# 1. Setup & Device Verification
ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")
OUT = DRIVE_ROOT / "checkpoints/pilot_baseline"
OUT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable a GPU runtime first."
assert MANIFEST.exists(), f"Manifest missing: {MANIFEST}"
assert CACHE.exists(), f"Cache missing: {CACHE}"

device = torch.device("cuda:0")
torch.manual_seed(42)
np.random.seed(42)

# 2. Filter to Available Cached Images
df = pd.read_parquet(MANIFEST)
df = df[df["experiment_split"] == "train"].copy()
df["sample_id"] = df["sample_id"].astype(int)

cached_ids = {int(p.stem) for p in CACHE.glob("*.png") if p.stem.isdigit()}
df = df[df["sample_id"].isin(cached_ids)].copy()

LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

target_cols = [f"target_{c}" for c in LABELS]
mask_cols = [f"mask_{c}" for c in LABELS]
assert len(target_cols) == 13 and all(c in df.columns for c in mask_cols)
assert len(df) >= 100, f"Too few cached images available: {len(df)}"

# Cap subset size to maintain fast iteration
df = df.sort_values("sample_id").tail(min(2000, len(df))).reset_index(drop=True)

# 3. Patient-Disjoint Internal Split
groups = df["deid_patient_id"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(splitter.split(df, groups=groups))
tr, va = df.iloc[tr_idx].copy().reset_index(drop=True), df.iloc[va_idx].copy().reset_index(drop=True)

print(f"=== Pilot Baseline Configuration ===")
print(f"Total cached images used: {len(df):,}")
print(f"Train split:              {len(tr):,} images ({tr['deid_patient_id'].nunique():,} patients)")
print(f"Internal val split:       {len(va):,} images ({va['deid_patient_id'].nunique():,} patients)")
overlap = len(set(tr.deid_patient_id) & set(va.deid_patient_id))
print(f"Patient overlap:          {overlap} (Strictly Disjoint: {overlap == 0})")
assert overlap == 0, "Patient leakage in internal split!"

# 4. Transforms & Dataset Definition
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = train_tf

class CachedXrayDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.targets = self.frame[target_cols].values.astype(np.float32)
        self.masks = self.frame[mask_cols].values.astype(np.float32)
        self.sample_ids = self.frame["sample_id"].values.astype(int)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        sid = self.sample_ids[i]
        path = CACHE / f"{sid:07d}.png"
        with Image.open(path) as im:
            img = self.transform(im.convert("L"))

        t = torch.nan_to_num(torch.from_numpy(self.targets[i]), nan=0.0)
        m = torch.from_numpy(self.masks[i])
        return img, t, m

train_loader = DataLoader(
    CachedXrayDataset(tr, train_tf),
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    CachedXrayDataset(va, eval_tf),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# 5. Loss & Model Initialization
def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    numerator = (element_loss * mask).sum(dim=1)
    denominator = mask.sum(dim=1).clamp_min(1.0)
    return numerator / denominator

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(LABELS))
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# 6. Evaluation Routine
@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    all_logits, all_targets, all_masks = [], [], []
    total_val_loss, count = 0.0, 0

    for imgs, tgts, msks in loader:
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        msks = msks.to(device, non_blocking=True)

        out = net(imgs)
        loss = per_image_masked_bce(out, tgts, msks).mean()

        total_val_loss += loss.item() * len(imgs)
        count += len(imgs)

        all_logits.append(torch.sigmoid(out).cpu().numpy())
        all_targets.append(tgts.cpu().numpy())
        all_masks.append(msks.cpu().numpy())

    probs = np.vstack(all_logits)
    labels_arr = np.vstack(all_targets)
    masks_arr = np.vstack(all_masks)

    auc_scores = {}
    for j, label_name in enumerate(LABELS):
        valid = masks_arr[:, j] == 1.0
        y_true = labels_arr[valid, j]
        y_score = probs[valid, j]

        # Only evaluate AUROC when both positive and negative cases exist in valid mask
        if len(np.unique(y_true)) == 2:
            try:
                auc_scores[label_name] = roc_auc_score(y_true, y_score)
            except ValueError:
                pass

    macro_auc = float(np.mean(list(auc_scores.values()))) if auc_scores else float("nan")
    return total_val_loss / max(count, 1), macro_auc, auc_scores

# 7. Training Loop (2 Epochs)
print("\n=== Launching 2-Epoch Training Run ===")
best_val_auc = -1.0

for epoch in range(1, 3):
    model.train()
    running_loss, total_seen = 0.0, 0
    t0 = time.perf_counter()

    for step, (imgs, tgts, msks) in enumerate(train_loader, start=1):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        msks = msks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        out = model(imgs)
        loss = per_image_masked_bce(out, tgts, msks).mean()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(imgs)
        total_seen += len(imgs)

    train_loss = running_loss / max(total_seen, 1)
    val_loss, val_auc, per_class_auc = evaluate(model, val_loader)
    elapsed = time.perf_counter() - t0

    print(
        f"Epoch [{epoch}/2] ({elapsed:.1f}s) | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro AUROC: {val_auc:.4f} "
        f"({len(per_class_auc)}/{len(LABELS)} classes evaluated)"
    )

    # Persist Checkpoints
    ckpt_path = OUT / f"resnet18_pilot_epoch{epoch}.pt"
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_auc": val_auc,
        "per_class_auc": per_class_auc,
        "labels": LABELS,
    }, ckpt_path)

print("\n=== PILOT COMPLETE ===")
print(f"Checkpoints directory: {OUT}")
print(f"Latest model saved:    resnet18_pilot_epoch2.pt")

=== Pilot Baseline Configuration ===
Total cached images used: 2,000
Train split:              1,578 images (560 patients)
Internal val split:       422 images (141 patients)
Patient overlap:          0 (Strictly Disjoint: True)

=== Launching 2-Epoch Training Run ===
[Cache Worker] Processed: 2,176/73,329 | Rate: 2.80 img/s | Remaining: 7.0 hrs
Epoch [1/2] (12.4s) | Train Loss: 0.4964 | Val Loss: 0.4111 | Val Macro AUROC: 0.5586 (12/13 classes evaluated)
Epoch [2/2] (11.5s) | Train Loss: 0.2758 | Val Loss: 0.4084 | Val Macro AUROC: 0.5519 (12/13 classes evaluated)

=== PILOT COMPLETE ===
Checkpoints directory: /content/drive/MyDrive/DP-MMFL/checkpoints/pilot_baseline
Latest model saved:    resnet18_pilot_epoch2.pt


In [ ]:
from pathlib import Path

checkpoint = Path(
    "/content/drive/MyDrive/DP-MMFL/checkpoints/pilot_baseline/"
    "resnet18_pilot_epoch2.pt"
)

local_cache = Path("/content/dp_mmfl_image_cache_256/train")
drive_cache = Path(
    "/content/drive/MyDrive/DP-MMFL/data/image_cache_256/train"
)

print("Pilot checkpoint exists:", checkpoint.exists())
print("Local cached images:", len(list(local_cache.glob("*.png"))))
print("Drive cached images:", len(list(drive_cache.glob("*.png"))))
print("Cache worker active:",
      cache_thread.is_alive() if "cache_thread" in globals() else "Check notebook output")

Pilot checkpoint exists: False
Local cached images: 0
Drive cached images: 0
Cache worker active: Check notebook output


In [ ]:
# ==============================================================================
# Pilot Checkpoint Audit & Detailed Per-Label AUROC Inspection
# ==============================================================================
from pathlib import Path
import pandas as pd
import torch

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/DP-MMFL/checkpoints/"
    "pilot_baseline/resnet18_pilot_epoch2.pt"
)

assert CHECKPOINT_PATH.exists(), f"Checkpoint missing: {CHECKPOINT_PATH}"

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

print("=== Checkpoint Metadata Integrity ===")
print("Checkpoint loaded successfully from:", CHECKPOINT_PATH)
print("Saved epoch:             ", ckpt.get("epoch", "unknown"))
print("Final Train Loss:        ", f"{ckpt.get('train_loss', float('nan')):.4f}")
print("Final Val Loss:          ", f"{ckpt.get('val_loss', float('nan')):.4f}")
print("Macro Val AUROC:         ", f"{ckpt.get('val_auc', float('nan')):.4f}")

# Reconcile key names between training script variants
labels = ckpt.get("labels") or ckpt.get("target_cols") or []
per_class_auc = ckpt.get("per_class_auc", {})

print(f"\nRecorded Pathologies ({len(labels)} total):")

if per_class_auc:
    auc_table = []
    for label in labels:
        score = per_class_auc.get(label, None)
        status = f"{score:.4f}" if score is not None else "N/A (insufficient pos/neg in val)"
        auc_table.append({"Pathology": label, "Validation AUROC": status})

    print(pd.DataFrame(auc_table).to_string(index=False))
else:
    print("No per-class AUROC dictionary found in checkpoint dictionary.")

# Verify state_dict presence and head dimension
state_dict = ckpt.get("model_state", {})
fc_weight = state_dict.get("fc.weight")
if fc_weight is not None:
    print(f"\nModel Head Dimension (Output Logits): {fc_weight.shape[0]} classes (Matches 13: {fc_weight.shape[0] == 13})")
else:
    print("\nWarning: 'fc.weight' key not found in saved model_state.")

=== Checkpoint Metadata Integrity ===
Checkpoint loaded successfully from: /content/drive/MyDrive/DP-MMFL/checkpoints/pilot_baseline/resnet18_pilot_epoch2.pt
Saved epoch:              2
Final Train Loss:         0.2758
Final Val Loss:           0.4084
Macro Val AUROC:          0.5519

Recorded Pathologies (13 total):
                 Pathology                  Validation AUROC
Enlarged Cardiomediastinum                            0.3621
              Cardiomegaly                            0.4533
              Lung Opacity                            0.6638
               Lung Lesion                            0.7381
                     Edema                            0.4475
             Consolidation                            0.5590
                 Pneumonia                            0.5111
               Atelectasis                            0.5116
              Pneumothorax                            0.5253
          Pleural Effusion                            0.4518
          

In [ ]:
# ==============================================================================
# Frozen Training Split Pathology Audit & Live Cache Representation Check
# ==============================================================================
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

CACHE = Path("/content/dp_mmfl_image_cache_256/train")

assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"
assert CACHE.exists(), f"Cache directory not found: {CACHE}"

df = pd.read_parquet(MANIFEST)
df = df[df["experiment_split"] == "train"].copy()

target_cols = [c for c in df.columns if c.startswith("target_")]

rows = []
for target in target_cols:
    label_name = target.replace("target_", "")
    mask_col = f"mask_{label_name}"

    observed = df[mask_col].fillna(0).astype(float) > 0
    labels = pd.to_numeric(df[target], errors="coerce")
    valid = observed & labels.notna()
    y = labels[valid]

    pos_count = int((y == 1).sum())
    neg_count = int((y == 0).sum())
    valid_count = int(valid.sum())

    rows.append({
        "Pathology": label_name,
        "Observed": valid_count,
        "Positive": pos_count,
        "Negative": neg_count,
        "Positive %": round(100.0 * (pos_count / max(valid_count, 1)), 2),
    })

audit = pd.DataFrame(rows)
print("=== FROZEN TRAINING LABEL AUDIT (73,640 Frontal Images) ===")
print(audit.to_string(index=False))

# Live local cache representation against the frozen training manifest
cached_ids = {
    int(p.stem) for p in CACHE.glob("*.png")
    if p.stem.isdigit()
}
available = df[df["sample_id"].astype(int).isin(cached_ids)]

print("\n=== LIVE CACHE STATUS ===")
print(f"Cached training images represented in manifest: {len(available):,} / {len(df):,} ({len(available)/len(df):.2%})")
print(f"Cached unique patients:                         {available['deid_patient_id'].nunique():,} / {df['deid_patient_id'].nunique():,}")

worker_status = cache_thread.is_alive() if "cache_thread" in globals() else "Thread variable not in current cell scope (check origin cell)"
print(f"Cache worker active:                             {worker_status}")

=== FROZEN TRAINING LABEL AUDIT (73,640 Frontal Images) ===
                 Pathology  Observed  Positive  Negative  Positive %
Enlarged Cardiomediastinum     13624      2897     10727       21.26
              Cardiomegaly     20256     11730      8526       57.91
              Lung Opacity     38547     36466      2081       94.60
               Lung Lesion      4620      4120       500       89.18
                     Edema     25553     17986      7567       70.39
             Consolidation     15691      4738     10953       30.20
                 Pneumonia      3367      1799      1568       53.43
               Atelectasis     12860     12615       245       98.09
              Pneumothorax     28350      6259     22091       22.08
          Pleural Effusion     45645     31359     14286       68.70
             Pleural Other      1708      1691        17       99.00
                  Fracture      4888      3383      1505       69.21
           Support Devices     45034     44

In [ ]:
# ==============================================================================
# Cache Delta Benchmark (60s Window): Local SSD & Drive Throughput
# ==============================================================================
from pathlib import Path
import time

local = Path("/content/dp_mmfl_image_cache_256/train")
drive = Path("/content/drive/MyDrive/DP-MMFL/data/image_cache_256/train")

local_before = len(list(local.glob("*.png")))
drive_before = len(list(drive.glob("*.png")))

print("Monitoring cache ingestion across 60 seconds...")
time.sleep(60)

local_after = len(list(local.glob("*.png")))
drive_after = len(list(drive.glob("*.png")))

local_delta = local_after - local_before
drive_delta = drive_after - drive_before

print("\n=== CACHE PROGRESS REPORT (60s WINDOW) ===")
print(f"Local SSD delta:       +{local_delta} images ({local_delta / 60.0:.2f} img/s)")
print(f"Drive sync delta:      +{drive_delta} images ({drive_delta / 60.0:.2f} img/s)")
print(f"Local SSD total:       {local_after:,} images")
print(f"Drive total:           {drive_after:,} images")

worker_alive = cache_thread.is_alive() if "cache_thread" in globals() else "N/A"
print(f"Cache worker active:   {worker_alive}")

Monitoring cache ingestion across 60 seconds...
[Cache Worker] Processed: 4,224/73,329 | Rate: 2.91 img/s | Remaining: 6.6 hrs
[Cache Worker] Processed: 4,288/73,329 | Rate: 2.92 img/s | Remaining: 6.6 hrs
[Cache Worker] Processed: 4,352/73,329 | Rate: 2.92 img/s | Remaining: 6.6 hrs

=== CACHE PROGRESS REPORT (60s WINDOW) ===
Local SSD delta:       +175 images (2.92 img/s)
Drive sync delta:      +256 images (4.27 img/s)
Local SSD total:       5,039 images
Drive total:           4,352 images
Cache worker active:   True


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_CACHE = Path("/content/drive/MyDrive/DP-MMFL/data/image_cache_256/train")
total_saved = len(list(DRIVE_CACHE.glob("*.png")))
print(f"Permanently cached in Drive: {total_saved:,} / 73,640 images ({total_saved / 73640:.1%})")

Mounted at /content/drive
Permanently cached in Drive: 0 / 73,640 images (0.0%)


In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image

ROOT = Path("/content/DP-MMFL")
MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
CACHE = Path("/content/drive/MyDrive/DP-MMFL/data/image_cache_256/train")

df = pd.read_parquet(MANIFEST)
train = df[df["experiment_split"] == "train"].copy()

cached_ids = {
    int(p.stem)
    for p in CACHE.glob("*.png")
    if p.stem.isdigit()
}

manifest_ids = set(train["sample_id"].astype(int))
matched_ids = cached_ids & manifest_ids
extra_ids = cached_ids - manifest_ids

print("Frozen training images:", len(manifest_ids))
print("PNG files in Drive:", len(cached_ids))
print("Matching training images:", len(matched_ids))
print("Unexpected image IDs:", len(extra_ids))
print("Missing training images:", len(manifest_ids - matched_ids))

# Test a sample of cached files for readability.
sample_paths = [
    CACHE / f"{sid:07d}.png"
    for sid in list(matched_ids)[:100]
]

readable = 0
for path in sample_paths:
    try:
        with Image.open(path) as im:
            im.verify()
        readable += 1
    except Exception:
        pass

print(f"Readable sampled files: {readable}/{len(sample_paths)}")

In [ ]:
# ============================================================
# DP-MMFL EXPERIMENT-DAY PREFLIGHT
# Read-only audit of manifests, caches, splits and checkpoints
# ============================================================

from pathlib import Path
import hashlib
import json
import random
import threading
import time

import numpy as np
import pandas as pd
from PIL import Image

# ---------- 1. Paths ----------

DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

FROZEN_MANIFESTS = [
    DRIVE_ROOT / "results/final_protocol/"
    "chexpert_plus_25k_frozen_manifest.parquet",

    Path("/content/DP-MMFL/results/final_protocol/"
         "chexpert_plus_25k_frozen_manifest.parquet"),
]

# Diagnostic reference only. Do not substitute this automatically
# for the frozen 25k-patient experiment manifest.
LEGACY_MANIFEST = (
    DRIVE_ROOT / "data/processed/chexpert_plus_manifest.parquet"
)

CACHE_DIRS = {
    "local_train": Path("/content/dp_mmfl_image_cache_256/train"),
    "drive_train": DRIVE_ROOT / "data/image_cache_256/train",
}

CHECKPOINTS = {
    "pilot_epoch2": (
        DRIVE_ROOT / "checkpoints/pilot_baseline/"
        "resnet18_pilot_epoch2.pt"
    ),
    "E2_multimodal": (
        DRIVE_ROOT / "results/E2_multimodal/checkpoints/best.pt"
    ),
}

AUDIT_DIR = DRIVE_ROOT / "results/experiment_day"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

# ---------- 2. Helpers ----------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def inventory_cache(folder):
    if not folder.exists():
        return {
            "path": str(folder),
            "exists": False,
            "files": [],
            "ids": set(),
            "nonnumeric": 0,
        }

    files = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    ids = {
        int(p.stem)
        for p in files
        if p.stem.isdigit()
    }

    return {
        "path": str(folder),
        "exists": True,
        "files": files,
        "ids": ids,
        "nonnumeric": sum(not p.stem.isdigit() for p in files),
    }


def verify_images(files, max_samples=128, verify_all=False):
    # Ignore recently modified files while a cache worker may be writing.
    stable = [
        p for p in files
        if p.exists()
        and p.stat().st_size > 0
        and time.time() - p.stat().st_mtime > 15
    ]

    if verify_all:
        chosen = stable
    else:
        rng = random.Random(42)
        chosen = rng.sample(stable, min(max_samples, len(stable)))

    bad = []

    for path in chosen:
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception as exc:
            bad.append({
                "file": str(path),
                "error": str(exc)[:150],
            })

    return {
        "checked": len(chosen),
        "bad_count": len(bad),
        "bad_examples": bad[:10],
    }


# ---------- 3. Locate and compare manifests ----------

print("=" * 65)
print("DP-MMFL PREFLIGHT")
print("=" * 65)

found_manifests = [
    p for p in FROZEN_MANIFESTS if p.is_file()
]

manifest_hashes = {
    str(p): sha256_file(p)
    for p in found_manifests
}

print("\nFROZEN MANIFESTS")
for path, digest in manifest_hashes.items():
    print(f"{path}")
    print(f"  SHA256: {digest}")

manifest_hash_conflict = (
    len(set(manifest_hashes.values())) > 1
)

if manifest_hash_conflict:
    print("WARNING: Frozen manifest copies differ.")

manifest_path = (
    found_manifests[0]
    if found_manifests
    else None
)

print("Legacy manifest exists:", LEGACY_MANIFEST.is_file())
print("Frozen manifest found:", manifest_path is not None)

df = None
train_df = None
expected_train_ids = set()
label_audit = pd.DataFrame()
split_overlap = {}
split_col = None
patient_col = None

if manifest_path is not None:
    df = pd.read_parquet(manifest_path)

    print("\nMANIFEST DETAILS")
    print("Selected path:", manifest_path)
    print("Shape:", df.shape)
    print("Columns:", len(df.columns))

    split_col = next(
        (
            c for c in
            ["experiment_split", "split", "data_split"]
            if c in df.columns
        ),
        None,
    )

    patient_col = next(
        (
            c for c in
            ["deid_patient_id", "patient_id"]
            if c in df.columns
        ),
        None,
    )

    if split_col:
        split_values = (
            df[split_col].astype(str).str.lower().str.strip()
        )

        print("\nSPLIT COUNTS")
        print(split_values.value_counts(dropna=False).to_string())

        train_df = df[
            split_values.isin(["train", "training"])
        ].copy()

        if train_df.empty:
            print("WARNING: No rows identified as training.")
    else:
        train_df = df.copy()
        print("WARNING: No recognized split column.")

    if "sample_id" in df.columns:
        sample_ids = pd.to_numeric(
            train_df["sample_id"], errors="coerce"
        ).dropna().astype("int64")

        expected_train_ids = set(sample_ids)

        print("\nTRAINING DATA")
        print("Training rows:", len(train_df))
        print("Unique training sample IDs:", len(expected_train_ids))
        print(
            "Duplicate sample IDs in manifest:",
            int(df["sample_id"].duplicated().sum()),
        )
    else:
        print("WARNING: sample_id column is missing.")

    if patient_col:
        print("Unique training patients:",
              train_df[patient_col].nunique())

        if split_col:
            split_values = (
                df[split_col].astype(str).str.lower().str.strip()
            )

            patient_sets = {
                split_name: set(
                    df.loc[
                        split_values == split_name,
                        patient_col
                    ].dropna().astype(str)
                )
                for split_name in split_values.unique()
            }

            names = list(patient_sets)

            print("\nPATIENT OVERLAP BETWEEN SPLITS")
            for i, left in enumerate(names):
                for right in names[i + 1:]:
                    overlap = len(
                        patient_sets[left] & patient_sets[right]
                    )
                    split_overlap[f"{left}__{right}"] = overlap
                    print(f"{left} vs {right}: {overlap}")

    # Label audit using the existing observation masks.
    if train_df is not None:
        target_cols = [
            c for c in train_df.columns
            if c.startswith("target_")
        ]

        rows = []

        for target in target_cols:
            mask_col = (
                "mask_" + target.removeprefix("target_")
            )

            if mask_col not in train_df.columns:
                continue

            y = pd.to_numeric(
                train_df[target], errors="coerce"
            )
            observed = (
                pd.to_numeric(
                    train_df[mask_col], errors="coerce"
                ).fillna(0) > 0
            )

            valid = observed & y.isin([0, 1])
            positives = int((valid & (y == 1)).sum())
            negatives = int((valid & (y == 0)).sum())
            n_observed = positives + negatives

            rows.append({
                "pathology": target.removeprefix("target_"),
                "observed": n_observed,
                "positive": positives,
                "negative": negatives,
                "positive_pct": (
                    100 * positives / n_observed
                    if n_observed else np.nan
                ),
            })

        label_audit = pd.DataFrame(rows)

        print("\nLABEL AUDIT")
        if not label_audit.empty:
            print(label_audit.to_string(index=False))

# ---------- 4. Cache inventory and integrity ----------

cache_info = {}

for name, folder in CACHE_DIRS.items():
    inv = inventory_cache(folder)
    cache_info[name] = inv

    files = inv["files"]

    zero_byte = [
        str(p) for p in files
        if p.stat().st_size == 0
    ]

    # Check every stable local image if the worker is stopped.
    # While downloading, sample stable local files instead.
    worker_obj = globals().get("cache_thread")
    worker_active = (
        bool(worker_obj.is_alive())
        if worker_obj is not None
        else None
    )

    verify_all = (
        name == "local_train"
        and worker_active is False
    )

    integrity = verify_images(
        files,
        max_samples=128 if name == "local_train" else 64,
        verify_all=verify_all,
    )

    inv["integrity"] = integrity
    inv["zero_byte_count"] = len(zero_byte)

    print(f"\nCACHE: {name}")
    print("Path:", folder)
    print("Exists:", inv["exists"])
    print("Image files:", len(files))
    print("Numeric sample IDs:", len(inv["ids"]))
    print("Non-numeric filenames:", inv["nonnumeric"])
    print("Zero-byte files:", len(zero_byte))
    print("Integrity files checked:", integrity["checked"])
    print("Integrity failures:", integrity["bad_count"])

    if integrity["bad_examples"]:
        print("Bad image examples:", integrity["bad_examples"])

# ---------- 5. Determine download / sync status ----------

local_ids = cache_info["local_train"]["ids"]
drive_ids = cache_info["drive_train"]["ids"]

missing_local = expected_train_ids - local_ids
missing_drive = expected_train_ids - drive_ids
missing_both = expected_train_ids - (local_ids | drive_ids)

local_complete = (
    bool(expected_train_ids)
    and not missing_local
)
drive_complete = (
    bool(expected_train_ids)
    and not missing_drive
)

print("\n" + "=" * 65)
print("CACHE COMPLETION STATUS")
print("=" * 65)

print("Expected training IDs:", len(expected_train_ids))
print("Local cache coverage:", len(expected_train_ids & local_ids))
print("Missing locally:", len(missing_local))
print("Drive cache coverage:", len(expected_train_ids & drive_ids))
print("Missing on Drive:", len(missing_drive))
print("Missing from BOTH locations:", len(missing_both))
print("Complete locally:", local_complete)
print("Complete on Drive:", drive_complete)

worker_obj = globals().get("cache_thread")
worker_active = (
    bool(worker_obj.is_alive())
    if worker_obj is not None
    else None
)

print("Cache worker active:", worker_active)
print(
    "Other active Python threads:",
    [
        t.name for t in threading.enumerate()
        if t is not threading.main_thread()
    ],
)

if not expected_train_ids:
    cache_status = "UNKNOWN: frozen manifest/sample IDs unavailable"
elif local_complete and drive_complete:
    cache_status = "COMPLETE: local and Drive caches"
elif local_complete:
    cache_status = "DOWNLOAD COMPLETE LOCALLY; DRIVE SYNC INCOMPLETE"
elif drive_complete:
    cache_status = "DRIVE COMPLETE; LOCAL STAGING NEEDED"
elif not missing_both:
    cache_status = "ALL IDS EXIST ACROSS CACHES; CONSOLIDATION NEEDED"
elif worker_active is True:
    cache_status = "DOWNLOAD STILL RUNNING"
elif worker_active is False:
    cache_status = "INCOMPLETE: RESUME MISSING IMAGES"
else:
    cache_status = "INCOMPLETE: VERIFY WORKER STATE"

print("\nFINAL CACHE STATUS:", cache_status)

# ---------- 6. Checkpoint inventory ----------

checkpoint_info = {}

for name, path in CHECKPOINTS.items():
    exists = path.is_file()

    checkpoint_info[name] = {
        "path": str(path),
        "exists": exists,
        "size_bytes": path.stat().st_size if exists else None,
    }

    print(f"\nCHECKPOINT: {name}")
    print("Path:", path)
    print("Exists:", exists)

    if exists:
        print("Size (MB):", round(path.stat().st_size / 1e6, 2))

# Find additional experiment checkpoints.
additional_checkpoints = []

for base in [DRIVE_ROOT / "checkpoints",
             DRIVE_ROOT / "results"]:
    if base.exists():
        additional_checkpoints.extend(
            p for p in base.rglob("*.pt")
            if p.is_file()
        )

additional_checkpoints = sorted(
    set(additional_checkpoints),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

print("\nRECENT CHECKPOINT FILES")
for p in additional_checkpoints[:20]:
    print(p)

pilot_path = CHECKPOINTS["pilot_epoch2"]

if pilot_path.is_file():
    try:
        import torch

        try:
            ckpt = torch.load(
                pilot_path,
                map_location="cpu",
                weights_only=False,
            )
        except TypeError:
            ckpt = torch.load(
                pilot_path,
                map_location="cpu",
            )

        print("\nPILOT CHECKPOINT METADATA")
        print("Epoch:", ckpt.get("epoch", "not recorded"))
        print("Model state available:",
              "model_state_dict" in ckpt)
        print("Target columns:",
              ckpt.get("target_cols", "not recorded"))
        print("Recorded training IDs:",
              len(ckpt.get("train_sample_ids", [])))
        print("Recorded validation IDs:",
              len(ckpt.get("validation_sample_ids", [])))

    except Exception as exc:
        print("WARNING: Could not load pilot checkpoint:", exc)

# ---------- 7. Save audit artifacts ----------

if DRIVE_ROOT.exists():
    AUDIT_DIR.mkdir(parents=True, exist_ok=True)

    if not label_audit.empty:
        label_audit.to_csv(
            AUDIT_DIR / "label_audit.csv",
            index=False,
        )

    if train_df is not None and "sample_id" in train_df.columns:
        missing_rows = train_df[
            pd.to_numeric(
                train_df["sample_id"], errors="coerce"
            ).isin(missing_both)
        ]

        missing_rows.to_csv(
            AUDIT_DIR / "missing_from_both_caches.csv",
            index=False,
        )

    report = {
        "manifest_path": (
            str(manifest_path) if manifest_path else None
        ),
        "manifest_hashes": manifest_hashes,
        "manifest_hash_conflict": manifest_hash_conflict,
        "manifest_rows": len(df) if df is not None else None,
        "training_rows": len(train_df) if train_df is not None else None,
        "expected_training_ids": len(expected_train_ids),
        "local_cache_ids": len(local_ids),
        "drive_cache_ids": len(drive_ids),
        "missing_local": len(missing_local),
        "missing_drive": len(missing_drive),
        "missing_both": len(missing_both),
        "local_complete": local_complete,
        "drive_complete": drive_complete,
        "worker_active": worker_active,
        "cache_status": cache_status,
        "split_patient_overlap": split_overlap,
        "checkpoints": checkpoint_info,
        "label_count": len(label_audit),
        "timestamp": pd.Timestamp.now().isoformat(),
    }

    with open(AUDIT_DIR / "preflight_audit.json", "w") as f:
        json.dump(report, f, indent=2, default=str)

    print("\nAudit saved to:", AUDIT_DIR)
else:
    print(
        "\nDrive is not mounted at the expected path. "
        "No audit files were saved."
    )

print("\nPREFLIGHT COMPLETE")

DP-MMFL PREFLIGHT

FROZEN MANIFESTS
/content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
  SHA256: 5dfd914db6d1fb872059f323e214e76ef07b109d9b49bb39151d851326855fa6
Legacy manifest exists: True
Frozen manifest found: True

MANIFEST DETAILS
Selected path: /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
Shape: (111562, 32)
Columns: 32

SPLIT COUNTS
experiment_split
train    73640
test     19349
val      18573

TRAINING DATA
Training rows: 73640
Unique training sample IDs: 73640
Duplicate sample IDs in manifest: 0
Unique training patients: 25000

PATIENT OVERLAP BETWEEN SPLITS
train vs val: 0
train vs test: 0
val vs test: 0

LABEL AUDIT
                 pathology  observed  positive  negative  positive_pct
Enlarged Cardiomediastinum     13624      2897     10727     21.263946
              Cardiomegaly     20256     11730      8526     57.908768
              Lung Opacity     38547     36466      208

In [ ]:
# ==============================================================================
# Cell 1: Pre-flight Disk Check & Atomic Local SSD Hydration
# Uses temporary filenames (.tmp) prior to atomic rename to prevent corrupt PNGs
# ==============================================================================
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import shutil
import time

total, used, free = shutil.disk_usage("/content")
free_gib = free / (1024**3)
print(f"=== Local Disk Pre-flight Check ===")
print(f"Free local storage: {free_gib:.2f} GiB")
assert free_gib > 4.0, f"Insufficient local disk space ({free_gib:.2f} GiB). Need >= 4 GiB."

DRIVE_CACHE = Path("/content/drive/MyDrive/DP-MMFL/data/image_cache_256/train")
LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

assert DRIVE_CACHE.exists(), f"Drive cache path missing: {DRIVE_CACHE}"

drive_files = list(DRIVE_CACHE.glob("*.png"))
existing_local = {p.name for p in LOCAL_CACHE.glob("*.png")}
to_copy = [p for p in drive_files if p.name not in existing_local]

print(f"Total images on Drive:     {len(drive_files):,}")
print(f"Already on local SSD:      {len(existing_local):,}")
print(f"Transferring to SSD:       {len(to_copy):,} images...")

def safe_copy_worker(src_path: Path):
    dst_final = LOCAL_CACHE / src_path.name
    dst_tmp = LOCAL_CACHE / f".{src_path.name}.tmp"
    try:
        shutil.copyfile(src_path, dst_tmp)
        dst_tmp.rename(dst_final)
    except Exception as exc:
        if dst_tmp.exists():
            dst_tmp.unlink(missing_ok=True)
        return src_path.name, repr(exc)
    return None

start_t = time.perf_counter()
failures = []

with ThreadPoolExecutor(max_workers=16) as pool:
    results = pool.map(safe_copy_worker, to_copy)
    for res in results:
        if res is not None:
            failures.append(res)

elapsed = time.perf_counter() - start_t
total_local = len(list(LOCAL_CACHE.glob("*.png")))

print(f"\n=== Hydration Complete ===")
print(f"Elapsed time:              {elapsed:.1f} s")
print(f"Failures encountered:      {len(failures)}")
print(f"Total valid PNGs on SSD:   {total_local:,}")

if failures:
    print(f"First 5 failure instances: {failures[:5]}")

=== Local Disk Pre-flight Check ===
Free local storage: 70.21 GiB
Total images on Drive:     38,912
Already on local SSD:      0
Transferring to SSD:       38,912 images...

=== Hydration Complete ===
Elapsed time:              619.9 s
Failures encountered:      0
Total valid PNGs on SSD:   38,912


In [ ]:
# ==============================================================================
# Cell 2: Rigorous Cohort Audit (Patient-Level Integrity & Label Counts)
# Strictly validates zero unexpected sample_ids and verifies observed pos/neg
# ==============================================================================
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")

assert MANIFEST.exists(), f"Manifest not found: {MANIFEST}"
assert LOCAL_CACHE.exists(), f"Cache not found: {LOCAL_CACHE}"

# 1. Load full frozen manifest
df = pd.read_parquet(MANIFEST)
train_df = df[df["experiment_split"] == "train"].copy()
train_df["sample_id"] = train_df["sample_id"].astype(int)

# 2. Match local cache against training manifest
ssd_stems = {
    int(p.stem) for p in LOCAL_CACHE.glob("*.png")
    if p.stem.isdigit()
}

expected_train_ids = set(train_df["sample_id"])
matching_ids = ssd_stems & expected_train_ids
unexpected_ids = ssd_stems - expected_train_ids

active_df = train_df[train_df["sample_id"].isin(matching_ids)].copy().reset_index(drop=True)

# 3. Patient-level statistics
total_train_patients = train_df["deid_patient_id"].nunique()
active_patients = active_df["deid_patient_id"].nunique()

print("=== 1. MANIFEST & CACHE COHORT INTEGRITY ===")
print(f"Local PNG files scanned:         {len(ssd_stems):,}")
print(f"Matching frozen train images:    {len(matching_ids):,} / {len(train_df):,} ({len(matching_ids)/len(train_df):.2%})")
print(f"Unexpected / Out-of-split IDs:   {len(unexpected_ids):,}")
print(f"Distinct patients represented:   {active_patients:,} / {total_train_patients:,} ({active_patients/total_train_patients:.2%})")
print(f"Mean images per patient:         {len(active_df)/max(active_patients, 1):.2f}")

assert len(unexpected_ids) == 0, f"Data leakage error: {len(unexpected_ids)} local files do not belong to train split."

# 4. Pathology Observation & Class Imbalance Audit
LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

audit_records = []
for label in LABELS:
    target_col = f"target_{label}"
    mask_col = f"mask_{label}"

    # Must only count where mask == 1.0 and target is non-null
    observed_mask = (active_df[mask_col].fillna(0).astype(float) == 1.0)
    target_vals = active_df.loc[observed_mask, target_col].dropna().astype(float)

    pos_count = int((target_vals == 1.0).sum())
    neg_count = int((target_vals == 0.0).sum())
    total_obs = len(target_vals)
    pos_pct = (pos_count / total_obs * 100.0) if total_obs > 0 else float("nan")

    audit_records.append({
        "Pathology": label,
        "Observed": total_obs,
        "Positive": pos_count,
        "Negative": neg_count,
        "Positive %": f"{pos_pct:.2f}%",
        "Eval Viable (>5 neg & >5 pos)": (pos_count >= 5 and neg_count >= 5)
    })

audit_table = pd.DataFrame(audit_records)
print("\n=== 2. OBSERVED PATHOLOGY DISTRIBUTION (LOCAL SUBSET) ===")
print(audit_table.to_string(index=False))

=== 1. MANIFEST & CACHE COHORT INTEGRITY ===
Local PNG files scanned:         38,912
Matching frozen train images:    38,912 / 73,640 (52.84%)
Unexpected / Out-of-split IDs:   0
Distinct patients represented:   12,696 / 25,000 (50.78%)
Mean images per patient:         3.06

=== 2. OBSERVED PATHOLOGY DISTRIBUTION (LOCAL SUBSET) ===
                 Pathology  Observed  Positive  Negative Positive %  Eval Viable (>5 neg & >5 pos)
Enlarged Cardiomediastinum      7101      1489      5612     20.97%                           True
              Cardiomegaly     10841      6348      4493     58.56%                           True
              Lung Opacity     20257     19202      1055     94.79%                           True
               Lung Lesion      2458      2188       270     89.02%                           True
                     Edema     13476      9445      4031     70.09%                           True
             Consolidation      8343      2521      5822     30.22%      

In [ ]:
# ==============================================================================
# Centralized ResNet-18 Baseline: 38.9k Cohort, Patient-Disjoint, DP-Ready Loss
# ==============================================================================
import time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# 1. Paths & Setup
ROOT = Path("/content/DP-MMFL")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")

MANIFEST = ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"

LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints/centralized_baseline_38k"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]
target_cols = [f"target_{c}" for c in LABELS]
mask_cols = [f"mask_{c}" for c in LABELS]

# 2. Filter Manifest to Hydrated SSD Images
df = pd.read_parquet(MANIFEST)
train_df = df[df["experiment_split"] == "train"].copy()
train_df["sample_id"] = train_df["sample_id"].astype(int)

ssd_ids = {int(p.stem) for p in LOCAL_CACHE.glob("*.png") if p.stem.isdigit()}
active_df = train_df[train_df["sample_id"].isin(ssd_ids)].copy().reset_index(drop=True)

# 3. Patient-Disjoint Internal Validation Split (85/15)
groups = active_df["deid_patient_id"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, va_idx = next(splitter.split(active_df, groups=groups))

tr_df = active_df.iloc[tr_idx].copy().reset_index(drop=True)
va_df = active_df.iloc[va_idx].copy().reset_index(drop=True)

overlap = len(set(tr_df["deid_patient_id"]) & set(va_df["deid_patient_id"]))
assert overlap == 0, f"Patient leakage detected: {overlap} shared patients!"

print("=== COHORT PARTITION ===")
print(f"Training samples:   {len(tr_df):,} images ({tr_df['deid_patient_id'].nunique():,} patients)")
print(f"Validation samples: {len(va_df):,} images ({va_df['deid_patient_id'].nunique():,} patients)")
print(f"Patient overlap:    {overlap}")

# 4. Dataset & Transforms
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FastCachedCheXpertDataset(Dataset):
    def __init__(self, frame, cache_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.cache_dir = cache_dir
        self.transform = transform
        self.sample_ids = self.frame["sample_id"].values.astype(int)
        self.targets = self.frame[target_cols].values.astype(np.float32)
        self.masks = self.frame[mask_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        img_path = self.cache_dir / f"{sid:07d}.png"
        with Image.open(img_path) as img:
            t_img = self.transform(img.convert("L"))

        t = torch.nan_to_num(torch.from_numpy(self.targets[idx]), nan=0.0)
        m = torch.from_numpy(self.masks[idx])
        return t_img, t, m

train_loader = DataLoader(
    FastCachedCheXpertDataset(tr_df, LOCAL_CACHE, train_tf),
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    FastCachedCheXpertDataset(va_df, LOCAL_CACHE, val_tf),
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# 5. Model, Loss & Optimizer
def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    numerator = (element_loss * mask).sum(dim=1)
    denominator = mask.sum(dim=1).clamp_min(1.0)
    return numerator / denominator

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(LABELS))
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

# 6. Evaluation Function
@torch.no_grad()
def evaluate_metrics(net, loader):
    net.eval()
    total_loss, total_count = 0.0, 0
    all_preds, all_targets, all_masks = [], [], []

    for imgs, tgts, msks in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        tgts = tgts.to(DEVICE, non_blocking=True)
        msks = msks.to(DEVICE, non_blocking=True)

        logits = net(imgs)
        loss = per_image_masked_bce(logits, tgts, msks).mean()

        total_loss += loss.item() * len(imgs)
        total_count += len(imgs)

        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(tgts.cpu().numpy())
        all_masks.append(msks.cpu().numpy())

    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    masks = np.vstack(all_masks)

    per_class_auc = {}
    support_stats = {}

    for j, label in enumerate(LABELS):
        valid = masks[:, j] == 1.0
        y_true = targets[valid, j]
        y_score = preds[valid, j]

        pos_count = int((y_true == 1.0).sum())
        neg_count = int((y_true == 0.0).sum())
        support_stats[label] = {"pos": pos_count, "neg": neg_count}

        if pos_count > 0 and neg_count > 0:
            try:
                per_class_auc[label] = roc_auc_score(y_true, y_score)
            except ValueError:
                per_class_auc[label] = np.nan
        else:
            per_class_auc[label] = np.nan

    valid_aucs = [v for v in per_class_auc.values() if not np.isnan(v)]
    macro_auc = float(np.mean(valid_aucs)) if valid_aucs else float("nan")
    val_loss = total_loss / max(total_count, 1)

    return val_loss, macro_auc, per_class_auc, support_stats

# 7. Training Execution (5 Epochs)
NUM_EPOCHS = 5
print(f"\n=== LAUNCHING CENTRALIZED BASELINE ({NUM_EPOCHS} EPOCHS) ===")
best_auc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, seen = 0.0, 0
    t_start = time.perf_counter()

    for step, (imgs, tgts, msks) in enumerate(train_loader, start=1):
        imgs = imgs.to(DEVICE, non_blocking=True)
        tgts = tgts.to(DEVICE, non_blocking=True)
        msks = msks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss = per_image_masked_bce(logits, tgts, msks).mean()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(imgs)
        seen += len(imgs)

    scheduler.step()
    epoch_duration = time.perf_counter() - t_start
    train_loss = running_loss / max(seen, 1)

    val_loss, macro_auc, class_auc, stats = evaluate_metrics(model, val_loader)

    print(
        f"Epoch [{epoch}/{NUM_EPOCHS}] ({epoch_duration:.1f}s) | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro AUROC: {macro_auc:.4f} "
        f"({sum(~np.isnan(list(class_auc.values())))}/{len(LABELS)} classes evaluable)"
    )

    # Persist Checkpoint & Evaluation Metadata
    is_best = macro_auc > best_auc
    if is_best:
        best_auc = macro_auc

    ckpt_payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auc": macro_auc,
        "per_class_auc": class_auc,
        "support_stats": stats,
        "labels": LABELS,
        "num_training_images": len(tr_df),
        "num_val_images": len(va_df),
    }

    torch.save(ckpt_payload, CHECKPOINT_DIR / f"resnet18_epoch_{epoch}.pt")
    if is_best:
        torch.save(ckpt_payload, CHECKPOINT_DIR / "resnet18_best.pt")

print(f"\nBaseline training complete. Checkpoints saved to: {CHECKPOINT_DIR}")

=== COHORT PARTITION ===
Training samples:   32,934 images (10,791 patients)
Validation samples: 5,978 images (1,905 patients)
Patient overlap:    0
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100%|██████████| 44.7M/44.7M [00:00<00:00,


=== LAUNCHING CENTRALIZED BASELINE (5 EPOCHS) ===


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [1/5] (110.4s) | Train Loss: 0.4167 | Val Loss: 0.4013 | Val Macro AUROC: 0.4996 (13/13 classes evaluable)


...

Epoch [3/5] (128.1s) | Train Loss: 0.3905 | Val Loss: 0.4067 | Val Macro AUROC: 0.5120 (13/13 classes evaluable)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [4/5] (129.1s) | Train Loss: 0.3570 | Val Loss: 0.4220 | Val Macro AUROC: 0.5131 (13/13 classes evaluable)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [5/5] (129.4s) | Train Loss: 0.3035 | Val Loss: 0.4358 | Val Macro AUROC: 0.5082 (13/13 classes evaluable)

Baseline training complete. Checkpoints saved to: /content/drive/MyDrive/DP-MMFL/checkpoints/centralized_baseline_38k


In [ ]:
# ==============================================================================
# Inspect Per-Class AUROC and Class Supports from Best Checkpoint
# ==============================================================================
from pathlib import Path
import pandas as pd
import torch

CHECKPOINT_PATH = Path("/content/drive/MyDrive/DP-MMFL/checkpoints/centralized_baseline_38k/resnet18_best.pt")
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

print(f"=== Best Model Performance Summary (Epoch {ckpt['epoch']}) ===")
print(f"Train Loss:      {ckpt['train_loss']:.4f}")
print(f"Val Loss:        {ckpt['val_loss']:.4f}")
print(f"Macro Val AUROC: {ckpt['val_macro_auc']:.4f}\n")

per_class_auc = ckpt.get("per_class_auc", {})
support_stats = ckpt.get("support_stats", {})

rows = []
for label in ckpt["labels"]:
    stats = support_stats.get(label, {})
    pos = stats.get("pos", 0)
    neg = stats.get("neg", 0)
    auc = per_class_auc.get(label, float("nan"))
    rows.append({
        "Pathology": label,
        "Val Pos": pos,
        "Val Neg": neg,
        "Observed": pos + neg,
        "Val AUROC": f"{auc:.4f}" if not pd.isna(auc) else "NaN"
    })

audit_df = pd.DataFrame(rows)
print(audit_df.to_string(index=False))

=== Best Model Performance Summary (Epoch 2) ===
Train Loss:      0.4003
Val Loss:        0.3998
Macro Val AUROC: 0.5191

                 Pathology  Val Pos  Val Neg  Observed Val AUROC
Enlarged Cardiomediastinum      220      856      1076    0.5336
              Cardiomegaly      943      683      1626    0.5390
              Lung Opacity     2949      154      3103    0.5427
               Lung Lesion      338       35       373    0.5242
                     Edema     1465      587      2052    0.5203
             Consolidation      397      875      1272    0.5146
                 Pneumonia      144      154       298    0.4881
               Atelectasis     1038       17      1055    0.4818
              Pneumothorax      553     1712      2265    0.5287
          Pleural Effusion     2541     1096      3637    0.5444
             Pleural Other      165        3       168    0.4283
                  Fracture      316      107       423    0.5840
           Support Devices     35

In [ ]:
# ==============================================================================
# DP-MMFL: Inventory Code Assets across src/, scripts/, and notebooks/
# Identifies existing text encoders, fusion modules, FL harnesses, and DP hooks
# ==============================================================================
from pathlib import Path

ROOT = Path("/content/DP-MMFL")

terms = (
    "text", "multimodal", "fusion", "federated", "fedavg",
    "privacy", "adaptive", "fairness", "robust", "compress",
    "evaluate", "train", "model"
)

for folder in ("src/dp_mmfl", "scripts", "notebooks"):
    base = ROOT / folder
    print(f"\n=== {folder} ===")
    if base.exists():
        matched_files = [
            p for p in sorted(base.rglob("*"))
            if p.is_file() and any(t in p.name.lower() for t in terms)
        ]
        if matched_files:
            for p in matched_files:
                print(f"{p.relative_to(ROOT):<60} ({p.stat().st_size:,} bytes)")
        else:
            print("No matching files found.")
    else:
        print("Directory does not exist.")


=== src/dp_mmfl ===
Directory does not exist.

=== scripts ===
Directory does not exist.

=== notebooks ===
Directory does not exist.


In [ ]:
# ==============================================================================
# Environment Asset Locator: Scan /content and Drive for Existing Modules
# ==============================================================================
from pathlib import Path

search_roots = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/DP-MMFL")
]

terms = ("dp_mmfl", "fusion", "multimodal", "federated", "text", "chexpert")

print("=== Scanning for Existing Code & Modules ===")
found_roots = set()

for root in search_roots:
    if not root.exists():
        continue
    for p in root.glob("*"):
        if any(t in p.name.lower() for t in terms):
            print(f"Candidate path: {p} (type: {'dir' if p.is_dir() else 'file'})")
            if p.is_dir():
                found_roots.add(p)

print("\n=== Manifest & Text Columns Check ===")
import pandas as pd

MANIFEST = Path("/content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet")
if not MANIFEST.exists():
    MANIFEST = Path("/content/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet")

if MANIFEST.exists():
    df_sample = pd.read_parquet(MANIFEST).head(5)
    print(f"Manifest found at: {MANIFEST}")
    text_cols = [c for c in df_sample.columns if any(k in c.lower() for k in ("text", "report", "impression", "findings"))]
    print(f"Available text/report columns: {text_cols}")
    demographic_cols = [c for c in df_sample.columns if any(k in c.lower() for k in ("sex", "gender", "age", "race", "ethnicity"))]
    print(f"Available demographic columns (for E6 Fairness): {demographic_cols}")
else:
    print("Manifest not found in standard paths.")

=== Scanning for Existing Code & Modules ===
Candidate path: /content/dp_mmfl_image_cache_256 (type: dir)

=== Manifest & Text Columns Check ===
Manifest found at: /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
Available text/report columns: ['report']
Available demographic columns (for E6 Fairness): ['path_to_image']


In [ ]:
# ==============================================================================
# Comprehensive Data & Codebase Audit:
# 1. Manifest split & report content completeness
# 2. Report string inspection for explicit pathology leakage
# 3. Cache coverage against the frozen training manifest
# 4. Source tree scan for pre-existing .py and .ipynb implementations
# ==============================================================================
import re
from pathlib import Path
import pandas as pd

ROOT = Path("/content/DP-MMFL")
MANIFEST = Path(
    "/content/drive/MyDrive/DP-MMFL/results/"
    "final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
)
CACHE = Path("/content/dp_mmfl_image_cache_256/train")

assert MANIFEST.exists(), f"Frozen manifest not found at: {MANIFEST}"
assert CACHE.exists(), f"Local image cache directory not found at: {CACHE}"

df = pd.read_parquet(MANIFEST)
labels = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

print("=== 1. MANIFEST STRUCTURE & REPORTS ===")
print("Shape:           ", df.shape)
print("Split counts:\n", df["experiment_split"].value_counts().to_string())
missing_rep = int(df["report"].isna().sum())
empty_rep = int(df["report"].fillna("").str.strip().eq("").sum())
print(f"\nReports missing:  {missing_rep:,}")
print(f"Reports empty:    {empty_rep:,}")

print("\n=== 2. REPORT SAMPLES (CHECKING FOR DIRECT LABEL LEAKAGE) ===")
for i, report in enumerate(df["report"].dropna().head(3), 1):
    cleaned = str(report).strip()
    print(f"\n--- Report {i} ---")
    print(cleaned[:600] + ("..." if len(cleaned) > 600 else ""))

print("\n=== 3. CACHE COVERAGE ===")
cache_ids = {
    int(p.stem) for p in CACHE.glob("*.png")
    if p.stem.isdigit()
}
train_mask = df["experiment_split"].astype(str).str.lower().eq("train")
train = df[train_mask]
matched_ids = int(train["sample_id"].astype(int).isin(cache_ids).sum())
print(f"Total cached PNG files on SSD:        {len(cache_ids):,}")
print(f"Cached IDs belonging to train split:  {matched_ids:,} / {len(train):,} ({matched_ids/len(train):.2%})")

print("\n=== 4. EXISTING MODULES AUDIT ===")
found_any_code = False
for folder in ["src/dp_mmfl", "scripts", "notebooks"]:
    base = ROOT / folder
    print(f"\n{folder}:")
    if base.exists():
        py_files = [p for p in sorted(base.rglob("*")) if p.is_file() and p.suffix in {".py", ".ipynb"}]
        if py_files:
            found_any_code = True
            for p in py_files:
                print(f"  {p.relative_to(ROOT)} ({p.stat().st_size:,} bytes)")
        else:
            print("  (Directory exists, but no .py or .ipynb files found)")
    else:
        print("  (Directory does not exist)")

if not found_any_code:
    print("\nNote: No custom Python source modules detected on disk under /content/DP-MMFL. All workflows will run via direct notebook-defined execution.")

=== 1. MANIFEST STRUCTURE & REPORTS ===
Shape:            (111562, 32)
Split counts:
 experiment_split
train    73640
test     19349
val      18573

Reports missing:  0
Reports empty:    0

=== 2. REPORT SAMPLES (CHECKING FOR DIRECT LABEL LEAKAGE) ===

--- Report 1 ---
NARRATIVE:
CHEST, ONE VIEW: 2-10-2001
FINDINGS: Costophrenic angles sharp, without evidence of effusion.
The cardiomediastinal silhouette is normal. Vessels mildly
indistinct with prominence of interstitial structures, suggesting
mild, pulmonary edema. Left subclavian central venous catheter is
seen, tip in mid SVC. No pneumothorax.
IMPRESSION:
1. NO EVIDENCE OF PNEUMOTHORAX.
2. MILD INTERSTITIAL PULMONARY EDEMA.
END OF IMPRESSION:
SUMMARY: 2
I have personally reviewed the images for this examination and agree
with the report transcribed above.
By: Dr. Juarez Tucker N  on: 2-10-2001
 _________...

--- Report 2 ---
NARRATIVE:
CHEST X-RAY: 4/27/07
COMPARISON: 4/27/2007.
CLINICAL HISTORY: Dyspnea. Multiple myeloma.
IMPRESSI

In [ ]:

# DP-MMFL — Report-only leakage audit
# Uses frozen train/val splits; does not touch the test set.

import re
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

MANIFEST = Path(
    "/content/drive/MyDrive/DP-MMFL/results/"
    "final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
)
OUT_DIR = Path("/content/drive/MyDrive/DP-MMFL/results/experiments")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(MANIFEST)
df["report"] = df["report"].fillna("").astype(str)

# Infer label/mask columns from the manifest schema.
target_cols = [c for c in df.columns if c.startswith("target_")]
mask_cols = [c for c in df.columns if c.startswith("mask_")]

assert target_cols, f"No target_* columns found. Columns: {df.columns.tolist()}"
assert len(target_cols) == len(mask_cols), (
    f"Target/mask count mismatch: {len(target_cols)} vs {len(mask_cols)}"
)

train = df[df["experiment_split"].astype(str).str.lower() == "train"].copy()
val = df[df["experiment_split"].astype(str).str.lower().isin(["val", "valid", "validation"])].copy()

print("Train reports:", len(train), "| Validation reports:", len(val))
print("Target columns:", len(target_cols))

# Fit vocabulary only on training reports to prevent validation leakage.
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=3,
    strip_accents="unicode",
    sublinear_tf=True,
    dtype=np.float32
)

X_train = vectorizer.fit_transform(train["report"])
X_val = vectorizer.transform(val["report"])

rows = []

for target_col in target_cols:
    suffix = target_col[len("target_"):]
    mask_col = "mask_" + suffix

    if mask_col not in train.columns or mask_col not in val.columns:
        print("Skipping missing mask:", target_col)
        continue

    ytr = pd.to_numeric(train[target_col], errors="coerce").to_numpy()
    mtr = pd.to_numeric(train[mask_col], errors="coerce").fillna(0).to_numpy().astype(bool)
    yv = pd.to_numeric(val[target_col], errors="coerce").to_numpy()
    mv = pd.to_numeric(val[mask_col], errors="coerce").fillna(0).to_numpy().astype(bool)

    # Train/evaluate only on observed labels. Missing labels are not negatives.
    tr_idx = mtr & np.isfinite(ytr)
    va_idx = mv & np.isfinite(yv)

    y_train = ytr[tr_idx].astype(int)
    y_val = yv[va_idx].astype(int)

    if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2:
        print(f"Skipping {suffix}: only one class in train or validation")
        continue

    model = LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        class_weight="balanced",
        random_state=42
    )
    model.fit(X_train[tr_idx], y_train)
    scores = model.predict_proba(X_val[va_idx])[:, 1]
    auc = roc_auc_score(y_val, scores)

    rows.append({
        "label": suffix,
        "train_observed": int(tr_idx.sum()),
        "val_observed": int(va_idx.sum()),
        "val_positive_rate": float(y_val.mean()),
        "report_only_val_auroc": float(auc)
    })
    print(f"{suffix:35s} AUROC={auc:.4f} | val n={va_idx.sum()}")

results = pd.DataFrame(rows)
if not results.empty:
    print("\nMacro AUROC:", round(results["report_only_val_auroc"].mean(), 4))
    print(results.to_string(index=False))
    out_path = OUT_DIR / "report_only_leakage_audit.csv"
    results.to_csv(out_path, index=False)
    print("\nSaved:", out_path)
else:
    print("No evaluable labels. Check target/mask column names and values.")


Train reports: 73640 | Validation reports: 18573
Target columns: 13
Enlarged Cardiomediastinum          AUROC=0.5184 | val n=3416
Cardiomegaly                        AUROC=0.5151 | val n=5252
Lung Opacity                        AUROC=0.5341 | val n=9849
Lung Lesion                         AUROC=0.5175 | val n=1176
Edema                               AUROC=0.4989 | val n=6393
Consolidation                       AUROC=0.5083 | val n=3911
Pneumonia                           AUROC=0.5417 | val n=859
Atelectasis                         AUROC=0.4981 | val n=3349
Pneumothorax                        AUROC=0.5144 | val n=7100
Pleural Effusion                    AUROC=0.5256 | val n=11329
Pleural Other                       AUROC=0.4254 | val n=457
Fracture                            AUROC=0.5370 | val n=1359
Support Devices                     AUROC=0.4931 | val n=11190

Macro AUROC: 0.5098
                     label  train_observed  val_observed  val_positive_rate  report_only_val_auroc
Enlarg

In [ ]:
# ==============================================================================
# DP-MMFL: Locate Manifest + Comprehensive Text Leakage Audit (Read-Only)
# ==============================================================================
import os
import re
import glob
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# 1. Locate Manifest
candidates = [
    "/content/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet",
    "/content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet",
]

path = next((p for p in candidates if os.path.isfile(p)), None)

if path is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Drive mount note:", str(e)[:250])
    path = next((p for p in candidates if os.path.isfile(p)), None)

if path is None:
    roots = [
        "/content/DP-MMFL",
        "/content/drive/MyDrive/DP-MMFL",
        "/content/drive/MyDrive",
    ]
    found = []
    for r in roots:
        if os.path.isdir(r):
            found.extend(glob.glob(f"{r}/**/chexpert_plus_*frozen_manifest.parquet", recursive=True))
            found.extend(glob.glob(f"{r}/**/chexpert*manifest*.parquet", recursive=True))
    if found:
        path = sorted(found, key=len)[0]

if path is None:
    print("MANIFEST NOT FOUND")
    print("Directories checked: /content, /content/drive/MyDrive, /content/drive/MyDrive/DP-MMFL")
else:
    print(f"=== MANIFEST FOUND ===")
    print(f"Path: {path}\n")

    df = pd.read_parquet(path)
    train_df = df[df["experiment_split"].astype(str).str.lower() == "train"].copy()

    # 2. Identify Text & Section Columns
    text_cols = [c for c in df.columns if any(k in c.lower() for k in ("report", "text", "impression", "finding", "history", "indication"))]
    print(f"Available text/report columns: {text_cols}")

    # 3. Target Pathologies Audit
    LABELS = [
        "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
        "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
        "Atelectasis", "Pneumothorax", "Pleural Effusion",
        "Pleural Other", "Fracture", "Support Devices"
    ]

    report_series = train_df["report"].dropna().astype(str)

    # 4. Check for Section Breakdown (Clinical History vs Impression/Findings)
    section_patterns = {
        "Clinical History / Indication": r"(CLINICAL HISTORY|INDICATION|REASON FOR EXAM|HISTORY):?",
        "Findings": r"(FINDINGS|TECHNIQUE):?",
        "Impression": r"(IMPRESSION|SUMMARY|CONCLUSION):?",
    }

    section_stats = {}
    for name, pattern in section_patterns.items():
        count = report_series.str.contains(pattern, case=False, regex=True).sum()
        section_stats[name] = {
            "Present Count": count,
            "Percentage": f"{100.0 * count / max(len(report_series), 1):.2f}%"
        }

    print("\n=== REPORT STRUCTURAL SECTIONS ===")
    print(pd.DataFrame(section_stats).T.to_string())

    # 5. Measure Label Leakage Across Reports
    print("\n=== EXPLICIT TARGET LABEL LEAKAGE RISK ===")
    leak_records = []
    for label in LABELS:
        # Construct boundary-aware regex for the pathology term
        pattern = r"\b" + re.escape(label.lower()) + r"\b"
        total_mentions = report_series.str.lower().str.contains(pattern, regex=True).sum()

        target_col = f"target_{label}"
        mask_col = f"mask_{label}"

        pos_mentions = 0
        if target_col in train_df.columns and mask_col in train_df.columns:
            obs = (train_df[mask_col].fillna(0).astype(float) == 1.0)
            is_pos = (train_df.loc[obs, target_col].astype(float) == 1.0)
            pos_indices = train_df.loc[obs].loc[is_pos].index
            pos_reports = train_df.loc[pos_indices, "report"].dropna().astype(str)
            pos_mentions = pos_reports.str.lower().str.contains(pattern, regex=True).sum()

        leak_records.append({
            "Pathology": label,
            "Total Text Mentions": total_mentions,
            "Report Mention %": f"{100.0 * total_mentions / max(len(report_series), 1):.2f}%",
            "Mentions in Positive Cases": pos_mentions
        })

    print(pd.DataFrame(leak_records).to_string(index=False))

    # 6. Print 2 Sample Section Extractions
    print("\n=== EXTRACTED SECTION SAMPLES ===")
    sample_reports = report_series.head(2).tolist()
    for idx, rep in enumerate(sample_reports, 1):
        print(f"\n--- Sample {idx} Extracted Breakdown ---")
        history_match = re.search(r"(?:CLINICAL HISTORY|INDICATION):?\s*(.*?)(?=(?:FINDINGS|IMPRESSION|TECHNIQUE|COMPARISON|$))", rep, re.DOTALL | re.IGNORECASE)
        hist = history_match.group(1).strip() if history_match else "None Found"
        print(f"[Clinical History / Indication]: {hist[:200]}")

        findings_match = re.search(r"FINDINGS:?\s*(.*?)(?=(?:IMPRESSION|END OF IMPRESSION|$))", rep, re.DOTALL | re.IGNORECASE)
        find = findings_match.group(1).strip() if findings_match else "None Found"
        print(f"[Findings Excerpt]: {find[:200]}...")

=== MANIFEST FOUND ===
Path: /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet

Available text/report columns: ['report']

=== REPORT STRUCTURAL SECTIONS ===
                              Present Count Percentage
Clinical History / Indication         58115     78.92%
Findings                              34140     46.36%
Impression                            73640    100.00%

=== EXPLICIT TARGET LABEL LEAKAGE RISK ===
                 Pathology  Total Text Mentions Report Mention %  Mentions in Positive Cases
Enlarged Cardiomediastinum                    1            0.00%                           0
              Cardiomegaly                 9556           12.98%                        1549
              Lung Opacity                  316            0.43%                         153
               Lung Lesion                   31            0.04%                           2
                     Edema                30678           41.66%   

In [ ]:
# ================================================================
# DP-MMFL E2 PILOT: IMAGE + CLINICAL HISTORY / INDICATION
# One-cell, self-contained Colab experiment
#
# Key Features:
#   - Clinical History / Indication only (strictly excludes Findings/Impression)
#   - Explicit missingness handling (indicator token + zero representation)
#   - Patient-disjoint internal train/validation split
#   - GroupNorm ResNet-18 (DP-ready) + lightweight MiniLM text branch
#   - Per-image masked BCE loss with clamped class-imbalance weights
#   - Metric computation with observed positive/negative support auditing
# ================================================================
import copy
import glob
import os
import random
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

# 1. Reproducibility & Device Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {DEVICE}")

# 2. Paths & Directory Setup
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints/e2_multimodal_pilot"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_CANDIDATES = [
    DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet",
    Path("/content/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"),
]

manifest_path = next((p for p in MANIFEST_CANDIDATES if p.is_file()), None)
if manifest_path is None:
    found = glob.glob("/content/**/chexpert_plus_*frozen_manifest.parquet", recursive=True)
    if found:
        manifest_path = Path(found[0])

assert manifest_path is not None, "Frozen manifest not found. Verify Google Drive mount."
print(f"Manifest located: {manifest_path}")

LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]
target_cols = [f"target_{c}" for c in LABELS]
mask_cols = [f"mask_{c}" for c in LABELS]

# 3. Load & Filter Active Cached Images
df = pd.read_parquet(manifest_path)
train_df = df[df["experiment_split"].astype(str).str.lower() == "train"].copy()
train_df["sample_id"] = train_df["sample_id"].astype(int)

cached_sids = {int(p.stem) for p in LOCAL_CACHE.glob("*.png") if p.stem.isdigit()}
active_df = train_df[train_df["sample_id"].isin(cached_sids)].copy().reset_index(drop=True)
print(f"Active cached cohort: {len(active_df):,} images across {active_df['deid_patient_id'].nunique():,} patients")

# 4. Strict Clinical History Extraction (Zero Leakage)
def extract_clinical_history(report_text):
    if not isinstance(report_text, str) or not report_text.strip():
        return "", False

    # Extract strictly between History/Indication markers and any subsequent diagnostic sections
    pattern = (
        r"(?:CLINICAL HISTORY|INDICATION|REASON FOR EXAM|HISTORY):?\s*"
        r"(.*?)(?=(?:FINDINGS|IMPRESSION|TECHNIQUE|COMPARISON|SUMMARY|END OF IMPRESSION|$))"
    )
    match = re.search(pattern, report_text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        extracted = " ".join(match.group(1).split()).strip()
        # Ensure it is not an empty capture or stray artifact
        if len(extracted) > 2:
            return extracted[:256], True

    return "", False

extracted_data = [extract_clinical_history(r) for r in active_df["report"]]
active_df["clinical_history"] = [t[0] for t in extracted_data]
active_df["has_history"] = [t[1] for t in extracted_data]

has_hist_count = int(active_df["has_history"].sum())
print("\n=== CLINICAL HISTORY AUDIT ===")
print(f"Images with extracted history: {has_hist_count:,} / {len(active_df):,} ({has_hist_count / len(active_df):.2%})")
print(f"Images with missing history:   {len(active_df) - has_hist_count:,}")

print("\n--- Sample Extracted Clinical Histories ---")
samples_seen = 0
for idx, row in active_df.iterrows():
    if row["has_history"] and samples_seen < 3:
        print(f"Sample {samples_seen + 1}: {row['clinical_history']}")
        samples_seen += 1

# 5. Patient-Disjoint Internal Split
groups = active_df["deid_patient_id"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, va_idx = next(splitter.split(active_df, groups=groups))

tr_df = active_df.iloc[tr_idx].copy().reset_index(drop=True)
va_df = active_df.iloc[va_idx].copy().reset_index(drop=True)

overlap = len(set(tr_df["deid_patient_id"]) & set(va_df["deid_patient_id"]))
assert overlap == 0, f"Patient leakage detected: {overlap} shared patients!"

print(f"\n=== PATIENT-DISJOINT SPLIT ===")
print(f"Train split: {len(tr_df):,} images ({tr_df['deid_patient_id'].nunique():,} patients) | History coverage: {tr_df['has_history'].mean():.2%}")
print(f"Val split:   {len(va_df):,} images ({va_df['deid_patient_id'].nunique():,} patients) | History coverage: {va_df['has_history'].mean():.2%}")

# 6. Precompute Imbalance-Compensating Class Weights
pos_weights = []
for label in LABELS:
    target_col = f"target_{label}"
    mask_col = f"mask_{label}"
    valid = (tr_df[mask_col].fillna(0).astype(float) == 1.0)
    vals = tr_df.loc[valid, target_col].dropna().astype(float)
    pos = float((vals == 1.0).sum())
    neg = float((vals == 0.0).sum())
    pos_weights.append(np.clip(neg / max(pos, 1.0), 0.1, 10.0))
POS_WEIGHT_TENSOR = torch.tensor(pos_weights, dtype=torch.float32).to(DEVICE)

# 7. Tokenizer & Dataset
TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

img_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class MultimodalE2Dataset(Dataset):
    def __init__(self, frame, cache_dir, transform, tokenizer, max_len=64):
        self.frame = frame.reset_index(drop=True)
        self.cache_dir = cache_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.sample_ids = self.frame["sample_id"].values.astype(int)
        self.targets = self.frame[target_cols].values.astype(np.float32)
        self.masks = self.frame[mask_cols].values.astype(np.float32)
        self.has_history = self.frame["has_history"].values.astype(bool)

        # When history is absent, supply explicit token placeholder
        self.texts = [
            f"Clinical history: {h}" if present else "Clinical history: None recorded."
            for h, present in zip(self.frame["clinical_history"], self.has_history)
        ]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        img_path = self.cache_dir / f"{sid:07d}.png"
        with Image.open(img_path) as im:
            img = self.transform(im.convert("L"))

        encoded = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        t = torch.nan_to_num(torch.from_numpy(self.targets[idx]), nan=0.0)
        m = torch.from_numpy(self.masks[idx])
        hist_flag = torch.tensor(1.0 if self.has_history[idx] else 0.0, dtype=torch.float32)

        return {
            "image": img,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "targets": t,
            "masks": m,
            "has_history": hist_flag
        }

train_loader = DataLoader(
    MultimodalE2Dataset(tr_df, LOCAL_CACHE, img_tf, tokenizer),
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    MultimodalE2Dataset(va_df, LOCAL_CACHE, img_tf, tokenizer),
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# 8. DP-Compatible Multimodal Architecture (GroupNorm + MiniLM)
def convert_bn_to_gn(module, num_groups=32):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            num_channels = child.num_features
            groups = num_groups
            while num_channels % groups != 0:
                groups //= 2
            setattr(module, name, nn.GroupNorm(num_groups=groups, num_channels=num_channels))
        else:
            convert_bn_to_gn(child, num_groups=num_groups)
    return module

class MultimodalE2Classifier(nn.Module):
    def __init__(self, num_classes=13, text_model_name=TEXT_MODEL_NAME):
        super().__init__()
        # Visual Encoder (ResNet-18 GroupNorm)
        res = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.visual_backbone = convert_bn_to_gn(res)
        self.visual_dim = self.visual_backbone.fc.in_features
        self.visual_backbone.fc = nn.Identity()

        # Text Encoder (MiniLM, frozen lower layers for T4 speed and stability)
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        for param in self.text_encoder.embeddings.parameters():
            param.requires_grad = False
        for layer in self.text_encoder.encoder.layer[:4]:
            for param in layer.parameters():
                param.requires_grad = False
        self.text_dim = self.text_encoder.config.hidden_size

        # Modality Projections
        self.proj_v = nn.Sequential(nn.Linear(self.visual_dim, 256), nn.GELU())
        self.proj_t = nn.Sequential(nn.Linear(self.text_dim, 256), nn.GELU())
        self.fusion_norm = nn.LayerNorm(512)

        # Classification Head
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, input_ids, attention_mask, has_history):
        feat_v = self.visual_backbone(images)
        out_t = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)

        # Attention-masked mean pooling over tokens
        token_emb = out_t[0]
        mask_exp = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
        feat_t = torch.sum(token_emb * mask_exp, dim=1) / torch.clamp(mask_exp.sum(dim=1), min=1e-9)

        proj_v = self.proj_v(feat_v)
        proj_t = self.proj_t(feat_t)

        # Zero-mask text projection when history is genuinely missing
        gate = has_history.unsqueeze(1)
        proj_t_gated = proj_t * gate

        fused = self.fusion_norm(torch.cat([proj_v, proj_t_gated], dim=1))
        return self.classifier(fused)

model = MultimodalE2Classifier().to(DEVICE)

# 9. Masked BCE Loss Function
def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=POS_WEIGHT_TENSOR, reduction="none"
    )
    return (element_loss * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

# 10. Evaluation Subroutine
@torch.no_grad()
def evaluate_metrics(net, loader):
    net.eval()
    total_loss, total_seen = 0.0, 0
    all_preds, all_targets, all_masks = [], [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
        tgts = batch["targets"].to(DEVICE, non_blocking=True)
        msks = batch["masks"].to(DEVICE, non_blocking=True)
        hist = batch["has_history"].to(DEVICE, non_blocking=True)

        logits = net(imgs, ids, attn, hist)
        loss = per_image_masked_bce(logits, tgts, msks).mean()

        total_loss += loss.item() * len(imgs)
        total_seen += len(imgs)

        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(tgts.cpu().numpy())
        all_masks.append(msks.cpu().numpy())

    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    masks = np.vstack(all_masks)

    per_class_auc = {}
    support_counts = {}

    for j, label in enumerate(LABELS):
        valid = masks[:, j] == 1.0
        y_true = targets[valid, j]
        y_score = preds[valid, j]

        pos_cnt = int((y_true == 1.0).sum())
        neg_cnt = int((y_true == 0.0).sum())
        support_counts[label] = (pos_cnt, neg_cnt)

        if pos_cnt > 0 and neg_cnt > 0:
            try:
                per_class_auc[label] = roc_auc_score(y_true, y_score)
            except ValueError:
                per_class_auc[label] = np.nan
        else:
            per_class_auc[label] = np.nan

    valid_scores = [v for v in per_class_auc.values() if not np.isnan(v)]
    macro_auc = float(np.mean(valid_scores)) if valid_scores else float("nan")
    return total_loss / max(total_seen, 1), macro_auc, per_class_auc, support_counts

# 11. Training Loop (3 Epochs)
NUM_EPOCHS = 3
print(f"\n=== LAUNCHING E2 PILOT TRAINING ({NUM_EPOCHS} EPOCHS) ===")
best_auc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, seen = 0.0, 0
    t0 = time.perf_counter()

    for step, batch in enumerate(train_loader, start=1):
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
        tgts = batch["targets"].to(DEVICE, non_blocking=True)
        msks = batch["masks"].to(DEVICE, non_blocking=True)
        hist = batch["has_history"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs, ids, attn, hist)
        loss = per_image_masked_bce(logits, tgts, msks).mean()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(imgs)
        seen += len(imgs)

    scheduler.step()
    dur = time.perf_counter() - t0
    train_loss = running_loss / max(seen, 1)
    val_loss, val_macro_auc, class_auc, support_info = evaluate_metrics(model, val_loader)

    eval_count = sum(~np.isnan(list(class_auc.values())))
    print(
        f"Epoch [{epoch}/{NUM_EPOCHS}] ({dur:.1f}s) | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro AUROC: {val_macro_auc:.4f} "
        f"({eval_count}/13 classes evaluable)"
    )

    is_best = val_macro_auc > best_auc
    if is_best:
        best_auc = val_macro_auc

    ckpt_payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auc": val_macro_auc,
        "per_class_auc": class_auc,
        "support_counts": support_info,
        "labels": LABELS,
    }
    torch.save(ckpt_payload, CHECKPOINT_DIR / f"multimodal_e2_epoch_{epoch}.pt")
    if is_best:
        torch.save(ckpt_payload, CHECKPOINT_DIR / "multimodal_e2_best.pt")

print(f"\nE2 pilot run complete. Artifacts saved to: {CHECKPOINT_DIR}")

Executing on device: cuda:0
Manifest located: /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
Active cached cohort: 38,912 images across 12,696 patients

=== CLINICAL HISTORY AUDIT ===
Images with extracted history: 28,172 / 38,912 (72.40%)
Images with missing history:   10,740

--- Sample Extracted Clinical Histories ---
Sample 1: History of uterine cancer.
Sample 2: 35 -year-old female with history of trauma.
Sample 3: 80-year-old man with CLL, central line placement.

=== PATIENT-DISJOINT SPLIT ===
Train split: 32,934 images (10,791 patients) | History coverage: 72.45%
Val split:   5,978 images (1,905 patients) | History coverage: 72.10%


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== LAUNCHING E2 PILOT TRAINING (3 EPOCHS) ===
Epoch [1/3] (172.4s) | Train Loss: 0.4327 | Val Loss: 0.4251 | Val Macro AUROC: 0.4635 (13/13 classes evaluable)
Epoch [2/3] (166.9s) | Train Loss: 0.4299 | Val Loss: 0.4260 | Val Macro AUROC: 0.4795 (13/13 classes evaluable)
Epoch [3/3] (165.6s) | Train Loss: 0.4259 | Val Loss: 0.4264 | Val Macro AUROC: 0.5068 (13/13 classes evaluable)

E2 pilot run complete. Artifacts saved to: /content/drive/MyDrive/DP-MMFL/checkpoints/e2_multimodal_pilot


In [ ]:
# ================================================================
# DP-MMFL E2: CONTROLLED IMAGE-ONLY vs IMAGE + CLINICAL HISTORY
# Google Colab / Tesla T4
#
# Key Specifications:
# - Uses ONLY clinical history / indication, strictly excluding Findings/Impression.
# - Patient-disjoint train/validation split (GroupShuffleSplit on deid_patient_id).
# - Identical splits, seed, weights, and loss for Image-Only vs Multimodal models.
# - Masked BCE loss with clamped pos_weight applied only to observed labels.
# - Saves checkpoints, prediction arrays, metrics JSON, and protocol to Drive.
# ================================================================
import copy
import gc
import json
import os
import random
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
SEED = 42
EPOCHS = 5
BATCH_SIZE = 32
NUM_WORKERS = 2
LR_IMAGE = 1e-5
LR_HEAD = 3e-4
WEIGHT_DECAY = 1e-4
MAX_HISTORY_TOKENS = 96
VAL_FRACTION = 0.15
PATIENCE = 3
IMAGE_SIZE = 224

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {DEVICE}")

DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
OUTPUT_DIR = DRIVE_ROOT / "results/e2_controlled_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = Path("/content/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet")

assert MANIFEST.exists(), f"Manifest missing at: {MANIFEST}"
assert LOCAL_CACHE.exists(), f"Image cache missing at: {LOCAL_CACHE}"

LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]
target_cols = [f"target_{c}" for c in LABELS]
mask_cols = [f"mask_{c}" for c in LABELS]

# -------------------- 1. COHORT & TEXT EXTRACTION --------------------
df = pd.read_parquet(MANIFEST)
train_df = df[df["experiment_split"].astype(str).str.lower() == "train"].copy()
train_df["sample_id"] = train_df["sample_id"].astype(int)

cached_sids = {int(p.stem) for p in LOCAL_CACHE.glob("*.png") if p.stem.isdigit()}
active_df = train_df[train_df["sample_id"].isin(cached_sids)].copy().reset_index(drop=True)
print(f"Cached cohort: {len(active_df):,} images across {active_df['deid_patient_id'].nunique():,} unique patients")

def extract_clinical_history(report_text):
    if not isinstance(report_text, str) or not report_text.strip():
        return "", False
    # Strictly capture between history headers and diagnostic/technique boundaries
    pattern = (
        r"(?:CLINICAL HISTORY|INDICATION|REASON FOR EXAM|HISTORY):?\s*"
        r"(.*?)(?=(?:FINDINGS|IMPRESSION|TECHNIQUE|COMPARISON|SUMMARY|END OF IMPRESSION|$))"
    )
    match = re.search(pattern, report_text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        extracted = " ".join(match.group(1).split()).strip()
        if len(extracted) > 2:
            return extracted[:256], True
    return "", False

extracted_info = [extract_clinical_history(r) for r in active_df["report"]]
active_df["clinical_history"] = [t[0] for t in extracted_info]
active_df["has_history"] = [t[1] for t in extracted_info]

print(f"Clinical history present: {active_df['has_history'].sum():,} / {len(active_df):,} ({active_df['has_history'].mean():.2%})")

# Print verification samples to confirm no Findings/Impressions leaked
print("\n--- History Extraction Verification (First 3 Positive Matches) ---")
samples = active_df[active_df["has_history"]].head(3)
for idx, (_, row) in enumerate(samples.iterrows(), 1):
    print(f"Sample {idx}: {row['clinical_history']}")

# -------------------- 2. PATIENT-DISJOINT SPLIT --------------------
groups = active_df["deid_patient_id"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
tr_idx, va_idx = next(splitter.split(active_df, groups=groups))

tr_df = active_df.iloc[tr_idx].copy().reset_index(drop=True)
va_df = active_df.iloc[va_idx].copy().reset_index(drop=True)

overlap = len(set(tr_df["deid_patient_id"]) & set(va_df["deid_patient_id"]))
assert overlap == 0, f"Patient data leakage detected: {overlap} shared patients!"
print(f"\nCohort split (85/15): Train={len(tr_df):,} images ({tr_df['deid_patient_id'].nunique():,} pats) | Val={len(va_df):,} images ({va_df['deid_patient_id'].nunique():,} pats)")

# Compute stable pos_weight vector based solely on training split
pos_weights = []
for label in LABELS:
    t_col = f"target_{label}"
    m_col = f"mask_{label}"
    obs = (tr_df[m_col].fillna(0).astype(float) == 1.0)
    vals = tr_df.loc[obs, t_col].dropna().astype(float)
    pos = float((vals == 1.0).sum())
    neg = float((vals == 0.0).sum())
    w = np.clip(neg / max(pos, 1.0), 0.1, 10.0)
    pos_weights.append(w)
POS_WEIGHT_TENSOR = torch.tensor(pos_weights, dtype=torch.float32).to(DEVICE)

# -------------------- 3. DATASET & TRANSFORMS --------------------
TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

img_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ControlledE2Dataset(Dataset):
    def __init__(self, frame, cache_dir, transform, tokenizer, max_len=MAX_HISTORY_TOKENS):
        self.frame = frame.reset_index(drop=True)
        self.cache_dir = cache_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.sample_ids = self.frame["sample_id"].values.astype(int)
        self.targets = self.frame[target_cols].values.astype(np.float32)
        self.masks = self.frame[mask_cols].values.astype(np.float32)
        self.has_history = self.frame["has_history"].values.astype(bool)
        self.texts = [
            f"Clinical indication: {h}" if present else "Clinical indication: None recorded."
            for h, present in zip(self.frame["clinical_history"], self.has_history)
        ]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        img_path = self.cache_dir / f"{sid:07d}.png"
        with Image.open(img_path) as im:
            img = self.transform(im.convert("L"))

        encoded = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        t = torch.nan_to_num(torch.from_numpy(self.targets[idx]), nan=0.0)
        m = torch.from_numpy(self.masks[idx])
        h_flag = torch.tensor(1.0 if self.has_history[idx] else 0.0, dtype=torch.float32)

        return {
            "sample_id": sid,
            "image": img,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "targets": t,
            "masks": m,
            "has_history": h_flag
        }

train_loader = DataLoader(ControlledE2Dataset(tr_df, LOCAL_CACHE, img_tf, tokenizer), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(ControlledE2Dataset(va_df, LOCAL_CACHE, img_tf, tokenizer), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# -------------------- 4. MODELS (GROUPNORM RESNET-18 & MULTIMODAL FUSION) --------------------
def convert_bn_to_gn(module, num_groups=32):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            num_channels = child.num_features
            groups = num_groups
            while num_channels % groups != 0:
                groups //= 2
            setattr(module, name, nn.GroupNorm(num_groups=groups, num_channels=num_channels))
        else:
            convert_bn_to_gn(child, num_groups=num_groups)
    return module

class ImageOnlyModel(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = convert_bn_to_gn(base)
        in_feats = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_feats, num_classes)
        )

    def forward(self, images, **kwargs):
        feats = self.backbone(images)
        return self.classifier(feats)

class MultimodalModel(nn.Module):
    def __init__(self, num_classes=13, text_model_name=TEXT_MODEL_NAME):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.visual_backbone = convert_bn_to_gn(base)
        self.visual_dim = self.visual_backbone.fc.in_features
        self.visual_backbone.fc = nn.Identity()

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        for param in self.text_encoder.embeddings.parameters():
            param.requires_grad = False
        for layer in self.text_encoder.encoder.layer[:4]:
            for param in layer.parameters():
                param.requires_grad = False
        self.text_dim = self.text_encoder.config.hidden_size

        self.proj_v = nn.Sequential(nn.Linear(self.visual_dim, 256), nn.GELU())
        self.proj_t = nn.Sequential(nn.Linear(self.text_dim, 256), nn.GELU())
        self.fusion_norm = nn.LayerNorm(512)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, input_ids, attention_mask, has_history):
        feat_v = self.visual_backbone(images)
        out_t = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_emb = out_t[0]
        mask_exp = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
        feat_t = torch.sum(token_emb * mask_exp, dim=1) / torch.clamp(mask_exp.sum(dim=1), min=1e-9)

        proj_v = self.proj_v(feat_v)
        proj_t = self.proj_t(feat_t) * has_history.unsqueeze(1)
        fused = self.fusion_norm(torch.cat([proj_v, proj_t], dim=1))
        return self.classifier(fused)

def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=POS_WEIGHT_TENSOR, reduction="none"
    )
    return (element_loss * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

# -------------------- 5. EVALUATION SUITE --------------------
@torch.no_grad()
def evaluate_model(net, loader, is_multimodal=False):
    net.eval()
    total_loss, total_seen = 0.0, 0
    all_preds, all_targets, all_masks, all_sids = [], [], [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        tgts = batch["targets"].to(DEVICE, non_blocking=True)
        msks = batch["masks"].to(DEVICE, non_blocking=True)

        if is_multimodal:
            ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
            hist = batch["has_history"].to(DEVICE, non_blocking=True)
            logits = net(imgs, input_ids=ids, attention_mask=attn, has_history=hist)
        else:
            logits = net(imgs)

        loss = per_image_masked_bce(logits, tgts, msks).mean()
        total_loss += loss.item() * len(imgs)
        total_seen += len(imgs)

        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(tgts.cpu().numpy())
        all_masks.append(msks.cpu().numpy())
        all_sids.extend(batch["sample_id"].tolist())

    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    masks = np.vstack(all_masks)

    per_class_auc = {}
    support_counts = {}
    for j, label in enumerate(LABELS):
        valid = masks[:, j] == 1.0
        y_true = targets[valid, j]
        y_score = preds[valid, j]
        pos_cnt = int((y_true == 1.0).sum())
        neg_cnt = int((y_true == 0.0).sum())
        support_counts[label] = {"pos": pos_cnt, "neg": neg_cnt, "total": pos_cnt + neg_cnt}
        if pos_cnt > 0 and neg_cnt > 0:
            try:
                per_class_auc[label] = float(roc_auc_score(y_true, y_score))
            except ValueError:
                per_class_auc[label] = float("nan")
        else:
            per_class_auc[label] = float("nan")

    valid_scores = [v for v in per_class_auc.values() if not np.isnan(v)]
    macro_auc = float(np.mean(valid_scores)) if valid_scores else float("nan")
    val_loss = total_loss / max(total_seen, 1)

    return val_loss, macro_auc, per_class_auc, support_counts, (np.array(all_sids), preds)

# -------------------- 6. TRAINING HARNESS --------------------
def train_experiment(model_type="image_only"):
    is_multimodal = (model_type == "multimodal")
    tag = "Multimodal (Image + History)" if is_multimodal else "Image-Only Control"
    print(f"\n=======================================================")
    print(f"STARTING EXPERIMENT: {tag}")
    print(f"=======================================================")

    net = MultimodalModel().to(DEVICE) if is_multimodal else ImageOnlyModel().to(DEVICE)

    # Differential learning rate: lower for backbone, higher for classifier/projections
    if is_multimodal:
        backbone_params = list(net.visual_backbone.parameters()) + list(net.text_encoder.parameters())
        head_params = list(net.proj_v.parameters()) + list(net.proj_t.parameters()) + list(net.classifier.parameters()) + list(net.fusion_norm.parameters())
    else:
        backbone_params = list(net.backbone.parameters())
        head_params = list(net.classifier.parameters())

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": LR_IMAGE},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_auc = -1.0
    best_epoch = 0
    patience_counter = 0
    best_results = {}
    history = []

    for epoch in range(1, EPOCHS + 1):
        net.train()
        running_loss, total_seen = 0.0, 0
        t0 = time.perf_counter()

        for batch in train_loader:
            imgs = batch["image"].to(DEVICE, non_blocking=True)
            tgts = batch["targets"].to(DEVICE, non_blocking=True)
            msks = batch["masks"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            if is_multimodal:
                ids = batch["input_ids"].to(DEVICE, non_blocking=True)
                attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
                hist = batch["has_history"].to(DEVICE, non_blocking=True)
                logits = net(imgs, input_ids=ids, attention_mask=attn, has_history=hist)
            else:
                logits = net(imgs)

            loss = per_image_masked_bce(logits, tgts, msks).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(imgs)
            total_seen += len(imgs)

        scheduler.step()
        epoch_sec = time.perf_counter() - t0
        train_loss = running_loss / max(total_seen, 1)

        val_loss, val_auc, class_auc, support_info, (sids, preds) = evaluate_model(net, val_loader, is_multimodal=is_multimodal)
        eval_count = sum(~np.isnan(list(class_auc.values())))

        print(f"Epoch [{epoch}/{EPOCHS}] ({epoch_sec:.1f}s) | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro AUROC: {val_auc:.4f} ({eval_count}/{len(LABELS)} classes)")

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_auc": val_auc,
            "per_class_auc": class_auc
        }
        history.append(epoch_record)

        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            patience_counter = 0
            best_results = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_macro_auc": val_auc,
                "per_class_auc": class_auc,
                "support_counts": support_info,
                "labels": LABELS,
                "model_type": model_type
            }
            # Save best checkpoint & prediction arrays
            torch.save(net.state_dict(), OUTPUT_DIR / f"{model_type}_best_model.pt")
            np.savez_compressed(OUTPUT_DIR / f"{model_type}_val_preds.npz", sample_ids=sids, predictions=preds)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping triggered at epoch {epoch} (Best Epoch: {best_epoch} with AUROC: {best_auc:.4f})")
                break

    del net, optimizer, scheduler
    torch.cuda.empty_cache()
    gc.collect()

    return best_results, history

# -------------------- 7. EXECUTE BOTH RUNS --------------------
image_best, image_hist = train_experiment(model_type="image_only")
mm_best, mm_hist = train_experiment(model_type="multimodal")

# -------------------- 8. SUMMARY COMPARISON & PERSISTENCE --------------------
delta_auc = mm_best["val_macro_auc"] - image_best["val_macro_auc"]

comparison_rows = []
for label in LABELS:
    img_score = image_best["per_class_auc"].get(label, np.nan)
    mm_score = mm_best["per_class_auc"].get(label, np.nan)
    sup = image_best["support_counts"][label]
    comparison_rows.append({
        "Pathology": label,
        "Val Pos": sup["pos"],
        "Val Neg": sup["neg"],
        "Image-Only AUROC": f"{img_score:.4f}" if not np.isnan(img_score) else "NaN",
        "Multimodal AUROC": f"{mm_score:.4f}" if not np.isnan(mm_score) else "NaN",
        "Delta": f"{(mm_score - img_score):+.4f}" if (not np.isnan(img_score) and not np.isnan(mm_score)) else "N/A"
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\n" + "="*80)
print("FINAL CONTROLLED E2 EXPERIMENTAL COMPARISON")
print("="*80)
print(f"Image-Only Baseline (Best Epoch {image_best['epoch']}):  Val Macro AUROC = {image_best['val_macro_auc']:.4f}")
print(f"Multimodal Baseline (Best Epoch {mm_best['epoch']}):  Val Macro AUROC = {mm_best['val_macro_auc']:.4f}")
print(f"Empirical Gain (Delta):                         {delta_auc:+.4f}")
print("-" * 80)
print(comparison_df.to_string(index=False))
print("="*80)

# Save comprehensive audit JSON and protocol documentation
experiment_summary = {
    "protocol": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "val_fraction": VAL_FRACTION,
        "max_history_tokens": MAX_HISTORY_TOKENS,
        "labels": LABELS,
        "train_samples": len(tr_df),
        "val_samples": len(va_df),
        "train_patients": int(tr_df["deid_patient_id"].nunique()),
        "val_patients": int(va_df["deid_patient_id"].nunique()),
    },
    "image_only_results": image_best,
    "multimodal_results": mm_best,
    "comparison_table": comparison_rows,
    "delta_macro_auc": delta_auc
}

with open(OUTPUT_DIR / "e2_controlled_results.json", "w") as f:
    json.dump(experiment_summary, f, indent=2)

print(f"\nAll artifacts (checkpoints, .npz predictions, results JSON) persisted to:\n{OUTPUT_DIR}")

Executing on device: cuda:0
Cached cohort: 38,912 images across 12,696 unique patients
Clinical history present: 28,172 / 38,912 (72.40%)

--- History Extraction Verification (First 3 Positive Matches) ---
Sample 1: History of uterine cancer.
Sample 2: 35 -year-old female with history of trauma.
Sample 3: 80-year-old man with CLL, central line placement.

Cohort split (85/15): Train=32,934 images (10,791 pats) | Val=5,978 images (1,905 pats)

STARTING EXPERIMENT: Image-Only Control
Epoch [1/5] (155.5s) | Train Loss: 0.4400 | Val Loss: 0.4270 | Val Macro AUROC: 0.4923 (13/13 classes)
Epoch [2/5] (134.4s) | Train Loss: 0.4319 | Val Loss: 0.4268 | Val Macro AUROC: 0.4845 (13/13 classes)
Epoch [3/5] (131.5s) | Train Loss: 0.4293 | Val Loss: 0.4284 | Val Macro AUROC: 0.4873 (13/13 classes)
Epoch [4/5] (131.7s) | Train Loss: 0.4274 | Val Loss: 0.4239 | Val Macro AUROC: 0.4991 (13/13 classes)
Epoch [5/5] (132.4s) | Train Loss: 0.4260 | Val Loss: 0.4239 | Val Macro AUROC: 0.4995 (13/13 classes

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Epoch [1/5] (184.9s) | Train Loss: 0.4319 | Val Loss: 0.4251 | Val Macro AUROC: 0.4874 (13/13 classes)
Epoch [2/5] (183.7s) | Train Loss: 0.4259 | Val Loss: 0.4254 | Val Macro AUROC: 0.4863 (13/13 classes)
Epoch [3/5] (181.8s) | Train Loss: 0.4230 | Val Loss: 0.4270 | Val Macro AUROC: 0.4825 (13/13 classes)
Epoch [4/5] (171.2s) | Train Loss: 0.4198 | Val Loss: 0.4305 | Val Macro AUROC: 0.4788 (13/13 classes)
Early stopping triggered at epoch 4 (Best Epoch: 1 with AUROC: 0.4874)

FINAL CONTROLLED E2 EXPERIMENTAL COMPARISON
Image-Only Baseline (Best Epoch 5):  Val Macro AUROC = 0.4995
Multimodal Baseline (Best Epoch 1):  Val Macro AUROC = 0.4874
Empirical Gain (Delta):                         -0.0121
--------------------------------------------------------------------------------
                 Pathology  Val Pos  Val Neg Image-Only AUROC Multimodal AUROC   Delta
Enlarged Cardiomediastinum      220      856           0.5020           0.4957 -0.0063
              Cardiomegaly      943  

In [ ]:
# ================================================================
# DP-MMFL — CONSOLIDATED PIPELINE AUDIT (READ-ONLY)
# Run this entire cell in Google Colab and share the complete output.
# ================================================================
import json
import os
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

print("=" * 90)
print("DP-MMFL CONSOLIDATED PIPELINE AUDIT")
print("=" * 90)

# ------------------------- 1. Environment -------------------------
print("\n[1] ENVIRONMENT")
try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "VRAM Total:",
            f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GiB",
        )
except Exception as e:
    print("PyTorch check failed:", repr(e))

# ------------------------- 2. Locate Manifest & Cache -------------------------
print("\n[2] PATH VERIFICATION")
DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
DRIVE_CACHE = DRIVE_ROOT / "data/image_cache_256/train"

candidates = [
    DRIVE_ROOT
    / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet",
    Path(
        "/content/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
    ),
]
manifest_path = next((p for p in candidates if p.is_file()), None)

print(f"Manifest path:   {manifest_path}")
print(
    f"Local cache:     {LOCAL_CACHE} (Exists: {LOCAL_CACHE.exists() and LOCAL_CACHE.is_dir()})"
)
print(
    f"Drive cache:     {DRIVE_CACHE} (Exists: {DRIVE_CACHE.exists() and DRIVE_CACHE.is_dir()})"
)

if manifest_path is None or not manifest_path.is_file():
    print(
        "\nERROR: Manifest could not be located. Ensure Google Drive is mounted."
    )
    raise SystemExit

df = pd.read_parquet(manifest_path)
print(f"Manifest shape:  {df.shape}")

# ------------------------- 3. Splits & Official Partitions -------------------------
print("\n[3] SPLIT INTEGRITY & PATIENT DISJOINTNESS")
split_col = "experiment_split" if "experiment_split" in df.columns else "split"
print(f"Split column: '{split_col}'")
print(df[split_col].value_counts().to_string())

splits = df[split_col].dropna().unique()
patient_sets = {}
for s in splits:
    sub = df[df[split_col] == s]
    pats = set(sub["deid_patient_id"].unique())
    patient_sets[s] = pats
    print(
        f"Split '{s}': {len(sub):,} images across {len(pats):,} distinct patients (avg {len(sub)/len(pats):.2f} img/pat)"
    )

overlap_found = False
for i, s1 in enumerate(splits):
    for s2 in splits[i + 1 :]:
        inter = patient_sets[s1] & patient_sets[s2]
        if len(inter) > 0:
            print(f"WARNING: Leakage! Split '{s1}' & '{s2}' share {len(inter)} patients!")
            overlap_found = True
if not overlap_found:
    print("Patient-disjointness verified across all official manifest splits: ZERO patient overlap.")

# ------------------------- 4. Cache Alignment & Decoding Test -------------------------
print("\n[4] CACHED IMAGES & DECODING INTEGRITY")
local_pngs = (
    list(LOCAL_CACHE.glob("*.png")) if LOCAL_CACHE.exists() else []
)
drive_pngs = (
    list(DRIVE_CACHE.glob("*.png")) if DRIVE_CACHE.exists() else []
)

print(f"Local SSD cached PNGs:   {len(local_pngs):,}")
print(f"Drive backup cached PNGs: {len(drive_pngs):,}")

local_sids = {int(p.stem) for p in local_pngs if p.stem.isdigit()}
train_mask = df[split_col].astype(str).str.lower() == "train"
train_df = df[train_mask].copy()
train_df["sample_id"] = train_df["sample_id"].astype(int)

train_sids = set(train_df["sample_id"])
matched_ids = local_sids & train_sids
out_of_split_ids = local_sids - train_sids

print(f"Cached PNGs matching train split: {len(matched_ids):,} / {len(train_df):,} ({len(matched_ids)/len(train_df):.2%})")
print(f"Out-of-split / Unexpected IDs:    {len(out_of_split_ids):,}")

active_df = train_df[train_df["sample_id"].isin(matched_ids)].copy().reset_index(drop=True)
print(f"Active cached cohort:             {len(active_df):,} images across {active_df['deid_patient_id'].nunique():,} unique patients")

# Integrity test: sample 100 images and decode via PIL
decode_errors = 0
sample_check = (
    np.random.choice(list(matched_ids), size=min(100, len(matched_ids)), replace=False)
    if matched_ids
    else []
)
for sid in sample_check:
    img_p = LOCAL_CACHE / f"{sid:07d}.png"
    if not img_p.exists():
        img_p = LOCAL_CACHE / f"{sid}.png"
    try:
        with Image.open(img_p) as im:
            im.verify()
    except Exception:
        decode_errors += 1
print(f"Image decode test on {len(sample_check)} samples: {decode_errors} errors")

# ------------------------- 5. Label & Mask Consistency Audit -------------------------
print("\n[5] OBSERVED LABELS & CLASS IMBALANCE (ACTIVE CACHED COHORT)")
LABELS = [
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

label_audit = []
for label in LABELS:
    t_col = f"target_{label}"
    m_col = f"mask_{label}"

    if t_col not in active_df.columns or m_col not in active_df.columns:
        label_audit.append({
            "Pathology": label,
            "Target In Manifest": t_col in active_df.columns,
            "Mask In Manifest": m_col in active_df.columns,
            "Observed": 0,
            "Pos": 0,
            "Neg": 0,
            "Pos %": np.nan,
            "Min Class Count": 0,
        })
        continue

    obs = active_df[m_col].fillna(0).astype(float) == 1.0
    vals = active_df.loc[obs, t_col].dropna().astype(float)
    pos = int((vals == 1.0).sum())
    neg = int((vals == 0.0).sum())
    total_obs = pos + neg

    label_audit.append({
        "Pathology": label,
        "Target In Manifest": True,
        "Mask In Manifest": True,
        "Observed": total_obs,
        "Pos": pos,
        "Neg": neg,
        "Pos %": f"{100.0 * pos / max(total_obs, 1):.2f}%",
        "Min Class Count": min(pos, neg),
    })

audit_df = pd.DataFrame(label_audit)
print(audit_df.to_string(index=False))

# ------------------------- 6. Saved E1/E2 Artifacts -------------------------
print("\n[6] CHECKPOINT & EXPERIMENT ARTIFACT INVENTORY")
artifact_dirs = [
    DRIVE_ROOT / "checkpoints/centralized_baseline_38k",
    DRIVE_ROOT / "checkpoints/e2_multimodal_pilot",
    DRIVE_ROOT / "results/e2_controlled_experiment",
]

for d in artifact_dirs:
    print(f"\nScanning: {d}")
    if d.exists() and d.is_dir():
        files = list(d.glob("*"))
        if files:
            for f in sorted(files):
                print(f"  - {f.name:<40} ({f.stat().st_size:,} bytes)")
        else:
            print("  (Empty directory)")
    else:
        print("  (Directory does not exist)")

e2_json = (
    DRIVE_ROOT / "results/e2_controlled_experiment/e2_controlled_results.json"
)
if e2_json.exists():
    print("\nReading E2 Controlled Results JSON:")
    with open(e2_json) as jf:
        data = json.load(jf)
        img_auc = data.get("image_only_results", {}).get("val_macro_auc", "N/A")
        mm_auc = data.get("multimodal_results", {}).get("val_macro_auc", "N/A")
        delta = data.get("delta_macro_auc", "N/A")
        print(f"  Stored Image-Only Val Macro AUROC:  {img_auc}")
        print(f"  Stored Multimodal Val Macro AUROC:  {mm_auc}")
        print(f"  Stored Delta (Multimodal - Image):  {delta}")

print("\n" + "=" * 90)
print("AUDIT EXECUTION COMPLETE")
print("=" * 90)

DP-MMFL CONSOLIDATED PIPELINE AUDIT

[1] ENVIRONMENT
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM Total: 14.56 GiB

[2] PATH VERIFICATION
Manifest path:   /content/drive/MyDrive/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet
Local cache:     /content/dp_mmfl_image_cache_256/train (Exists: True)
Drive cache:     /content/drive/MyDrive/DP-MMFL/data/image_cache_256/train (Exists: True)
Manifest shape:  (111562, 32)

[3] SPLIT INTEGRITY & PATIENT DISJOINTNESS
Split column: 'experiment_split'
experiment_split
train    73640
test     19349
val      18573
Split 'train': 73,640 images across 25,000 distinct patients (avg 2.95 img/pat)
Split 'val': 18,573 images across 6,449 distinct patients (avg 2.88 img/pat)
Split 'test': 19,349 images across 6,452 distinct patients (avg 3.00 img/pat)
Patient-disjointness verified across all official manifest splits: ZERO patient overlap.

[4] CACHED IMAGES & DECODING INTEGRITY
Local SSD cached PNGs:   38,912
Drive 

FEDERATED

In [ ]:
# ==============================================================================
# DP-MMFL E3: FEDERATED MULTIMODAL BASELINE (FedAvg, K=5 Dirichlet Clients)
# ==============================================================================
import copy
import gc
import json
import os
import random
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

# -------------------- 1. EXPERIMENT CONFIG --------------------
SEED = 42
GLOBAL_ROUNDS = 5
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
NUM_CLIENTS = 5
DIRICHLET_ALPHA = 0.5
CLIENT_LR = 1e-4
CLIENT_WEIGHT_DECAY = 1e-4
MAX_HISTORY_TOKENS = 96
VAL_FRACTION = 0.15
IMAGE_SIZE = 224

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"=== DP-MMFL E3: FEDAVG MULTIMODAL SIMULATION ===")
print(f"Device: {DEVICE} | Clients: {NUM_CLIENTS} | Alpha: {DIRICHLET_ALPHA}")

DRIVE_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
LOCAL_CACHE = Path("/content/dp_mmfl_image_cache_256/train")
OUTPUT_DIR = DRIVE_ROOT / "results/e3_fedavg_multimodal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = DRIVE_ROOT / "results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet"
if not MANIFEST.exists():
    MANIFEST = Path("/content/DP-MMFL/results/final_protocol/chexpert_plus_25k_frozen_manifest.parquet")

assert MANIFEST.exists(), f"Manifest missing: {MANIFEST}"
assert LOCAL_CACHE.exists(), f"Local SSD cache missing: {LOCAL_CACHE}"

LABELS = [
    "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia",
    "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]
target_cols = [f"target_{c}" for c in LABELS]
mask_cols = [f"mask_{c}" for c in LABELS]

# -------------------- 2. COHORT & HISTORY EXTRACTION --------------------
df = pd.read_parquet(MANIFEST)
train_df = df[df["experiment_split"].astype(str).str.lower() == "train"].copy()
train_df["sample_id"] = train_df["sample_id"].astype(int)

cached_sids = {int(p.stem) for p in LOCAL_CACHE.glob("*.png") if p.stem.isdigit()}
active_df = train_df[train_df["sample_id"].isin(cached_sids)].copy().reset_index(drop=True)
print(f"Active cached cohort: {len(active_df):,} images across {active_df['deid_patient_id'].nunique():,} unique patients")

def extract_clinical_history(report_text):
    if not isinstance(report_text, str) or not report_text.strip():
        return "", False
    pattern = (
        r"(?:CLINICAL HISTORY|INDICATION|REASON FOR EXAM|HISTORY):?\s*"
        r"(.*?)(?=(?:FINDINGS|IMPRESSION|TECHNIQUE|COMPARISON|SUMMARY|END OF IMPRESSION|$))"
    )
    match = re.search(pattern, report_text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        extracted = " ".join(match.group(1).split()).strip()
        if len(extracted) > 2:
            return extracted[:256], True
    return "", False

extracted_info = [extract_clinical_history(r) for r in active_df["report"]]
active_df["clinical_history"] = [t[0] for t in extracted_info]
active_df["has_history"] = [t[1] for t in extracted_info]

# -------------------- 3. PATIENT-DISJOINT INTERNAL VALIDATION SPLIT --------------------
groups = active_df["deid_patient_id"].astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
tr_idx, va_idx = next(splitter.split(active_df, groups=groups))

tr_df = active_df.iloc[tr_idx].copy().reset_index(drop=True)
va_df = active_df.iloc[va_idx].copy().reset_index(drop=True)

overlap = len(set(tr_df["deid_patient_id"]) & set(va_df["deid_patient_id"]))
assert overlap == 0, f"Patient leakage detected: {overlap} shared patients!"
print(f"Cohort split: Train={len(tr_df):,} images ({tr_df['deid_patient_id'].nunique():,} pats) | Val={len(va_df):,} images ({va_df['deid_patient_id'].nunique():,} pats)")

# -------------------- 4. GENUINE DIRICHLET NON-IID PARTITION --------------------
print(f"\nConstructing Dirichlet non-IID partition (alpha={DIRICHLET_ALPHA}) across {NUM_CLIENTS} institutions...")
patient_rows = []
for patient_id, group in tr_df.groupby("deid_patient_id", sort=False):
    row = {"deid_patient_id": patient_id}
    for t_col, m_col in zip(target_cols, mask_cols):
        y = pd.to_numeric(group[t_col], errors="coerce")
        m = pd.to_numeric(group[m_col], errors="coerce").fillna(0)
        observed = y[m.eq(1)].dropna()
        row[t_col] = int((observed.mean() >= 0.5)) if len(observed) else -1
    patient_rows.append(row)

patient_labels = pd.DataFrame(patient_rows)
patient_ids = patient_labels["deid_patient_id"].to_numpy()
label_matrix = patient_labels[target_cols].to_numpy(dtype=int)

rng = np.random.default_rng(SEED)
MAX_ATTEMPTS = 100
client_patient_ids = None

for attempt in range(MAX_ATTEMPTS):
    assignments = [[] for _ in range(NUM_CLIENTS)]
    # Use positive cases per pathology as anchor distributions
    for col_idx in range(len(target_cols)):
        pos_pats = np.where(label_matrix[:, col_idx] == 1)[0]
        if len(pos_pats) == 0:
            continue
        proportions = rng.dirichlet(np.repeat(DIRICHLET_ALPHA, NUM_CLIENTS))
        splits = np.split(rng.permutation(pos_pats), (np.cumsum(proportions)[:-1] * len(pos_pats)).astype(int))
        for c_idx, sub in enumerate(splits):
            assignments[c_idx].extend(patient_ids[sub])

    # Assign negative/neutral patients evenly
    neg_pats = np.where((label_matrix == 1).sum(axis=1) == 0)[0]
    neg_splits = np.array_split(rng.permutation(patient_ids[neg_pats]), NUM_CLIENTS)
    for c_idx, sub in enumerate(neg_splits):
        assignments[c_idx].extend(sub)

    # Convert to deduplicated patient sets
    client_sets = [list(set(a)) for a in assignments]
    if all(len(s) > 0 for s in client_sets):
        client_patient_ids = client_sets
        break

assert client_patient_ids is not None, "Failed to establish non-empty Dirichlet partition."

client_dfs = []
print("=== FEDERATED INSTITUTIONAL PROFILE ===")
for k in range(NUM_CLIENTS):
    c_pats = set(client_patient_ids[k])
    c_df = tr_df[tr_df["deid_patient_id"].isin(c_pats)].copy().reset_index(drop=True)
    client_dfs.append(c_df)
    print(f"Client {k+1}: {len(c_df):,} images ({len(c_pats):,} patients) | History present: {c_df['has_history'].mean():.2%}")

# -------------------- 5. VECTORIZED PRE-TOKENIZATION --------------------
TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

print("\nPre-tokenizing clinical histories to eliminate CPU bottleneck during training...")
def pretokenize_corpus(df_split):
    formatted_texts = [
        f"Clinical indication: {h}" if present else "Clinical indication: None recorded."
        for h, present in zip(df_split["clinical_history"], df_split["has_history"])
    ]
    enc = tokenizer(
        formatted_texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_HISTORY_TOKENS,
        return_tensors="pt"
    )
    return enc["input_ids"], enc["attention_mask"]

val_ids, val_attn = pretokenize_corpus(va_df)

client_tokenized = []
for c_df in client_dfs:
    ids, attn = pretokenize_corpus(c_df)
    client_tokenized.append((ids, attn))

# -------------------- 6. DATASET & DATALOADERS --------------------
img_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FastMultimodalDataset(Dataset):
    def __init__(self, frame, input_ids, attention_mask, cache_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.cache_dir = cache_dir
        self.transform = transform

        self.sample_ids = self.frame["sample_id"].values.astype(int)
        self.targets = self.frame[target_cols].values.astype(np.float32)
        self.masks = self.frame[mask_cols].values.astype(np.float32)
        self.has_history = torch.tensor(self.frame["has_history"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        sid = self.sample_ids[idx]
        img_path = self.cache_dir / f"{sid:07d}.png"
        with Image.open(img_path) as im:
            img = self.transform(im.convert("L"))

        t = torch.nan_to_num(torch.from_numpy(self.targets[idx]), nan=0.0)
        m = torch.from_numpy(self.masks[idx])

        return {
            "image": img,
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "targets": t,
            "masks": m,
            "has_history": self.has_history[idx]
        }

client_loaders = [
    DataLoader(
        FastMultimodalDataset(c_df, c_ids, c_attn, LOCAL_CACHE, img_tf),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    for c_df, (c_ids, c_attn) in zip(client_dfs, client_tokenized)
]

val_loader = DataLoader(
    FastMultimodalDataset(va_df, val_ids, val_attn, LOCAL_CACHE, img_tf),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# -------------------- 7. OPACUS-READY GROUPNORM BACKBONE & FUSION --------------------
def convert_bn_to_gn(module, num_groups=32):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            num_channels = child.num_features
            groups = num_groups
            while num_channels % groups != 0:
                groups //= 2
            setattr(module, name, nn.GroupNorm(num_groups=groups, num_channels=num_channels))
        else:
            convert_bn_to_gn(child, num_groups=num_groups)
    return module

class FedMultimodalClassifier(nn.Module):
    def __init__(self, num_classes=13, text_model_name=TEXT_MODEL_NAME):
        super().__init__()
        # Visual branch (GroupNorm ResNet-18)
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.visual_backbone = convert_bn_to_gn(base)
        self.visual_dim = self.visual_backbone.fc.in_features
        self.visual_backbone.fc = nn.Identity()

        # Text branch
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        for param in self.text_encoder.parameters():
            param.requires_grad = False
        self.text_dim = self.text_encoder.config.hidden_size

        # Projection & Fusion
        self.proj_v = nn.Sequential(nn.Linear(self.visual_dim, 256), nn.GELU())
        self.proj_t = nn.Sequential(nn.Linear(self.text_dim, 256), nn.GELU())
        self.fusion_norm = nn.LayerNorm(512)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, input_ids, attention_mask, has_history):
        feat_v = self.visual_backbone(images)
        with torch.no_grad():
            out_t = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
            token_emb = out_t[0]
            mask_exp = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
            feat_t = torch.sum(token_emb * mask_exp, dim=1) / torch.clamp(mask_exp.sum(dim=1), min=1e-9)

        proj_v = self.proj_v(feat_v)
        proj_t = self.proj_t(feat_t) * has_history.unsqueeze(1)
        fused = self.fusion_norm(torch.cat([proj_v, proj_t], dim=1))
        return self.classifier(fused)

# -------------------- 8. LOSS & BALANCING WEIGHTS --------------------
pos_weights = []
for label in LABELS:
    t_col = f"target_{label}"
    m_col = f"mask_{label}"
    obs = (tr_df[m_col].fillna(0).astype(float) == 1.0)
    vals = tr_df.loc[obs, t_col].dropna().astype(float)
    pos = float((vals == 1.0).sum())
    neg = float((vals == 0.0).sum())
    pos_weights.append(np.clip(neg / max(pos, 1.0), 0.2, 5.0))
POS_WEIGHT_TENSOR = torch.tensor(pos_weights, dtype=torch.float32).to(DEVICE)

def per_image_masked_bce(logits, targets, mask):
    targets = torch.nan_to_num(targets.float(), nan=0.0)
    mask = mask.float()
    element_loss = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=POS_WEIGHT_TENSOR, reduction="none"
    )
    return (element_loss * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

# -------------------- 9. EVALUATION FUNCTION --------------------
@torch.no_grad()
def evaluate_global(net, loader):
    net.eval()
    total_loss, total_seen = 0.0, 0
    all_preds, all_targets, all_masks = [], [], []

    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
        tgts = batch["targets"].to(DEVICE, non_blocking=True)
        msks = batch["masks"].to(DEVICE, non_blocking=True)
        hist = batch["has_history"].to(DEVICE, non_blocking=True)

        logits = net(imgs, ids, attn, hist)
        loss = per_image_masked_bce(logits, tgts, msks).mean()

        total_loss += loss.item() * len(imgs)
        total_seen += len(imgs)

        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(tgts.cpu().numpy())
        all_masks.append(msks.cpu().numpy())

    preds = np.vstack(all_preds)
    targets = np.vstack(all_targets)
    masks = np.vstack(all_masks)

    per_class_auc = {}
    support_counts = {}
    for j, label in enumerate(LABELS):
        valid = masks[:, j] == 1.0
        y_true = targets[valid, j]
        y_score = preds[valid, j]
        pos_cnt = int((y_true == 1.0).sum())
        neg_cnt = int((y_true == 0.0).sum())
        support_counts[label] = {"pos": pos_cnt, "neg": neg_cnt}
        if pos_cnt > 0 and neg_cnt > 0:
            try:
                per_class_auc[label] = float(roc_auc_score(y_true, y_score))
            except ValueError:
                per_class_auc[label] = float("nan")
        else:
            per_class_auc[label] = float("nan")

    valid_scores = [v for v in per_class_auc.values() if not np.isnan(v)]
    macro_auc = float(np.mean(valid_scores)) if valid_scores else float("nan")
    val_loss = total_loss / max(total_seen, 1)

    return val_loss, macro_auc, per_class_auc, support_counts

# -------------------- 10. FEDAVG SIMULATION LOOP --------------------
global_model = FedMultimodalClassifier().to(DEVICE)

# Calculate proportional client aggregation weights
total_client_samples = sum(len(c_df) for c_df in client_dfs)
client_weights = [len(c_df) / total_client_samples for c_df in client_dfs]
print(f"Aggregation weights (n_k / N): {[round(w, 4) for w in client_weights]}")

print("\n" + "=" * 80)
print(f"STARTING FEDAVG COMMUNICATION ROUNDS (R={GLOBAL_ROUNDS}, E={LOCAL_EPOCHS})")
print("=" * 80)

best_auc = 0.0
best_round = 0
round_history = []

for r in range(1, GLOBAL_ROUNDS + 1):
    t_round = time.perf_counter()
    local_state_dicts = []
    client_losses = []

    for k in range(NUM_CLIENTS):
        local_model = copy.deepcopy(global_model)
        local_model.train()

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, local_model.parameters()),
            lr=CLIENT_LR,
            weight_decay=CLIENT_WEIGHT_DECAY
        )

        c_loss, c_seen = 0.0, 0
        for epoch in range(LOCAL_EPOCHS):
            for batch in client_loaders[k]:
                imgs = batch["image"].to(DEVICE, non_blocking=True)
                ids = batch["input_ids"].to(DEVICE, non_blocking=True)
                attn = batch["attention_mask"].to(DEVICE, non_blocking=True)
                tgts = batch["targets"].to(DEVICE, non_blocking=True)
                msks = batch["masks"].to(DEVICE, non_blocking=True)
                hist = batch["has_history"].to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                logits = local_model(imgs, ids, attn, hist)
                loss = per_image_masked_bce(logits, tgts, msks).mean()
                loss.backward()
                optimizer.step()

                c_loss += loss.item() * len(imgs)
                c_seen += len(imgs)

        client_losses.append(c_loss / max(c_seen, 1))
        local_state_dicts.append(local_model.state_dict())
        del local_model, optimizer
        torch.cuda.empty_cache()

    # Server Parameter Aggregation (FedAvg)
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        if global_dict[key].dtype in (torch.float32, torch.float64):
            stacked = torch.stack([
                local_state_dicts[k][key] * client_weights[k]
                for k in range(NUM_CLIENTS)
            ], dim=0)
            global_dict[key] = stacked.sum(dim=0)
    global_model.load_state_dict(global_dict)

    # Server Validation Evaluation
    val_loss, val_macro_auc, class_auc, support_info = evaluate_global(global_model, val_loader)
    eval_count = sum(~np.isnan(list(class_auc.values())))
    round_time = time.perf_counter() - t_round

    print(
        f"Round [{r}/{GLOBAL_ROUNDS}] ({round_time:.1f}s) | "
        f"Mean Client Loss: {np.mean(client_losses):.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro AUROC: {val_macro_auc:.4f} ({eval_count}/{len(LABELS)} classes)"
    )

    is_best = val_macro_auc > best_auc
    if is_best:
        best_auc = val_macro_auc
        best_round = r
        torch.save(global_model.state_dict(), OUTPUT_DIR / "fedavg_best_model.pt")

    round_history.append({
        "round": r,
        "round_time_seconds": round_time,
        "mean_client_loss": float(np.mean(client_losses)),
        "val_loss": val_loss,
        "val_macro_auc": val_macro_auc,
        "per_class_auc": class_auc
    })

# -------------------- 11. PERSIST RESULTS & DETAILED SUMMARY --------------------
torch.save(global_model.state_dict(), OUTPUT_DIR / "fedavg_final_round_model.pt")

best_ckpt = torch.load(OUTPUT_DIR / "fedavg_best_model.pt", map_location=DEVICE, weights_only=False)
global_model.load_state_dict(best_ckpt)
val_loss, best_macro_auc, best_class_auc, support_info = evaluate_global(global_model, val_loader)

summary_rows = []
for label in LABELS:
    score = best_class_auc.get(label, np.nan)
    sup = support_info[label]
    summary_rows.append({
        "Pathology": label,
        "Val Pos": sup["pos"],
        "Val Neg": sup["neg"],
        "FedAvg AUROC": f"{score:.4f}" if not np.isnan(score) else "NaN"
    })

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 80)
print(f"E3 FEDAVG EXPERIMENT COMPLETE (Best Round {best_round})")
print("=" * 80)
print(f"Peak Validation Macro AUROC: {best_macro_auc:.4f}")
print("-" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)

e3_summary_payload = {
    "protocol": {
        "seed": SEED,
        "num_clients": NUM_CLIENTS,
        "dirichlet_alpha": DIRICHLET_ALPHA,
        "global_rounds": GLOBAL_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "batch_size": BATCH_SIZE,
        "client_lr": CLIENT_LR,
        "client_weights": client_weights,
        "labels": LABELS
    },
    "best_round": best_round,
    "best_macro_auc": best_macro_auc,
    "best_per_class_auc": best_class_auc,
    "round_history": round_history
}

with open(OUTPUT_DIR / "e3_fedavg_results.json", "w") as f:
    json.dump(e3_summary_payload, f, indent=2)

print(f"\nAll artifacts (best model weights, results JSON) saved to:\n{OUTPUT_DIR}")

=== DP-MMFL E3: FEDAVG MULTIMODAL SIMULATION ===
Device: cuda:0 | Clients: 5 | Alpha: 0.5
Active cached cohort: 38,912 images across 12,696 unique patients
Cohort split: Train=32,934 images (10,791 pats) | Val=5,978 images (1,905 pats)

Constructing Dirichlet non-IID partition (alpha=0.5) across 5 institutions...
=== FEDERATED INSTITUTIONAL PROFILE ===
Client 1: 24,813 images (6,341 patients) | History present: 71.86%
Client 2: 24,074 images (6,259 patients) | History present: 72.24%
Client 3: 16,433 images (3,944 patients) | History present: 72.03%
Client 4: 24,176 images (6,018 patients) | History present: 72.00%
Client 5: 14,918 images (3,516 patients) | History present: 72.70%

Pre-tokenizing clinical histories to eliminate CPU bottleneck during training...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Aggregation weights (n_k / N): [0.2376, 0.2306, 0.1574, 0.2315, 0.1429]

STARTING FEDAVG COMMUNICATION ROUNDS (R=5, E=1)
